In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("/content/drive/MyDrive/data/nppes_sample_300000.csv")

print("File exists:", DATA_PATH.exists())
print("File path:", DATA_PATH)
print("File size (MB):", round(DATA_PATH.stat().st_size / (1024 * 1024), 2))

File exists: True
File path: /content/drive/MyDrive/data/nppes_sample_300000.csv
File size (MB): 160.22


In [ ]:
# Read only a small sample first
sample_df = pd.read_csv(
    DATA_PATH,
    nrows=10,
    low_memory=False
)

print("Columns:", len(sample_df.columns))
print("\nColumn names:")
for col in sample_df.columns:
    print("-", col)

print("\nSample:")
display(sample_df.head())

Columns: 330

Column names:
- NPI
- Entity Type Code
- Replacement NPI
- Employer Identification Number (EIN)
- Provider Organization Name (Legal Business Name)
- Provider Last Name (Legal Name)
- Provider First Name
- Provider Middle Name
- Provider Name Prefix Text
- Provider Name Suffix Text
- Provider Credential Text
- Provider Other Organization Name
- Provider Other Organization Name Type Code
- Provider Other Last Name
- Provider Other First Name
- Provider Other Middle Name
- Provider Other Name Prefix Text
- Provider Other Name Suffix Text
- Provider Other Credential Text
- Provider Other Last Name Type Code
- Provider First Line Business Mailing Address
- Provider Second Line Business Mailing Address
- Provider Business Mailing Address City Name
- Provider Business Mailing Address State Name
- Provider Business Mailing Address Postal Code
- Provider Business Mailing Address Country Code (If outside U.S.)
- Provider Business Mailing Address Telephone Number
- Provider Business

,NPI,Entity Type Code,Replacement NPI,Employer Identification Number (EIN),Provider Organization Name (Legal Business Name),Provider Last Name (Legal Name),Provider First Name,Provider Middle Name,Provider Name Prefix Text,Provider Name Suffix Text,...,Healthcare Provider Taxonomy Group_7,Healthcare Provider Taxonomy Group_8,Healthcare Provider Taxonomy Group_9,Healthcare Provider Taxonomy Group_10,Healthcare Provider Taxonomy Group_11,Healthcare Provider Taxonomy Group_12,Healthcare Provider Taxonomy Group_13,Healthcare Provider Taxonomy Group_14,Healthcare Provider Taxonomy Group_15,Certification Date
0,1184627820,1.0,NaN,NaN,NaN,WEISS,CARL,B,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,04/04/2023
1,1568460541,1.0,NaN,NaN,NaN,LONG,JOHN,P,NaN,JR.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1396729885,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1174610372,2.0,NaN,<UNAVAIL>,"JULIE K. KUEKER, O.D., P.A.",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1013997535,1.0,NaN,NaN,NaN,SERVICE,MICHAEL,JAMES,MR.,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import pandas as pd

chunk_size = 50_000

total_rows = 0
column_names = None

for chunk in pd.read_csv(DATA_PATH, chunksize=chunk_size, low_memory=False):
    total_rows += len(chunk)

    if column_names is None:
        column_names = chunk.columns.tolist()

print("Total rows:", total_rows)
print("Total columns:", len(column_names))

Total rows: 300000
Total columns: 330


In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("/content/drive/MyDrive/data/nppes_sample_300000.csv")

# Only load columns needed for the first geospatial inspection
GEO_COLUMNS = [
    "NPI",
    "Entity Type Code",

    "Provider Organization Name (Legal Business Name)",
    "Provider Last Name (Legal Name)",
    "Provider First Name",

    "Provider First Line Business Practice Location Address",
    "Provider Second Line Business Practice Location Address",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address Postal Code",
    "Provider Business Practice Location Address Country Code (If outside U.S.)",

    "Healthcare Provider Taxonomy Code_1",
    "Healthcare Provider Primary Taxonomy Switch_1",
]

geo_raw = pd.read_csv(
    DATA_PATH,
    usecols=GEO_COLUMNS,
    low_memory=False
)

print("Rows loaded:", len(geo_raw))
print("Columns loaded:", len(geo_raw.columns))

display(geo_raw.head())

Rows loaded: 300000
Columns loaded: 13


,NPI,Entity Type Code,Provider Organization Name (Legal Business Name),Provider Last Name (Legal Name),Provider First Name,Provider First Line Business Practice Location Address,Provider Second Line Business Practice Location Address,Provider Business Practice Location Address City Name,Provider Business Practice Location Address State Name,Provider Business Practice Location Address Postal Code,Provider Business Practice Location Address Country Code (If outside U.S.),Healthcare Provider Taxonomy Code_1,Healthcare Provider Primary Taxonomy Switch_1
0,1184627820,1.0,NaN,WEISS,CARL,8405 N RUN MEDICAL DR,NaN,MECHANICSVILLE,VA,231162309,US,207XS0106X,N
1,1568460541,1.0,NaN,LONG,JOHN,2000 WASHINGTON ST,WHITE BUILDING 443,NEWTON,MA,024621650,US,208800000X,Y
2,1396729885,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1174610372,2.0,"JULIE K. KUEKER, O.D., P.A.",NaN,NaN,1870 W MAIN ST,NaN,SALEM,IL,628815838,US,152W00000X,Y
4,1013997535,1.0,NaN,SERVICE,MICHAEL,2000 ROOSEVELT AVE,NaN,HAVELOCK,NC,285323639,US,363LF0000X,Y


In [ ]:
ADDRESS_COLUMNS = [
    "Provider First Line Business Practice Location Address",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address Postal Code",
]

print("NPI uniqueness")
print("Unique NPIs:", geo_raw["NPI"].nunique())
print("Duplicate NPIs:", geo_raw["NPI"].duplicated().sum())

print("\nMissing-value counts")
for col in ADDRESS_COLUMNS:
    missing = geo_raw[col].isna().sum()
    pct = missing / len(geo_raw) * 100
    print(f"{col}: {missing:,} ({pct:.2f}%)")

NPI uniqueness
Unique NPIs: 300000
Duplicate NPIs: 0

Missing-value counts
Provider First Line Business Practice Location Address: 18,238 (6.08%)
Provider Business Practice Location Address City Name: 18,238 (6.08%)
Provider Business Practice Location Address State Name: 18,238 (6.08%)
Provider Business Practice Location Address Postal Code: 18,238 (6.08%)


In [ ]:
country_col = "Provider Business Practice Location Address Country Code (If outside U.S.)"

print(
    geo_raw[country_col]
    .fillna("US_OR_BLANK")
    .value_counts()
    .head(20)
)

Provider Business Practice Location Address Country Code (If outside U.S.)
US             281479
US_OR_BLANK     18238
DE                 94
JP                 61
IT                 28
KR                 25
UM                 25
CA                  9
ES                  9
GB                  5
BE                  4
EG                  2
IL                  2
AU                  2
PK                  2
DM                  1
SG                  1
EC                  1
MT                  1
HT                  1
Name: count, dtype: int64


In [ ]:
# ============================================================
# STAGE 2B — RECREATE THE 13-COLUMN GEOGRAPHIC DATASET
# ============================================================

import pandas as pd
import numpy as np

FILE_PATH = "/content/drive/MyDrive/data/nppes_sample_300000.csv"

# Exact columns needed for BE-3 geographic processing
columns_needed = [
    "NPI",
    "Entity Type Code",
    "Provider Organization Name (Legal Business Name)",
    "Provider Last Name (Legal Name)",
    "Provider First Name",
    "Provider First Line Business Practice Location Address",
    "Provider Second Line Business Practice Location Address",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address Postal Code",
    "Provider Business Practice Location Address Country Code (If outside U.S.)",
    "Healthcare Provider Taxonomy Code_1",
    "Healthcare Provider Primary Taxonomy Switch_1"
]

# Read only the required columns from the 300,000-row NPPES file
nppes_geo = pd.read_csv(
    FILE_PATH,
    usecols=columns_needed,
    dtype={
        "NPI": "string",
        "Provider Business Practice Location Address Postal Code": "string",
        "Provider Business Practice Location Address State Name": "string",
        "Healthcare Provider Taxonomy Code_1": "string"
    },
    low_memory=False
)

print("✓ nppes_geo created successfully")
print("Rows:", len(nppes_geo))
print("Columns:", len(nppes_geo.columns))

display(nppes_geo.head())

✓ nppes_geo created successfully
Rows: 300000
Columns: 13


,NPI,Entity Type Code,Provider Organization Name (Legal Business Name),Provider Last Name (Legal Name),Provider First Name,Provider First Line Business Practice Location Address,Provider Second Line Business Practice Location Address,Provider Business Practice Location Address City Name,Provider Business Practice Location Address State Name,Provider Business Practice Location Address Postal Code,Provider Business Practice Location Address Country Code (If outside U.S.),Healthcare Provider Taxonomy Code_1,Healthcare Provider Primary Taxonomy Switch_1
0,1184627820,1.0,NaN,WEISS,CARL,8405 N RUN MEDICAL DR,NaN,MECHANICSVILLE,VA,231162309,US,207XS0106X,N
1,1568460541,1.0,NaN,LONG,JOHN,2000 WASHINGTON ST,WHITE BUILDING 443,NEWTON,MA,024621650,US,208800000X,Y
2,1396729885,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,NaN,<NA>,NaN
3,1174610372,2.0,"JULIE K. KUEKER, O.D., P.A.",NaN,NaN,1870 W MAIN ST,NaN,SALEM,IL,628815838,US,152W00000X,Y
4,1013997535,1.0,NaN,SERVICE,MICHAEL,2000 ROOSEVELT AVE,NaN,HAVELOCK,NC,285323639,US,363LF0000X,Y


In [ ]:
# ============================================================
# STAGE 2B — CLEAN ZIP CODES & DETERMINE GEOGRAPHIC ELIGIBILITY
# ============================================================

import pandas as pd
import numpy as np

print("Starting rows:", len(nppes_geo))

# ------------------------------------------------------------
# 1. Create a clean 5-digit ZIP code
# ------------------------------------------------------------

zip_col = "Provider Business Practice Location Address Postal Code"
country_col = "Provider Business Practice Location Address Country Code (If outside U.S.)"

# Convert ZIP to string and keep only the first 5 characters
nppes_geo["zip_clean"] = (
    nppes_geo[zip_col]
    .astype("string")
    .str.strip()
    .str.extract(r"(\d{5})", expand=False)
)

# ------------------------------------------------------------
# 2. Clean state
# ------------------------------------------------------------

state_col = "Provider Business Practice Location Address State Name"

nppes_geo["state_clean"] = (
    nppes_geo[state_col]
    .astype("string")
    .str.strip()
    .str.upper()
)

# ------------------------------------------------------------
# 3. Clean country
# ------------------------------------------------------------

nppes_geo["country_clean"] = (
    nppes_geo[country_col]
    .astype("string")
    .str.strip()
    .str.upper()
)

# ------------------------------------------------------------
# 4. Define U.S. geographic eligibility
# ------------------------------------------------------------

# A provider is geographically eligible if:
# - country is US
# - state is present
# - ZIP is a valid 5-digit ZIP
# - practice address is present

nppes_geo["geographic_eligible"] = (
    (nppes_geo["country_clean"] == "US")
    & nppes_geo["state_clean"].notna()
    & nppes_geo["zip_clean"].notna()
)

# ------------------------------------------------------------
# 5. Create the geographic processing dataset
# ------------------------------------------------------------

geo_ready = nppes_geo[
    nppes_geo["geographic_eligible"]
].copy()

# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------

print("\n========== GEOGRAPHIC ELIGIBILITY SUMMARY ==========")

print("Total original rows:", len(nppes_geo))
print("Geographically eligible U.S. providers:", len(geo_ready))
print(
    "Not eligible:",
    len(nppes_geo) - len(geo_ready)
)

print("\nUnique states:", geo_ready["state_clean"].nunique())
print("Unique ZIP codes:", geo_ready["zip_clean"].nunique())

print("\nEligibility:")
print(
    nppes_geo["geographic_eligible"]
    .value_counts()
)

print("\nCountry distribution after cleaning:")
print(
    nppes_geo["country_clean"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nState distribution:")
print(
    geo_ready["state_clean"]
    .value_counts()
    .head(20)
)

# ------------------------------------------------------------
# 7. Validate ZIP codes
# ------------------------------------------------------------

print("\n========== ZIP VALIDATION ==========")

print(
    "Missing ZIP after cleaning:",
    geo_ready["zip_clean"].isna().sum()
)

print(
    "Invalid ZIP length:",
    (~geo_ready["zip_clean"].str.match(r"^\d{5}$", na=False)).sum()
)

# ------------------------------------------------------------
# 8. Preview
# ------------------------------------------------------------

preview_cols = [
    "NPI",
    "state_clean",
    "zip_clean",
    "Healthcare Provider Taxonomy Code_1"
]

print("\n========== GEO-READY PREVIEW ==========")

display(
    geo_ready[preview_cols].head(10)
)

print("\n✓ ZIP cleaning and geographic eligibility completed.")

Starting rows: 300000

========== GEOGRAPHIC ELIGIBILITY SUMMARY ==========
Total original rows: 300000
Geographically eligible U.S. providers: 281478
Not eligible: 18522

Unique states: 60
Unique ZIP codes: 16789

Eligibility:
geographic_eligible
True     281478
False     18522
Name: count, dtype: Int64

Country distribution after cleaning:
country_clean
US      281479
<NA>     18238
DE          94
JP          61
IT          28
KR          25
UM          25
CA           9
ES           9
GB           5
BE           4
EG           2
IL           2
AU           2
PK           2
DM           1
SG           1
EC           1
MT           1
HT           1
Name: count, dtype: Int64

State distribution:
state_clean
CA    20107
TX    20075
FL    18555
NY    17191
PA    16024
OH    12824
NC    10487
IL     8968
MI     8758
MA     8589
VA     7847
MN     7180
NJ     7086
GA     6884
IN     6442
AZ     6321
TN     6282
WA     5980
MD     5952
MO     5667
Name: count, dtype: Int64

========== ZIP V

,NPI,state_clean,zip_clean,Healthcare Provider Taxonomy Code_1
0,1184627820,VA,23116,207XS0106X
1,1568460541,MA,02462,208800000X
3,1174610372,IL,62881,152W00000X
4,1013997535,NC,28532,363LF0000X
5,1932105079,NY,11219,207V00000X
6,1497713234,NC,27021,207Q00000X
7,1083699102,AL,35233,207RC0200X
8,1669472643,NV,89109,207L00000X
9,1437113537,SC,29203,207V00000X
10,1083691083,MN,55905,208D00000X



✓ ZIP cleaning and geographic eligibility completed.


In [ ]:
# ============================================================
# STAGE 2C — INSPECT AVAILABLE GEOGRAPHIC REFERENCE DATA
# ============================================================

import os
from pathlib import Path

print("========== GOOGLE DRIVE DATA ==========")

data_dir = Path("/content/drive/MyDrive/data")

if data_dir.exists():
    for item in sorted(data_dir.iterdir()):
        print(item.name)
else:
    print("Data directory not found:", data_dir)

print("\n========== CURRENT WORKING DIRECTORY ==========")

for item in sorted(Path("/content").iterdir()):
    print(item)

========== GOOGLE DRIVE DATA ==========
geocode_test_input.csv
geographic_reference
nppes_sample_300000.csv

========== CURRENT WORKING DIRECTORY ==========
/content/.config
/content/drive
/content/sample_data


In [ ]:
# ============================================================
# STAGE 2C — DOWNLOAD CENSUS ZCTA REFERENCE DATA
# ============================================================

import os
import requests
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/data/geographic_reference")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2020 Census Gazetteer ZCTA file
ZCTA_URL = (
    "https://www2.census.gov/geo/docs/maps-data/data/gazetteer/"
    "2020_Gazetteer/2020_Gaz_zcta_national.zip"
)

ZIP_PATH = OUTPUT_DIR / "2020_Gaz_zcta_national.zip"

print("Downloading Census ZCTA reference file...")
print(ZCTA_URL)

response = requests.get(ZCTA_URL, timeout=120)
response.raise_for_status()

with open(ZIP_PATH, "wb") as f:
    f.write(response.content)

print("\n✓ Download complete")
print("Saved to:", ZIP_PATH)
print("File size:", round(ZIP_PATH.stat().st_size / (1024**2), 2), "MB")

https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2020_Gazetteer/2020_Gaz_zcta_national.zip

✓ Download complete
Saved to: /content/drive/MyDrive/data/geographic_reference/2020_Gaz_zcta_national.zip
File size: 0.95 MB


In [ ]:
# ============================================================
# STAGE 2C — EXTRACT & INSPECT CENSUS ZCTA DATA
# ============================================================

import zipfile
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("/content/drive/MyDrive/data/geographic_reference")
ZIP_PATH = OUTPUT_DIR / "2020_Gaz_zcta_national.zip"

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    print("Files inside Census ZIP:")
    for name in z.namelist():
        print(" -", name)

    z.extractall(OUTPUT_DIR)

print("\n✓ Census ZCTA files extracted")

# Find the extracted text file
txt_files = list(OUTPUT_DIR.glob("*.txt"))

print("\nExtracted TXT files:")
for f in txt_files:
    print(" -", f.name)

Files inside Census ZIP:
 - 2020_Gaz_zcta_national.txt

✓ Census ZCTA files extracted

Extracted TXT files:
 - 2020_Gaz_zcta_national.txt
 - tab20_zcta520_county20_natl.txt


In [ ]:
# ============================================================
# STAGE 2C-2 — DOWNLOAD 2020 ZCTA → COUNTY RELATIONSHIP
# ============================================================

import requests
from pathlib import Path

OUTPUT_DIR = Path("/content/drive/MyDrive/data/geographic_reference")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ZCTA_COUNTY_URL = (
    "https://www2.census.gov/geo/docs/maps-data/data/rel2020/"
    "zcta520/tab20_zcta520_county20_natl.txt"
)

OUTPUT_PATH = OUTPUT_DIR / "tab20_zcta520_county20_natl.txt"

print("Downloading Census ZCTA → County relationship file...")
print(ZCTA_COUNTY_URL)

response = requests.get(ZCTA_COUNTY_URL, timeout=120)
response.raise_for_status()

with open(OUTPUT_PATH, "wb") as f:
    f.write(response.content)

print("\n✓ Download complete")
print("Saved to:", OUTPUT_PATH)
print(
    "File size:",
    round(OUTPUT_PATH.stat().st_size / (1024**2), 2),
    "MB"
)

https://www2.census.gov/geo/docs/maps-data/data/rel2020/zcta520/tab20_zcta520_county20_natl.txt

✓ Download complete
Saved to: /content/drive/MyDrive/data/geographic_reference/tab20_zcta520_county20_natl.txt
File size: 6.51 MB


In [ ]:
# ============================================================
# STAGE 2C-2 — INSPECT ZCTA → COUNTY RELATIONSHIP FILE
# ============================================================

from pathlib import Path

FILE_PATH = Path(
    "/content/drive/MyDrive/data/geographic_reference/"
    "tab20_zcta520_county20_natl.txt"
)

print("File exists:", FILE_PATH.exists())
print("File size:", round(FILE_PATH.stat().st_size / (1024**2), 2), "MB")

print("\n========== FIRST 10 LINES ==========")

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for i in range(10):
        line = f.readline()
        if not line:
            break
        print(line.rstrip())

File exists: True
File size: 6.51 MB

========== FIRST 10 LINES ==========
﻿OID_ZCTA5_20|GEOID_ZCTA5_20|NAMELSAD_ZCTA5_20|AREALAND_ZCTA5_20|AREAWATER_ZCTA5_20|MTFCC_ZCTA5_20|CLASSFP_ZCTA5_20|FUNCSTAT_ZCTA5_20|OID_COUNTY_20|GEOID_COUNTY_20|NAMELSAD_COUNTY_20|AREALAND_COUNTY_20|AREAWATER_COUNTY_20|MTFCC_COUNTY_20|CLASSFP_COUNTY_20|FUNCSTAT_COUNTY_20|AREALAND_PART|AREAWATER_PART
||||||||27590114112812|01003|Baldwin County|4117656199|1132956041|G4020|H1|A|339765765|927218265
||||||||2759099719300|01007|Bibb County|1612188717|9572303|G4020|H1|A|92709500|11113
||||||||27590103020886|01015|Calhoun County|1569246127|16536293|G4020|H1|A|173314495|461654
||||||||27590336389978|01021|Chilton County|1794438835|20592116|G4020|H1|A|67098990|5170
||||||||2759075862059|01025|Clarke County|3207494086|36657889|G4020|H1|A|29757|593967
||||||||27590536790681|01029|Cleburne County|1450666453|2352217|G4020|H1|A|69471534|56495
||||||||27590352984259|01031|Coffee County|1758565928|3907189|G4020|H1|A|15440476|

In [ ]:
# ============================================================
# STAGE 2C-3 — LOAD & VALIDATE ZCTA → COUNTY RELATIONSHIP
# ============================================================

import pandas as pd
from pathlib import Path

RELATIONSHIP_PATH = Path(
    "/content/drive/MyDrive/data/geographic_reference/"
    "tab20_zcta520_county20_natl.txt"
)

zcta_county = pd.read_csv(
    RELATIONSHIP_PATH,
    sep="|",
    dtype="string",
    encoding="utf-8-sig"
)

# Remove completely empty columns created by trailing separators
zcta_county = zcta_county.dropna(axis=1, how="all")

print("========== ZCTA → COUNTY RELATIONSHIP ==========")
print("Rows:", len(zcta_county))
print("Columns:", len(zcta_county.columns))

print("\nColumns:")
for col in zcta_county.columns:
    print(" -", col)

print("\nPreview:")
display(zcta_county.head())

print("\nUnique ZCTAs:", zcta_county["GEOID_ZCTA5_20"].nunique())
print("Unique counties:", zcta_county["GEOID_COUNTY_20"].nunique())

print("\nZCTAs with multiple county relationships:")

county_counts = (
    zcta_county
    .groupby("GEOID_ZCTA5_20")
    .size()
)

print(
    "ZCTAs with >1 county:",
    (county_counts > 1).sum()
)

print(
    "Maximum counties associated with one ZCTA:",
    county_counts.max()
)

========== ZCTA → COUNTY RELATIONSHIP ==========
Rows: 47863
Columns: 18

Columns:
 - OID_ZCTA5_20
 - GEOID_ZCTA5_20
 - NAMELSAD_ZCTA5_20
 - AREALAND_ZCTA5_20
 - AREAWATER_ZCTA5_20
 - MTFCC_ZCTA5_20
 - CLASSFP_ZCTA5_20
 - FUNCSTAT_ZCTA5_20
 - OID_COUNTY_20
 - GEOID_COUNTY_20
 - NAMELSAD_COUNTY_20
 - AREALAND_COUNTY_20
 - AREAWATER_COUNTY_20
 - MTFCC_COUNTY_20
 - CLASSFP_COUNTY_20
 - FUNCSTAT_COUNTY_20
 - AREALAND_PART
 - AREAWATER_PART

Preview:


,OID_ZCTA5_20,GEOID_ZCTA5_20,NAMELSAD_ZCTA5_20,AREALAND_ZCTA5_20,AREAWATER_ZCTA5_20,MTFCC_ZCTA5_20,CLASSFP_ZCTA5_20,FUNCSTAT_ZCTA5_20,OID_COUNTY_20,GEOID_COUNTY_20,NAMELSAD_COUNTY_20,AREALAND_COUNTY_20,AREAWATER_COUNTY_20,MTFCC_COUNTY_20,CLASSFP_COUNTY_20,FUNCSTAT_COUNTY_20,AREALAND_PART,AREAWATER_PART
0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,27590114112812,01003,Baldwin County,4117656199,1132956041,G4020,H1,A,339765765,927218265
1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2759099719300,01007,Bibb County,1612188717,9572303,G4020,H1,A,92709500,11113
2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,27590103020886,01015,Calhoun County,1569246127,16536293,G4020,H1,A,173314495,461654
3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,27590336389978,01021,Chilton County,1794438835,20592116,G4020,H1,A,67098990,5170
4,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2759075862059,01025,Clarke County,3207494086,36657889,G4020,H1,A,29757,593967



Unique ZCTAs: 33791
Unique counties: 3234

ZCTAs with multiple county relationships:
ZCTAs with >1 county: 10186
Maximum counties associated with one ZCTA: 6


In [ ]:
# ============================================================
# STAGE 2C-4 — PRIMARY COUNTY PER ZCTA
# ============================================================

# Convert intersection land/water fields to numeric
zcta_county["AREALAND_PART"] = pd.to_numeric(
    zcta_county["AREALAND_PART"],
    errors="coerce"
)

zcta_county["AREAWATER_PART"] = pd.to_numeric(
    zcta_county["AREAWATER_PART"],
    errors="coerce"
)

# Rank counties within each ZCTA by land intersection
zcta_county_ranked = zcta_county.sort_values(
    ["GEOID_ZCTA5_20", "AREALAND_PART"],
    ascending=[True, False]
).copy()

# Select the county with the largest land intersection
zcta_primary_county = (
    zcta_county_ranked
    .drop_duplicates(
        subset=["GEOID_ZCTA5_20"],
        keep="first"
    )
    [[
        "GEOID_ZCTA5_20",
        "NAMELSAD_ZCTA5_20",
        "GEOID_COUNTY_20",
        "NAMELSAD_COUNTY_20",
        "AREALAND_PART",
        "AREAWATER_PART"
    ]]
    .reset_index(drop=True)
)

print("========== PRIMARY COUNTY LOOKUP ==========")
print("Unique ZCTAs:", len(zcta_primary_county))
print(
    "Unique primary counties:",
    zcta_primary_county["GEOID_COUNTY_20"].nunique()
)

display(zcta_primary_county.head(10))

========== PRIMARY COUNTY LOOKUP ==========
Unique ZCTAs: 33792
Unique primary counties: 3211


,GEOID_ZCTA5_20,NAMELSAD_ZCTA5_20,GEOID_COUNTY_20,NAMELSAD_COUNTY_20,AREALAND_PART,AREAWATER_PART
0,00601,ZCTA5 00601,72001,Adjuntas Municipio,164781682,799292
1,00602,ZCTA5 00602,72003,Aguada Municipio,78530159,4428428
2,00603,ZCTA5 00603,72005,Aguadilla Municipio,88747846,6276536
3,00606,ZCTA5 00606,72093,Maricao Municipio,94466099,12487
4,00610,ZCTA5 00610,72011,Añasco Municipio,93009966,4310530
5,00611,ZCTA5 00611,72141,Utuado Municipio,27570859,3631
6,00612,ZCTA5 00612,72013,Arecibo Municipio,195161357,12843144
7,00616,ZCTA5 00616,72013,Arecibo Municipio,28189755,143499
8,00617,ZCTA5 00617,72017,Barceloneta Municipio,47372606,1692937
9,00622,ZCTA5 00622,72023,Cabo Rojo Municipio,80724808,19583150


In [ ]:
# ============================================================
# STAGE 2C-5 — SAVE GEOGRAPHIC REFERENCE TABLES
# ============================================================

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/data/geographic_reference"
)

FULL_RELATIONSHIP_PATH = (
    OUTPUT_DIR / "zcta_county_relationship_2020.csv"
)

PRIMARY_COUNTY_PATH = (
    OUTPUT_DIR / "zcta_primary_county_2020.csv"
)

zcta_county.to_csv(
    FULL_RELATIONSHIP_PATH,
    index=False
)

zcta_primary_county.to_csv(
    PRIMARY_COUNTY_PATH,
    index=False
)

print("✓ Full relationship saved:")
print(FULL_RELATIONSHIP_PATH)

print("\n✓ Primary county lookup saved:")
print(PRIMARY_COUNTY_PATH)

✓ Full relationship saved:
/content/drive/MyDrive/data/geographic_reference/zcta_county_relationship_2020.csv

✓ Primary county lookup saved:
/content/drive/MyDrive/data/geographic_reference/zcta_primary_county_2020.csv


In [ ]:
# ============================================================
# STAGE 2C-6 — CLEAN ZCTA REFERENCE & JOIN TO NPPES
# ============================================================

import pandas as pd
from pathlib import Path

PRIMARY_COUNTY_PATH = Path(
    "/content/drive/MyDrive/data/geographic_reference/"
    "zcta_primary_county_2020.csv"
)

# Load the saved primary lookup
zcta_primary_county = pd.read_csv(
    PRIMARY_COUNTY_PATH,
    dtype={
        "GEOID_ZCTA5_20": "string",
        "GEOID_COUNTY_20": "string",
        "NAMELSAD_ZCTA5_20": "string",
        "NAMELSAD_COUNTY_20": "string"
    }
)

print("Rows before cleaning:", len(zcta_primary_county))

# Remove rows without a valid ZCTA
zcta_primary_county = zcta_primary_county[
    zcta_primary_county["GEOID_ZCTA5_20"]
    .astype("string")
    .str.match(r"^\d{5}$", na=False)
].copy()

# Normalize to exactly 5 digits
zcta_primary_county["zcta_clean"] = (
    zcta_primary_county["GEOID_ZCTA5_20"]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)

# Normalize county FIPS
zcta_primary_county["county_fips"] = (
    zcta_primary_county["GEOID_COUNTY_20"]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)

print("Rows after cleaning:", len(zcta_primary_county))
print(
    "Unique ZCTAs:",
    zcta_primary_county["zcta_clean"].nunique()
)

print("\nMissing ZCTA:", zcta_primary_county["zcta_clean"].isna().sum())
print("Missing county FIPS:", zcta_primary_county["county_fips"].isna().sum())

display(
    zcta_primary_county[
        [
            "zcta_clean",
            "NAMELSAD_ZCTA5_20",
            "county_fips",
            "NAMELSAD_COUNTY_20"
        ]
    ].head(10)
)

Rows before cleaning: 33792
Rows after cleaning: 33791
Unique ZCTAs: 33791

Missing ZCTA: 0
Missing county FIPS: 0


,zcta_clean,NAMELSAD_ZCTA5_20,county_fips,NAMELSAD_COUNTY_20
0,00601,ZCTA5 00601,72001,Adjuntas Municipio
1,00602,ZCTA5 00602,72003,Aguada Municipio
2,00603,ZCTA5 00603,72005,Aguadilla Municipio
3,00606,ZCTA5 00606,72093,Maricao Municipio
4,00610,ZCTA5 00610,72011,Añasco Municipio
5,00611,ZCTA5 00611,72141,Utuado Municipio
6,00612,ZCTA5 00612,72013,Arecibo Municipio
7,00616,ZCTA5 00616,72013,Arecibo Municipio
8,00617,ZCTA5 00617,72017,Barceloneta Municipio
9,00622,ZCTA5 00622,72023,Cabo Rojo Municipio


In [ ]:
# ============================================================
# STAGE 2C-7 — JOIN NPPES PROVIDERS TO ZCTA / COUNTY
# ============================================================

# Make sure NPPES ZIP is a clean 5-character string
geo_ready["zcta_clean"] = (
    geo_ready["zip_clean"]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)

county_lookup = zcta_primary_county[
    [
        "zcta_clean",
        "NAMELSAD_ZCTA5_20",
        "county_fips",
        "NAMELSAD_COUNTY_20"
    ]
].drop_duplicates(
    subset=["zcta_clean"]
)

# Join
geo_ready = geo_ready.merge(
    county_lookup,
    on="zcta_clean",
    how="left",
    validate="many_to_one"
)

print("========== NPPES → ZCTA / COUNTY JOIN ==========")

print("NPPES geographic-ready providers:", len(geo_ready))

print(
    "Providers with ZCTA match:",
    geo_ready["NAMELSAD_ZCTA5_20"].notna().sum()
)

print(
    "Providers without ZCTA match:",
    geo_ready["NAMELSAD_ZCTA5_20"].isna().sum()
)

print(
    "ZCTA match rate:",
    round(
        geo_ready["NAMELSAD_ZCTA5_20"].notna().mean() * 100,
        2
    ),
    "%"
)

print(
    "Providers with county match:",
    geo_ready["county_fips"].notna().sum()
)

print(
    "Providers without county match:",
    geo_ready["county_fips"].isna().sum()
)

print("\n========== PREVIEW ==========")

display(
    geo_ready[
        [
            "NPI",
            "state_clean",
            "zip_clean",
            "zcta_clean",
            "NAMELSAD_ZCTA5_20",
            "county_fips",
            "NAMELSAD_COUNTY_20",
            "Healthcare Provider Taxonomy Code_1"
        ]
    ].head(15)
)

========== NPPES → ZCTA / COUNTY JOIN ==========
NPPES geographic-ready providers: 281478
Providers with ZCTA match: 276704
Providers without ZCTA match: 4774
ZCTA match rate: 98.3 %
Providers with county match: 276704
Providers without county match: 4774

========== PREVIEW ==========


,NPI,state_clean,zip_clean,zcta_clean,NAMELSAD_ZCTA5_20,county_fips,NAMELSAD_COUNTY_20,Healthcare Provider Taxonomy Code_1
0,1184627820,VA,23116,23116,ZCTA5 23116,51085,Hanover County,207XS0106X
1,1568460541,MA,02462,02462,ZCTA5 02462,25017,Middlesex County,208800000X
2,1174610372,IL,62881,62881,ZCTA5 62881,17121,Marion County,152W00000X
3,1013997535,NC,28532,28532,ZCTA5 28532,37049,Craven County,363LF0000X
4,1932105079,NY,11219,11219,ZCTA5 11219,36047,Kings County,207V00000X
5,1497713234,NC,27021,27021,ZCTA5 27021,37169,Stokes County,207Q00000X
6,1083699102,AL,35233,35233,ZCTA5 35233,01073,Jefferson County,207RC0200X
7,1669472643,NV,89109,89109,ZCTA5 89109,32003,Clark County,207L00000X
8,1437113537,SC,29203,29203,ZCTA5 29203,45079,Richland County,207V00000X
9,1083691083,MN,55905,55905,ZCTA5 55905,27109,Olmsted County,208D00000X


In [ ]:
# ============================================================
# STAGE 2C-8 — GEOGRAPHIC JOIN QA
# ============================================================

print("========== GEOGRAPHIC JOIN QA ==========")

# Check that provider row count did not change
print(
    "Provider rows preserved:",
    len(geo_ready) == 281478
)

# Check duplicate NPI
print(
    "Duplicate NPIs:",
    geo_ready["NPI"].duplicated().sum()
)

# Check duplicate ZCTA lookup
print(
    "Duplicate ZCTA values in lookup:",
    county_lookup["zcta_clean"].duplicated().sum()
)

# Providers without county mapping
unmatched = geo_ready[
    geo_ready["county_fips"].isna()
].copy()

print(
    "\nUnmatched provider rows:",
    len(unmatched)
)

if len(unmatched) > 0:
    print("\nUnmatched ZIP examples:")
    display(
        unmatched[
            [
                "zip_clean",
                "state_clean",
                "Provider Business Practice Location Address City Name"
            ]
        ]
        .drop_duplicates()
        .head(20)
    )

print("\nState-level provider counts:")
display(
    geo_ready["state_clean"]
    .value_counts()
    .head(20)
)

print("\n✓ ZCTA / county geographic join completed.")

========== GEOGRAPHIC JOIN QA ==========
Provider rows preserved: True
Duplicate NPIs: 0
Duplicate ZCTA values in lookup: 0

Unmatched provider rows: 4774

Unmatched ZIP examples:


,zip_clean,state_clean,Provider Business Practice Location Address City Name
18,94143,CA,SAN FRANCISCO
130,79430,TX,LUBBOCK
161,20593,DC,WASHINGTON
196,76508,TX,TEMPLE
308,38157,TN,MEMPHIS
511,14263,NY,BUFFALO
513,00785,PR,GUAYAMA
674,84132,UT,SALT LAKE CITY
683,63902,MO,POPLAR BLUFF
731,27157,NC,WINSTON SALEM



State-level provider counts:


,count
state_clean,
CA,20107
TX,20075
FL,18555
NY,17191
PA,16024
OH,12824
NC,10487
IL,8968
MI,8758



✓ ZCTA / county geographic join completed.


In [ ]:
# ============================================================
# STAGE 2C-9 — DIAGNOSE UNMATCHED NPPES ZIP CODES
# ============================================================

unmatched = geo_ready[
    geo_ready["county_fips"].isna()
].copy()

print("========== UNMATCHED ZIP ANALYSIS ==========")

print("Unmatched provider rows:", len(unmatched))
print(
    "Unique unmatched ZIPs:",
    unmatched["zip_clean"].nunique()
)

print(
    "Unique unmatched states:",
    unmatched["state_clean"].nunique()
)

print("\n========== UNMATCHED PROVIDERS BY STATE ==========")

unmatched_state = (
    unmatched["state_clean"]
    .value_counts(dropna=False)
)

display(unmatched_state)

print("\n========== UNMATCHED ZIP FREQUENCY ==========")

unmatched_zip_counts = (
    unmatched
    .groupby(
        ["state_clean", "zip_clean"],
        dropna=False
    )
    .size()
    .reset_index(name="provider_count")
    .sort_values(
        "provider_count",
        ascending=False
    )
)

display(unmatched_zip_counts.head(50))

print("\n========== SAMPLE UNMATCHED PROVIDERS ==========")

display(
    unmatched[
        [
            "NPI",
            "state_clean",
            "zip_clean",
            "Provider Business Practice Location Address City Name",
            "Healthcare Provider Taxonomy Code_1"
        ]
    ].head(30)
)

========== UNMATCHED ZIP ANALYSIS ==========
Unmatched provider rows: 4774
Unique unmatched ZIPs: 650
Unique unmatched states: 55

========== UNMATCHED PROVIDERS BY STATE ==========


,count
state_clean,
OH,619
NC,487
MA,372
TX,351
CA,294
CT,271
NY,201
PA,191
WA,177



========== UNMATCHED ZIP FREQUENCY ==========


,state_clean,zip_clean,provider_count
463,OH,44195,577
364,NC,27157,398
303,MA,01655,269
598,TX,76508,266
139,CA,94143,228
160,CT,06030,176
651,WA,98431,131
456,NY,14263,119
399,NH,03756,109
191,DE,19718,77



========== SAMPLE UNMATCHED PROVIDERS ==========


,NPI,state_clean,zip_clean,Provider Business Practice Location Address City Name,Healthcare Provider Taxonomy Code_1
18,1437197597,CA,94143,SAN FRANCISCO,363L00000X
130,1609810977,TX,79430,LUBBOCK,332B00000X
161,1558349084,DC,20593,WASHINGTON,247200000X
196,1275594632,TX,76508,TEMPLE,207RG0100X
308,1063491280,TN,38157,MEMPHIS,207RP1001X
370,1114984648,CA,94143,SAN FRANCISCO,207R00000X
511,1669470001,NY,14263,BUFFALO,2086X0206X
513,1871592519,PR,00785,GUAYAMA,1223G0001X
568,1053397208,TX,79430,LUBBOCK,103G00000X
674,1659480465,UT,84132,SALT LAKE CITY,225100000X


In [ ]:
# ============================================================
# STAGE 2C-10 — CHECK UNMATCHED ZIPs AGAINST ZCTA GAZETTEER
# ============================================================

ZCTA_GAZETTEER_PATH = Path(
    "/content/drive/MyDrive/data/geographic_reference/"
    "2020_Gaz_zcta_national.txt"
)

zcta_gazetteer = pd.read_csv(
    ZCTA_GAZETTEER_PATH,
    sep="\t",
    dtype="string"
)

print("========== ZCTA GAZETTEER ==========")
print("Rows:", len(zcta_gazetteer))
print("Columns:", list(zcta_gazetteer.columns))

display(zcta_gazetteer.head())

# Find the ZCTA column
print("\nColumn names:")
for col in zcta_gazetteer.columns:
    print(" -", repr(col))

========== ZCTA GAZETTEER ==========
Rows: 33144
Columns: ['GEOID', 'ALAND', 'AWATER', 'ALAND_SQMI', 'AWATER_SQMI', 'INTPTLAT', 'INTPTLONG                                                                                                                                  ']


,GEOID,ALAND,AWATER,ALAND_SQMI,AWATER_SQMI,INTPTLAT,INTPTLONG
0,00601,166659744,799292,64.348,0.309,18.180555,-66.749961 ...
1,00602,79307538,4428428,30.621,1.71,18.361945,-67.175597 ...
2,00603,81887203,181412,31.617,0.07,18.455183,-67.119887 ...
3,00606,109579950,12487,42.309,0.005,18.158327,-66.932928 ...
4,00610,93013430,4172059,35.913,1.611,18.294032,-67.127156 ...



Column names:
 - 'GEOID'
 - 'ALAND'
 - 'AWATER'
 - 'ALAND_SQMI'
 - 'AWATER_SQMI'
 - 'INTPTLAT'
 - 'INTPTLONG                                                                                                                                  '


In [ ]:
# ============================================================
# STAGE 2C-11 — MATCH NPPES ZIPs TO CENSUS ZCTA CENTROIDS
# ============================================================

# Clean the Gazetteer column names first
zcta_gazetteer = zcta_gazetteer.copy()

zcta_gazetteer.columns = (
    zcta_gazetteer.columns
    .astype("string")
    .str.strip()
)

# Normalize ZCTA and coordinate columns
zcta_gazetteer["zcta_clean"] = (
    zcta_gazetteer["GEOID"]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)

zcta_gazetteer["centroid_latitude"] = pd.to_numeric(
    zcta_gazetteer["INTPTLAT"],
    errors="coerce"
)

zcta_gazetteer["centroid_longitude"] = pd.to_numeric(
    zcta_gazetteer["INTPTLONG"],
    errors="coerce"
)

# Only the fields we need
zcta_centroids = zcta_gazetteer[
    [
        "zcta_clean",
        "centroid_latitude",
        "centroid_longitude"
    ]
].drop_duplicates(
    subset=["zcta_clean"]
)

print("========== ZCTA CENTROID REFERENCE ==========")
print("Gazetteer rows:", len(zcta_gazetteer))
print("Unique ZCTAs:", zcta_centroids["zcta_clean"].nunique())

print(
    "Missing latitude:",
    zcta_centroids["centroid_latitude"].isna().sum()
)

print(
    "Missing longitude:",
    zcta_centroids["centroid_longitude"].isna().sum()
)

# Match only the currently unmatched providers
unmatched = geo_ready[
    geo_ready["county_fips"].isna()
].copy()

unmatched_centroid_check = unmatched.merge(
    zcta_centroids,
    on="zcta_clean",
    how="left",
    validate="many_to_one"
)

print("\n========== UNMATCHED ZIP → CENTROID ==========")

print(
    "Unmatched provider rows:",
    len(unmatched_centroid_check)
)

print(
    "Providers with Census ZCTA centroid:",
    unmatched_centroid_check["centroid_latitude"].notna().sum()
)

print(
    "Providers without Census ZCTA centroid:",
    unmatched_centroid_check["centroid_latitude"].isna().sum()
)

print(
    "Centroid recovery rate:",
    round(
        unmatched_centroid_check["centroid_latitude"]
        .notna()
        .mean() * 100,
        2
    ),
    "%"
)

print("\nSample recovered ZIPs:")

display(
    unmatched_centroid_check[
        [
            "state_clean",
            "zip_clean",
            "Provider Business Practice Location Address City Name",
            "centroid_latitude",
            "centroid_longitude"
        ]
    ]
    .drop_duplicates()
    .head(30)
)

========== ZCTA CENTROID REFERENCE ==========
Gazetteer rows: 33144
Unique ZCTAs: 33144
Missing latitude: 0
Missing longitude: 0

========== UNMATCHED ZIP → CENTROID ==========
Unmatched provider rows: 4774
Providers with Census ZCTA centroid: 436
Providers without Census ZCTA centroid: 4338
Centroid recovery rate: 9.13 %

Sample recovered ZIPs:


,state_clean,zip_clean,Provider Business Practice Location Address City Name,centroid_latitude,centroid_longitude
0,CA,94143,SAN FRANCISCO,<NA>,<NA>
1,TX,79430,LUBBOCK,<NA>,<NA>
2,DC,20593,WASHINGTON,38.866713,-77.010187
3,TX,76508,TEMPLE,31.077568,-97.364064
4,TN,38157,MEMPHIS,<NA>,<NA>
6,NY,14263,BUFFALO,<NA>,<NA>
7,PR,00785,GUAYAMA,<NA>,<NA>
9,UT,84132,SALT LAKE CITY,<NA>,<NA>
10,MO,63902,POPLAR BLUFF,36.76762,-90.427107
11,NC,27157,WINSTON SALEM,<NA>,<NA>


In [ ]:
# ============================================================
# STAGE 2C-12 — IDENTIFY ZIPs WITHOUT ZCTA CENTROIDS
# ============================================================

no_centroid = unmatched_centroid_check[
    unmatched_centroid_check["centroid_latitude"].isna()
].copy()

print("========== NO ZCTA CENTROID ==========")

print(
    "Provider rows without centroid:",
    len(no_centroid)
)

print(
    "Unique ZIPs without centroid:",
    no_centroid["zip_clean"].nunique()
)

print(
    "States without centroid:",
    no_centroid["state_clean"].nunique()
)

print("\nZIPs without centroid:")

display(
    no_centroid
    .groupby(
        [
            "state_clean",
            "zip_clean",
            "Provider Business Practice Location Address City Name"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="provider_count")
    .sort_values("provider_count", ascending=False)
    .head(50)
)

========== NO ZCTA CENTROID ==========
Provider rows without centroid: 4338
Unique ZIPs without centroid: 626
States without centroid: 55

ZIPs without centroid:


,state_clean,zip_clean,Provider Business Practice Location Address City Name,provider_count
466,OH,44195,CLEVELAND,577
367,NC,27157,WINSTON SALEM,384
303,MA,01655,WORCESTER,269
142,CA,94143,SAN FRANCISCO,228
162,CT,06030,FARMINGTON,176
666,WA,98431,TACOMA,123
459,NY,14263,BUFFALO,119
403,NH,03756,LEBANON,109
191,DE,19718,NEWARK,77
396,NE,68198,OMAHA,70


In [ ]:
# ============================================================
# STAGE 2C-13 — RELOAD + RECONCILE CENSUS ZCTA REFERENCES
# ============================================================

from pathlib import Path
import pandas as pd

RELATIONSHIP_PATH = Path(
    "/content/drive/MyDrive/data/geographic_reference/"
    "tab20_zcta520_county20_natl.txt"
)

# ------------------------------------------------------------
# 1. Reload the official Census ZCTA → County relationship
# ------------------------------------------------------------

zcta_county_relationship = pd.read_csv(
    RELATIONSHIP_PATH,
    sep="|",
    dtype="string",
    encoding="utf-8-sig"
)

# Remove completely empty columns
zcta_county_relationship = zcta_county_relationship.dropna(
    axis=1,
    how="all"
)

# Clean column names
zcta_county_relationship.columns = (
    zcta_county_relationship.columns
    .astype("string")
    .str.strip()
)

# Normalize ZCTA
zcta_county_relationship["zcta_clean"] = (
    zcta_county_relationship["GEOID_ZCTA5_20"]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)

print("========== RELATIONSHIP FILE ==========")
print("Rows:", len(zcta_county_relationship))
print(
    "Unique ZCTAs:",
    zcta_county_relationship["zcta_clean"].nunique()
)

print(
    "Missing ZCTA:",
    zcta_county_relationship["GEOID_ZCTA5_20"].isna().sum()
)

# ------------------------------------------------------------
# 2. Reconcile Gazetteer vs relationship file
# ------------------------------------------------------------

gaz_zctas = set(
    zcta_centroids["zcta_clean"]
    .dropna()
    .astype("string")
)

relationship_zctas = set(
    zcta_county_relationship["zcta_clean"]
    .dropna()
    .astype("string")
)

print("\n========== CENSUS ZCTA RECONCILIATION ==========")

print("Gazetteer unique ZCTAs:", len(gaz_zctas))
print("Relationship unique ZCTAs:", len(relationship_zctas))

print(
    "ZCTAs in Gazetteer but NOT relationship:",
    len(gaz_zctas - relationship_zctas)
)

print(
    "ZCTAs in relationship but NOT Gazetteer:",
    len(relationship_zctas - gaz_zctas)
)

print(
    "ZCTAs present in BOTH:",
    len(gaz_zctas & relationship_zctas)
)

# ------------------------------------------------------------
# 3. Examine the 436 provider cases
# ------------------------------------------------------------

centroid_recoverable = unmatched_centroid_check[
    unmatched_centroid_check["centroid_latitude"].notna()
].copy()

centroid_recoverable_zctas = set(
    centroid_recoverable["zcta_clean"]
    .dropna()
    .astype("string")
)

print("\n========== 436 CENTROID-RECOVERABLE PROVIDERS ==========")

print(
    "Provider rows:",
    len(centroid_recoverable)
)

print(
    "Unique ZCTAs:",
    len(centroid_recoverable_zctas)
)

print(
    "ZCTAs also in relationship:",
    len(
        centroid_recoverable_zctas
        & relationship_zctas
    )
)

print(
    "ZCTAs NOT in relationship:",
    len(
        centroid_recoverable_zctas
        - relationship_zctas
    )
)

print(
    "ZCTAs NOT in Gazetteer:",
    len(
        centroid_recoverable_zctas
        - gaz_zctas
    )
)

========== RELATIONSHIP FILE ==========
Rows: 47863
Unique ZCTAs: 33791
Missing ZCTA: 903

========== CENSUS ZCTA RECONCILIATION ==========
Gazetteer unique ZCTAs: 33144
Relationship unique ZCTAs: 33791
ZCTAs in Gazetteer but NOT relationship: 204
ZCTAs in relationship but NOT Gazetteer: 851
ZCTAs present in BOTH: 32940

========== 436 CENTROID-RECOVERABLE PROVIDERS ==========
Provider rows: 436
Unique ZCTAs: 24
ZCTAs also in relationship: 0
ZCTAs NOT in relationship: 24
ZCTAs NOT in Gazetteer: 0


In [ ]:
# ============================================================
# STAGE 2C-14 — INSPECT CENTROID-AVAILABLE / COUNTY-MISSING
# ============================================================

centroid_only = (
    unmatched_centroid_check[
        unmatched_centroid_check["centroid_latitude"].notna()
    ]
    .groupby(
        [
            "state_clean",
            "zip_clean",
            "Provider Business Practice Location Address City Name",
            "centroid_latitude",
            "centroid_longitude"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="provider_count")
    .sort_values("provider_count", ascending=False)
)

print("========== CENTROID AVAILABLE, COUNTY MISSING ==========")

print(
    "Provider rows:",
    centroid_recoverable.shape[0]
)

print(
    "Unique ZIPs:",
    centroid_only["zip_clean"].nunique()
)

display(centroid_only.head(100))

========== CENTROID AVAILABLE, COUNTY MISSING ==========
Provider rows: 436
Unique ZIPs: 24


,state_clean,zip_clean,Provider Business Practice Location Address City Name,centroid_latitude,centroid_longitude,provider_count
25,TX,76508,TEMPLE,31.077568,-97.364064,266
5,DC,20593,WASHINGTON,38.866713,-77.010187,72
11,MA,01199,SPRINGFIELD,42.120563,-72.604468,23
4,DC,20307,WASHINGTON,38.974966,-77.030471,19
15,NY,11425,JAMAICA,40.607719,-74.023923,9
16,NY,11425,ST. ALBANS,40.607719,-74.023923,8
22,PR,00936,SAN JUAN,18.395463,-66.073772,6
27,VA,23187,WILLIAMSBURG,37.268856,-76.720941,5
19,PA,18427,HAMLIN,41.404977,-75.406897,4
23,TN,37240,NASHVILLE,36.144893,-86.805471,2


In [ ]:
# ============================================================
# STAGE 2C-15 — BUILD COMPLETE UNRESOLVED ZIP INVENTORY
# ============================================================

# All providers that failed the Census ZCTA -> county lookup
unresolved = geo_ready[
    geo_ready["county_fips"].isna()
].copy()

# Determine whether a Census ZCTA centroid exists
unresolved["has_zcta_centroid"] = (
    unresolved["zcta_clean"]
    .isin(zcta_centroids["zcta_clean"])
)

# Summarize at ZIP level
unresolved_zip_inventory = (
    unresolved
    .groupby(
        [
            "state_clean",
            "zip_clean",
            "Provider Business Practice Location Address City Name",
            "has_zcta_centroid"
        ],
        dropna=False
    )
    .agg(
        provider_count=("NPI", "count"),
        taxonomy_count=(
            "Healthcare Provider Taxonomy Code_1",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "provider_count",
        ascending=False
    )
)

print("========== UNRESOLVED ZIP INVENTORY ==========")

print(
    "Unresolved provider rows:",
    len(unresolved)
)

print(
    "Unresolved unique ZIPs:",
    unresolved["zip_clean"].nunique()
)

print(
    "ZIPs with Census centroid:",
    unresolved_zip_inventory[
        unresolved_zip_inventory["has_zcta_centroid"]
    ]["zip_clean"].nunique()
)

print(
    "ZIPs without Census centroid:",
    unresolved_zip_inventory[
        ~unresolved_zip_inventory["has_zcta_centroid"]
    ]["zip_clean"].nunique()
)

print("\nProvider rows by resolution status:")

display(
    unresolved.groupby(
        "has_zcta_centroid"
    ).size().rename("provider_count")
)

print("\nTop unresolved ZIPs:")

display(
    unresolved_zip_inventory.head(100)
)

========== UNRESOLVED ZIP INVENTORY ==========
Unresolved provider rows: 4774
Unresolved unique ZIPs: 650
ZIPs with Census centroid: 24
ZIPs without Census centroid: 626

Provider rows by resolution status:


,provider_count
has_zcta_centroid,
False,4338
True,436



Top unresolved ZIPs:


,state_clean,zip_clean,Provider Business Practice Location Address City Name,has_zcta_centroid,provider_count,taxonomy_count
485,OH,44195,CLEVELAND,False,577,90
382,NC,27157,WINSTON SALEM,False,384,92
315,MA,01655,WORCESTER,False,269,58
638,TX,76508,TEMPLE,True,266,70
144,CA,94143,SAN FRANCISCO,False,228,55
...,...,...,...,...,...,...
568,PR,00929,SAN JUAN,False,6,6
524,PA,18765,WILKES BARRE,False,6,5
109,AZ,85287,TEMPE,False,6,4
562,PR,00919,HATO REY,False,5,5


In [ ]:
# ============================================================
# STAGE 2C-16 — UNRESOLVED ZIPs BY STATE
# ============================================================

unresolved_state_summary = (
    unresolved
    .groupby(
        ["state_clean", "has_zcta_centroid"],
        dropna=False
    )
    .size()
    .reset_index(name="provider_count")
    .sort_values(
        "provider_count",
        ascending=False
    )
)

display(unresolved_state_summary)

,state_clean,has_zcta_centroid,provider_count
49,OH,False,619
40,NC,False,487
29,MA,False,348
9,CA,False,294
12,CT,False,271
...,...,...,...
7,AR,True,1
11,CO,True,1
19,HI,False,1
21,IA,True,1


In [ ]:
# ============================================================
# STAGE 2C-17 — PREPARE UNRESOLVED PROVIDER ADDRESSES
# FOR CENSUS GEOCODING
# ============================================================

# Providers that have neither a Census ZCTA centroid
# nor a county from the Census ZCTA relationship.
geocode_candidates = unresolved[
    ~unresolved["has_zcta_centroid"]
].copy()

print("========== GEOCODING CANDIDATES ==========")

print(
    "Provider rows:",
    len(geocode_candidates)
)

print(
    "Unique NPIs:",
    geocode_candidates["NPI"].nunique()
)

# ------------------------------------------------------------
# Build a normalized address key
# ------------------------------------------------------------

address_cols = [
    "Provider First Line Business Practice Location Address",
    "Provider Second Line Business Practice Location Address",
    "Provider Business Practice Location Address City Name",
    "state_clean",
    "zip_clean"
]

for col in address_cols:
    geocode_candidates[col] = (
        geocode_candidates[col]
        .astype("string")
        .str.strip()
    )

geocode_candidates["street_1"] = (
    geocode_candidates[
        "Provider First Line Business Practice Location Address"
    ]
)

geocode_candidates["street_2"] = (
    geocode_candidates[
        "Provider Second Line Business Practice Location Address"
    ]
)

geocode_candidates["city_geocode"] = (
    geocode_candidates[
        "Provider Business Practice Location Address City Name"
    ]
)

geocode_candidates["state_geocode"] = (
    geocode_candidates["state_clean"]
)

geocode_candidates["zip_geocode"] = (
    geocode_candidates["zip_clean"]
)

# Combined address string for inspection
geocode_candidates["full_address"] = (
    geocode_candidates["street_1"].fillna("")
    + " "
    + geocode_candidates["street_2"].fillna("")
    + ", "
    + geocode_candidates["city_geocode"].fillna("")
    + ", "
    + geocode_candidates["state_geocode"].fillna("")
    + " "
    + geocode_candidates["zip_geocode"].fillna("")
).str.replace(
    r"\s+",
    " ",
    regex=True
).str.strip()

# ------------------------------------------------------------
# Deduplicate exact practice addresses
# ------------------------------------------------------------

address_inventory = (
    geocode_candidates
    .groupby(
        [
            "street_1",
            "street_2",
            "city_geocode",
            "state_geocode",
            "zip_geocode"
        ],
        dropna=False
    )
    .agg(
        provider_count=("NPI", "count"),
        npi_count=("NPI", "nunique")
    )
    .reset_index()
    .sort_values(
        "provider_count",
        ascending=False
    )
)

print("\n========== ADDRESS DEDUPLICATION ==========")

print(
    "Provider rows requiring geocoding:",
    len(geocode_candidates)
)

print(
    "Unique exact practice addresses:",
    len(address_inventory)
)

print(
    "Potential request reduction:",
    round(
        (1 - len(address_inventory) / len(geocode_candidates))
        * 100,
        2
    ),
    "%"
)

print("\nTop repeated unresolved addresses:")

display(
    address_inventory.head(50)
)

========== GEOCODING CANDIDATES ==========
Provider rows: 4338
Unique NPIs: 4338

========== ADDRESS DEDUPLICATION ==========
Provider rows requiring geocoding: 4338
Unique exact practice addresses: 1968
Potential request reduction: 54.63 %

Top repeated unresolved addresses:


,street_1,street_2,city_geocode,state_geocode,zip_geocode,provider_count,npi_count
1267,9500 EUCLID AVE,<NA>,CLEVELAND,OH,44195,503,503
1612,MEDICAL CENTER BLVD,<NA>,WINSTON SALEM,NC,27157,356,356
604,263 FARMINGTON AVE,<NA>,FARMINGTON,CT,06030,118,118
992,55 LAKE AVE N,<NA>,WORCESTER,MA,01655,92,92
1487,ELM AND CARLTON ST,<NA>,BUFFALO,NY,14263,84,84
731,400 PARNASSUS AVE,<NA>,SAN FRANCISCO,CA,94143,70,70
871,505 PARNASSUS AVE,<NA>,SAN FRANCISCO,CA,94143,54,54
77,1 MEDICAL CENTER DR,<NA>,LEBANON,NH,03756,48,48
1298,988102 NEBRASKA MEDICAL CTR,<NA>,OMAHA,NE,68198,45,45
1082,701 N 1ST ST,<NA>,SPRINGFIELD,IL,62781,38,38


In [ ]:
# ============================================================
# STAGE 2C-18 — ADDRESS QUALITY CHECK
# ============================================================

address_quality = pd.DataFrame({
    "field": [
        "street_1",
        "street_2",
        "city",
        "state",
        "zip"
    ],
    "missing_count": [
        geocode_candidates["street_1"].isna().sum(),
        geocode_candidates["street_2"].isna().sum(),
        geocode_candidates["city_geocode"].isna().sum(),
        geocode_candidates["state_geocode"].isna().sum(),
        geocode_candidates["zip_geocode"].isna().sum()
    ]
})

address_quality["missing_pct"] = (
    address_quality["missing_count"]
    / len(geocode_candidates)
    * 100
)

display(address_quality)

,field,missing_count,missing_pct
0,street_1,0,0.000000
1,street_2,2962,68.280314
2,city,0,0.000000
3,state,0,0.000000
4,zip,0,0.000000


In [ ]:
# ============================================================
# STAGE 2C-19 — BUILD DEDUPLICATED GEOCODING REQUEST TABLE
# ============================================================

import re

geo_requests = geocode_candidates.copy()

# ------------------------------------------------------------
# Conservative address normalization
# ------------------------------------------------------------

def normalize_address(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).upper().strip()

    # Normalize whitespace
    value = re.sub(r"\s+", " ", value)

    # Normalize common punctuation only
    value = value.replace(".", "")
    value = value.replace(",", "")

    return value


geo_requests["street_1_norm"] = (
    geo_requests["street_1"]
    .map(normalize_address)
)

geo_requests["street_2_norm"] = (
    geo_requests["street_2"]
    .map(normalize_address)
)

geo_requests["city_norm"] = (
    geo_requests["city_geocode"]
    .map(normalize_address)
)

geo_requests["state_norm"] = (
    geo_requests["state_geocode"]
    .astype("string")
    .str.upper()
    .str.strip()
)

geo_requests["zip_norm"] = (
    geo_requests["zip_geocode"]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)

# ------------------------------------------------------------
# Deduplicate on normalized full practice address
# ------------------------------------------------------------

geo_request_table = (
    geo_requests
    .groupby(
        [
            "street_1_norm",
            "street_2_norm",
            "city_norm",
            "state_norm",
            "zip_norm"
        ],
        dropna=False
    )
    .agg(
        provider_count=("NPI", "count"),
        original_street_1=("street_1", "first"),
        original_street_2=("street_2", "first"),
        original_city=("city_geocode", "first"),
        original_state=("state_geocode", "first"),
        original_zip=("zip_geocode", "first")
    )
    .reset_index()
)

print("========== NORMALIZED GEOCODING REQUEST TABLE ==========")

print(
    "Original provider rows:",
    len(geocode_candidates)
)

print(
    "Exact address rows:",
    len(address_inventory)
)

print(
    "Normalized unique addresses:",
    len(geo_request_table)
)

print(
    "Additional reduction from normalization:",
    round(
        (
            1
            - len(geo_request_table)
            / len(address_inventory)
        ) * 100,
        2
    ),
    "%"
)

print(
    "Total provider coverage:",
    geo_request_table["provider_count"].sum()
)

display(
    geo_request_table
    .sort_values(
        "provider_count",
        ascending=False
    )
    .head(100)
)

========== NORMALIZED GEOCODING REQUEST TABLE ==========
Original provider rows: 4338
Exact address rows: 1968
Normalized unique addresses: 1949
Additional reduction from normalization: 0.97 %
Total provider coverage: 4338


,street_1_norm,street_2_norm,city_norm,state_norm,zip_norm,provider_count,original_street_1,original_street_2,original_city,original_state,original_zip
1255,9500 EUCLID AVE,NaN,CLEVELAND,OH,44195,503,9500 EUCLID AVE,<NA>,CLEVELAND,OH,44195
1596,MEDICAL CENTER BLVD,NaN,WINSTON SALEM,NC,27157,359,MEDICAL CENTER BLVD.,<NA>,WINSTON SALEM,NC,27157
598,263 FARMINGTON AVE,NaN,FARMINGTON,CT,06030,118,263 FARMINGTON AVE,<NA>,FARMINGTON,CT,06030
982,55 LAKE AVE N,NaN,WORCESTER,MA,01655,92,55 LAKE AVE N,<NA>,WORCESTER,MA,01655
1474,ELM AND CARLTON ST,NaN,BUFFALO,NY,14263,84,ELM AND CARLTON ST,<NA>,BUFFALO,NY,14263
...,...,...,...,...,...,...,...,...,...,...,...
551,2555 JIMMY JOHNSON BLVD,NaN,PORT ARTHUR,TX,77643,4,2555 JIMMY JOHNSON BLVD,<NA>,PORT ARTHUR,TX,77643
215,1141 BEACH DR E,NaN,RETSIL,WA,98378,4,1141 BEACH DR E,<NA>,RETSIL,WA,98378
160,101 S NEWELL DRIVE,NaN,GAINESVILLE,FL,32611,4,101 S. NEWELL DRIVE,<NA>,GAINESVILLE,FL,32611
700,3700 WASHINGTON AVE,NaN,EVANSVILLE,IN,47750,4,3700 WASHINGTON AVE,<NA>,EVANSVILLE,IN,47750


In [ ]:
# ============================================================
# STAGE 2C-20 — GEOGRAPHIC SCOPE BREAKDOWN
# ============================================================

geocode_scope = (
    geo_request_table
    .groupby("state_norm")
    .agg(
        unique_addresses=("state_norm", "size"),
        provider_count=("provider_count", "sum")
    )
    .reset_index()
    .sort_values(
        "provider_count",
        ascending=False
    )
)

print("========== GEOCODING SCOPE BY STATE ==========")

display(geocode_scope)

========== GEOCODING SCOPE BY STATE ==========


,state_norm,unique_addresses,provider_count
39,OH,100,619
31,NC,85,487
23,MA,112,348
7,CA,124,294
9,CT,113,271
42,PA,83,187
38,NY,49,182
51,WA,110,177
1,AE,113,133
43,PR,127,131


In [ ]:
# ============================================================
# STAGE 2C-21 — IDENTIFY SPECIAL / TERRITORY JURISDICTIONS
# ============================================================

special_states = [
    "PR", "VI", "GU",
    "AE", "AP", "AA"
]

special_geocode = geo_request_table[
    geo_request_table["state_norm"].isin(special_states)
].copy()

ordinary_geocode = geo_request_table[
    ~geo_request_table["state_norm"].isin(special_states)
].copy()

print("========== SPECIAL JURISDICTIONS ==========")

print(
    "Unique addresses:",
    len(special_geocode)
)

print(
    "Provider rows:",
    special_geocode["provider_count"].sum()
)

display(
    special_geocode.sort_values(
        "provider_count",
        ascending=False
    )
)

print("\n========== ORDINARY US JURISDICTIONS ==========")

print(
    "Unique addresses:",
    len(ordinary_geocode)
)

print(
    "Provider rows:",
    ordinary_geocode["provider_count"].sum()
)

========== SPECIAL JURISDICTIONS ==========
Unique addresses: 333
Provider rows: 366


,street_1_norm,street_2_norm,city_norm,state_norm,zip_norm,provider_count,original_street_1,original_street_2,original_city,original_state,original_zip
1700,PSC 475 BOX 1,NaN,FPO,AP,96350,6,PSC 475 BOX 1,<NA>,FPO,AP,96350
1819,UNIT 33100,NaN,APO,AE,09180,6,UNIT 33100,<NA>,APO,AE,09180
1820,UNIT 33100 BOX LANDSTUHL,NaN,APO,AE,09180,5,UNIT 33100 BOX LANDSTUHL,<NA>,APO,AE,09180
1753,RAF LAKENHEATH 48 MDG/SGHC,UNIT 5115,APO,AE,09461,3,RAF LAKENHEATH 48 MDG/SGHC,UNIT 5115,APO,AE,09461
1539,LANDSTUHL REGIONAL MEDICAL CENTER,UNIT 33100,APO,AE,09180,3,LANDSTUHL REGIONAL MEDICAL CENTER,UNIT 33100,APO,AE,09180
...,...,...,...,...,...,...,...,...,...,...,...
1379,CARR #2 KM 1293 BO VICTORIA,OFICINA #13 AGUADILLA MEDICAL SERVICES,AGUADILLA,PR,00605,1,CARR #2 KM 129.3 BO VICTORIA,OFICINA #13 AGUADILLA MEDICAL SERVICES,AGUADILLA,PR,00605
1378,CARR #14,ANEXO HOSP ONCOLOGICO,PONCE,PR,00732,1,CARR #14,ANEXO HOSP ONCOLOGICO,PONCE,PR,00732
1375,CAMP SCHWAB BMC,3D MED BN 3D MLG C CO,FPO,AP,96604,1,CAMP SCHWAB BMC,"3D MED BN, 3D MLG, C CO",FPO,AP,96604
1374,CAMP LESTER NAVAL HOSPITAL OKINAWA,PSC 482,FPO,AP,96362,1,CAMP LESTER NAVAL HOSPITAL OKINAWA,PSC 482,FPO,AP,96362



========== ORDINARY US JURISDICTIONS ==========
Unique addresses: 1616
Provider rows: 3972


In [ ]:
# ============================================================
# STAGE 2C-22 — FINAL ORDINARY ADDRESS VALIDATION
# ============================================================

ordinary_validation = pd.DataFrame({
    "field": [
        "street_1",
        "street_2",
        "city",
        "state",
        "zip"
    ],
    "missing_count": [
        ordinary_geocode["street_1_norm"].isna().sum(),
        ordinary_geocode["street_2_norm"].isna().sum(),
        ordinary_geocode["city_norm"].isna().sum(),
        ordinary_geocode["state_norm"].isna().sum(),
        ordinary_geocode["zip_norm"].isna().sum()
    ]
})

ordinary_validation["missing_pct"] = (
    ordinary_validation["missing_count"]
    / len(ordinary_geocode)
    * 100
)

print("========== ORDINARY ADDRESS VALIDATION ==========")

display(ordinary_validation)

print("Unique ordinary addresses:", len(ordinary_geocode))
print(
    "Provider rows represented:",
    ordinary_geocode["provider_count"].sum()
)

print(
    "States represented:",
    ordinary_geocode["state_norm"].nunique()
)

print(
    "Invalid ZIP length:",
    (
        ordinary_geocode["zip_norm"]
        .astype("string")
        .str.len()
        .ne(5)
    ).sum()
)

========== ORDINARY ADDRESS VALIDATION ==========


,field,missing_count,missing_pct
0,street_1,0,0.000000
1,street_2,711,43.997525
2,city,0,0.000000
3,state,0,0.000000
4,zip,0,0.000000


Unique ordinary addresses: 1616
Provider rows represented: 3972
States represented: 49
Invalid ZIP length: 0


In [ ]:
# ============================================================
# STAGE 2C-23 — SAVE GEOCODING REQUEST CHECKPOINT
# ============================================================

geocode_request_path = (
    "/content/drive/MyDrive/data/geographic_reference/"
    "nppes_geocoding_requests_ordinary_us.csv"
)

ordinary_geocode.to_csv(
    geocode_request_path,
    index=False
)

print("Saved:", geocode_request_path)
print("Rows:", len(ordinary_geocode))

Saved: /content/drive/MyDrive/data/geographic_reference/nppes_geocoding_requests_ordinary_us.csv
Rows: 1616


In [ ]:
# ============================================================
# STAGE 2C-24 — CONTROLLED CENSUS GEOCODER TEST BATCH
# ============================================================

# Select a diverse test set:
# - 10 highest-volume addresses
# - 10 lower-volume addresses
# - 5 additional addresses from different states

top_addresses = (
    ordinary_geocode
    .sort_values("provider_count", ascending=False)
    .head(10)
)

bottom_addresses = (
    ordinary_geocode
    .sort_values("provider_count", ascending=True)
    .head(10)
)

state_diverse = (
    ordinary_geocode
    .sort_values("provider_count", ascending=False)
    .drop_duplicates("state_norm")
    .head(5)
)

geocoder_test = (
    pd.concat(
        [top_addresses, bottom_addresses, state_diverse],
        ignore_index=True
    )
    .drop_duplicates(
        subset=[
            "street_1_norm",
            "street_2_norm",
            "city_norm",
            "state_norm",
            "zip_norm"
        ]
    )
    .reset_index(drop=True)
)

print("========== CENSUS GEOCODER TEST BATCH ==========")

print("Test addresses:", len(geocoder_test))
print(
    "Providers represented:",
    geocoder_test["provider_count"].sum()
)

display(
    geocoder_test[
        [
            "street_1_norm",
            "street_2_norm",
            "city_norm",
            "state_norm",
            "zip_norm",
            "provider_count"
        ]
    ]
)

========== CENSUS GEOCODER TEST BATCH ==========
Test addresses: 20
Providers represented: 1421


,street_1_norm,street_2_norm,city_norm,state_norm,zip_norm,provider_count
0,9500 EUCLID AVE,NaN,CLEVELAND,OH,44195,503
1,MEDICAL CENTER BLVD,NaN,WINSTON SALEM,NC,27157,359
2,263 FARMINGTON AVE,NaN,FARMINGTON,CT,06030,118
3,55 LAKE AVE N,NaN,WORCESTER,MA,01655,92
4,ELM AND CARLTON ST,NaN,BUFFALO,NY,14263,84
5,400 PARNASSUS AVE,NaN,SAN FRANCISCO,CA,94143,70
6,505 PARNASSUS AVE,NaN,SAN FRANCISCO,CA,94143,54
7,1 MEDICAL CENTER DR,NaN,LEBANON,NH,03756,48
8,988102 NEBRASKA MEDICAL CTR,NaN,OMAHA,NE,68198,45
9,701 N 1ST ST,NaN,SPRINGFIELD,IL,62781,38


In [ ]:
# ============================================================
# STAGE 2C-25 — CENSUS GEOCODER CONTROLLED TEST
# ============================================================

import requests
import pandas as pd
import time

CENSUS_GEOCODER_URL = (
    "https://geocoding.geo.census.gov/geocoder/locations/address"
)

test_results = []

for idx, row in geocoder_test.iterrows():

    params = {
        "street": row["street_1_norm"],
        "city": row["city_norm"],
        "state": row["state_norm"],
        "zip": row["zip_norm"],
        "benchmark": "Public_AR_Current",
        "format": "json"
    }

    try:
        response = requests.get(
            CENSUS_GEOCODER_URL,
            params=params,
            timeout=30
        )

        response.raise_for_status()
        data = response.json()

        matches = (
            data
            .get("result", {})
            .get("addressMatches", [])
        )

        if matches:

            match = matches[0]

            coordinates = match.get(
                "coordinates",
                {}
            )

            test_results.append({
                "request_id": idx,
                "street_1": row["street_1_norm"],
                "street_2": row["street_2_norm"],
                "city": row["city_norm"],
                "state": row["state_norm"],
                "zip": row["zip_norm"],
                "provider_count": row["provider_count"],

                "match_count": len(matches),

                "matched_address": match.get(
                    "matchedAddress"
                ),

                "longitude": coordinates.get(
                    "x"
                ),

                "latitude": coordinates.get(
                    "y"
                ),

                "tiger_line_id": match.get(
                    "tigerLine",
                    {}
                ).get(
                    "tigerLineId"
                ),

                "side": match.get(
                    "tigerLine",
                    {}
                ).get(
                    "side"
                ),

                "geographies": match.get(
                    "addressComponents",
                    {}
                )
            })

        else:

            test_results.append({
                "request_id": idx,
                "street_1": row["street_1_norm"],
                "street_2": row["street_2_norm"],
                "city": row["city_norm"],
                "state": row["state_norm"],
                "zip": row["zip_norm"],
                "provider_count": row["provider_count"],

                "match_count": 0,
                "matched_address": None,
                "longitude": None,
                "latitude": None,
                "tiger_line_id": None,
                "side": None,
                "geographies": None
            })

    except Exception as e:

        test_results.append({
            "request_id": idx,
            "street_1": row["street_1_norm"],
            "street_2": row["street_2_norm"],
            "city": row["city_norm"],
            "state": row["state_norm"],
            "zip": row["zip_norm"],
            "provider_count": row["provider_count"],

            "match_count": None,
            "matched_address": None,
            "longitude": None,
            "latitude": None,
            "tiger_line_id": None,
            "side": None,
            "geographies": None,
            "error": str(e)
        })

    # Be polite to the public API
    time.sleep(0.1)


census_test_results = pd.DataFrame(test_results)

print("========== CENSUS GEOCODER TEST RESULTS ==========")

print(
    "Requests:",
    len(census_test_results)
)

print(
    "Matched:",
    (
        census_test_results["match_count"]
        .fillna(0)
        .gt(0)
        .sum()
    )
)

print(
    "Unmatched:",
    (
        census_test_results["match_count"]
        .fillna(0)
        .eq(0)
        .sum()
    )
)

print(
    "Coordinates returned:",
    census_test_results[
        ["latitude", "longitude"]
    ].notna().all(axis=1).sum()
)

display(
    census_test_results[
        [
            "request_id",
            "street_1",
            "street_2",
            "city",
            "state",
            "zip",
            "provider_count",
            "match_count",
            "matched_address",
            "latitude",
            "longitude",
            "tiger_line_id",
            "side"
        ]
    ]
)

========== CENSUS GEOCODER TEST RESULTS ==========
Requests: 20
Matched: 16
Unmatched: 4
Coordinates returned: 16


,request_id,street_1,street_2,city,state,zip,provider_count,match_count,matched_address,latitude,longitude,tiger_line_id,side
0,0,9500 EUCLID AVE,NaN,CLEVELAND,OH,44195,503,1,"9500 EUCLID AVE, CLEVELAND, OH, 44195",41.503346,-81.622884,638278962,L
1,1,MEDICAL CENTER BLVD,NaN,WINSTON SALEM,NC,27157,359,0,None,NaN,NaN,None,None
2,2,263 FARMINGTON AVE,NaN,FARMINGTON,CT,06030,118,0,None,NaN,NaN,None,None
3,3,55 LAKE AVE N,NaN,WORCESTER,MA,01655,92,1,"55 N LAKE AVE, WORCESTER, MA, 01655",42.277713,-71.759328,40100681,L
4,4,ELM AND CARLTON ST,NaN,BUFFALO,NY,14263,84,1,"ELM ST & CARLTON ST, BUFFALO, NY, 14203",42.898510,-78.865003,,
5,5,400 PARNASSUS AVE,NaN,SAN FRANCISCO,CA,94143,70,1,"400 PARNASSUS AVE, SAN FRANCISCO, CA, 94143",37.763598,-122.457630,192282295,R
6,6,505 PARNASSUS AVE,NaN,SAN FRANCISCO,CA,94143,54,1,"505 PARNASSUS AVE, SAN FRANCISCO, CA, 94143",37.762922,-122.459584,192282295,L
7,7,1 MEDICAL CENTER DR,NaN,LEBANON,NH,03756,48,0,None,NaN,NaN,None,None
8,8,988102 NEBRASKA MEDICAL CTR,NaN,OMAHA,NE,68198,45,0,None,NaN,NaN,None,None
9,9,701 N 1ST ST,NaN,SPRINGFIELD,IL,62781,38,1,"701 N 1ST ST, SPRINGFIELD, IL, 62781",39.808626,-89.654723,640035722,L


In [ ]:
# ============================================================
# STAGE 2C-26 — PRODUCTION CENSUS GEOCODER
# ============================================================

import requests
import pandas as pd
import time
from pathlib import Path

CENSUS_GEOCODER_URL = (
    "https://geocoding.geo.census.gov/geocoder/locations/address"
)

production_results = []

for idx, row in ordinary_geocode.iterrows():

    params = {
        "street": row["street_1_norm"],
        "city": row["city_norm"],
        "state": row["state_norm"],
        "zip": row["zip_norm"],
        "benchmark": "Public_AR_Current",
        "format": "json"
    }

    result = {
        "request_id": idx,

        "street_1_norm": row["street_1_norm"],
        "street_2_norm": row["street_2_norm"],
        "city_norm": row["city_norm"],
        "state_norm": row["state_norm"],
        "zip_norm": row["zip_norm"],

        "provider_count": row["provider_count"],

        "geocoding_source": "Census_Geocoder",
        "geocoding_status": "error",

        "geocoding_match_count": None,

        "geocoded_matched_address": None,

        "provider_latitude": None,
        "provider_longitude": None,

        "geocoded_tiger_line_id": None,
        "geocoded_side": None
    }

    try:

        response = requests.get(
            CENSUS_GEOCODER_URL,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        data = response.json()

        matches = (
            data
            .get("result", {})
            .get("addressMatches", [])
        )

        result["geocoding_match_count"] = len(matches)

        if matches:

            match = matches[0]

            coordinates = match.get(
                "coordinates",
                {}
            )

            result["geocoding_status"] = "matched"

            result["geocoded_matched_address"] = (
                match.get("matchedAddress")
            )

            result["provider_longitude"] = (
                coordinates.get("x")
            )

            result["provider_latitude"] = (
                coordinates.get("y")
            )

            tiger_line = match.get(
                "tigerLine",
                {}
            )

            result["geocoded_tiger_line_id"] = (
                tiger_line.get("tigerLineId")
            )

            result["geocoded_side"] = (
                tiger_line.get("side")
            )

        else:

            result["geocoding_status"] = "unmatched"

    except Exception as e:

        result["geocoding_status"] = "error"
        result["geocoding_error"] = str(e)

    production_results.append(result)

    # Small delay between requests
    time.sleep(0.1)


census_geocoding_results = pd.DataFrame(
    production_results
)

print("========== PRODUCTION GEOCODING RESULTS ==========")

print(
    "Requests:",
    len(census_geocoding_results)
)

print(
    "\nStatus distribution:"
)

display(
    census_geocoding_results[
        "geocoding_status"
    ]
    .value_counts(dropna=False)
)

print(
    "\nCoordinate coverage:",
    census_geocoding_results[
        ["provider_latitude", "provider_longitude"]
    ]
    .notna()
    .all(axis=1)
    .sum()
)

print(
    "\nProvider rows represented:",
    census_geocoding_results[
        "provider_count"
    ].sum()
)

print(
    "\nProvider rows with coordinates:"
)

print(
    census_geocoding_results.loc[
        census_geocoding_results[
            "provider_latitude"
        ].notna(),
        "provider_count"
    ].sum()
)

display(
    census_geocoding_results.head(20)
)


========== PRODUCTION GEOCODING RESULTS ==========
Requests: 1616

Status distribution:


,count
geocoding_status,
matched,925
unmatched,691



Coordinate coverage: 925

Provider rows represented: 3972

Provider rows with coordinates:
2426


,request_id,street_1_norm,street_2_norm,city_norm,state_norm,zip_norm,provider_count,geocoding_source,geocoding_status,geocoding_match_count,geocoded_matched_address,provider_latitude,provider_longitude,geocoded_tiger_line_id,geocoded_side
0,1,# L-3539,NaN,COLUMBUS,OH,43260,1,Census_Geocoder,unmatched,0,None,NaN,NaN,None,None
1,4,000 UNIVERSITY DRIVE C,132 Y-A,PITTSBURGH,PA,15240,1,Census_Geocoder,unmatched,0,None,NaN,NaN,None,None
2,5,0610 TERRAPIN TRAIL,UNIVERSITY OF MARYLAND,COLLEGE PARK,MD,20741,1,Census_Geocoder,unmatched,0,None,NaN,NaN,None,None
3,6,1 AMERICAN SQ,SUITE 185,INDIANAPOLIS,IN,46282,1,Census_Geocoder,unmatched,0,None,NaN,NaN,None,None
4,7,1 AMERICAN SQ STE 185,NaN,INDIANAPOLIS,IN,46282,1,Census_Geocoder,unmatched,0,None,NaN,NaN,None,None
5,8,1 AYERS CIRCLE,BLDG H-1 NAVAL BRANCH HEALTH CLINIC,PORTSMOUTH,NH,03804,1,Census_Geocoder,unmatched,0,None,NaN,NaN,None,None
6,9,1 AYRES CIRCLE,NAVAL BRANCH HEALTH CLINIC BUILDING H-1,PORTSMOUTH,NH,03804,1,Census_Geocoder,unmatched,0,None,NaN,NaN,None,None
7,10,1 ELIZABETH PL,4TH FLOOR,DAYTON,OH,45408,1,Census_Geocoder,unmatched,0,None,NaN,NaN,None,None
8,11,1 ELIZABETH PL,SUITE 190,DAYTON,OH,45408,1,Census_Geocoder,unmatched,0,None,NaN,NaN,None,None
9,12,1 ELIZABETH PLACE,SUITE 300,DAYTON,OH,45408,1,Census_Geocoder,unmatched,0,None,NaN,NaN,None,None


In [ ]:
# ============================================================
# RECOVER STAGE 2C-26 RESULTS WITHOUT API CALLS
# ============================================================

if "census_geocoding_results" in globals():

    print("census_geocoding_results already exists.")
    print("Shape:", census_geocoding_results.shape)

elif "production_results" in globals():

    census_geocoding_results = pd.DataFrame(
        production_results
    )

    print("Recovered from production_results.")
    print("Shape:", census_geocoding_results.shape)

else:

    print("PRODUCTION RESULTS ARE NOT CURRENTLY IN MEMORY.")
    print("Do NOT run another geocoding cell yet.")
    print("The Colab runtime has likely been reset.")

census_geocoding_results already exists.
Shape: (1616, 15)


In [ ]:
# ============================================================
# STAGE 2C-26A — PERSIST PRODUCTION GEOCODING RESULTS
# ============================================================

GEOCODE_RESULTS_PATH = (
    "/content/drive/MyDrive/data/geographic_reference/"
    "nppes_census_geocoding_results_ordinary_us.csv"
)

census_geocoding_results.to_csv(
    GEOCODE_RESULTS_PATH,
    index=False
)

print("Saved:")
print(GEOCODE_RESULTS_PATH)

print(
    "Rows:",
    len(census_geocoding_results)
)

Saved:
/content/drive/MyDrive/data/geographic_reference/nppes_census_geocoding_results_ordinary_us.csv
Rows: 1616


In [ ]:
# ============================================================
# STAGE 2C-27 — ANALYZE UNMATCHED CENSUS GEOCODER ADDRESSES
# ============================================================

unmatched_geocode = (
    census_geocoding_results[
        census_geocoding_results["geocoding_status"] == "unmatched"
    ]
    .copy()
)

print("========== UNMATCHED GEOCODING INVENTORY ==========")

print(
    "Unmatched unique addresses:",
    len(unmatched_geocode)
)

print(
    "Providers represented:",
    unmatched_geocode["provider_count"].sum()
)

print(
    "States represented:",
    unmatched_geocode["state_norm"].nunique()
)

print("\nTop states by unresolved provider count:")

display(
    unmatched_geocode
    .groupby("state_norm")
    .agg(
        unique_addresses=("request_id", "count"),
        provider_count=("provider_count", "sum")
    )
    .reset_index()
    .sort_values("provider_count", ascending=False)
    .head(25)
)

print("\nHighest-volume unresolved addresses:")

display(
    unmatched_geocode[
        [
            "street_1_norm",
            "street_2_norm",
            "city_norm",
            "state_norm",
            "zip_norm",
            "provider_count"
        ]
    ]
    .sort_values("provider_count", ascending=False)
    .head(100)
)

========== UNMATCHED GEOCODING INVENTORY ==========
Unmatched unique addresses: 691
Providers represented: 1546
States represented: 47

Top states by unresolved provider count:


,state_norm,unique_addresses,provider_count
26,NC,58,446
6,CT,58,188
29,NH,65,119
43,WA,65,82
28,NE,18,73
23,MO,10,60
41,VA,32,56
38,TN,37,47
13,IL,24,44
9,FL,33,42



Highest-volume unresolved addresses:


,street_1_norm,street_2_norm,city_norm,state_norm,zip_norm,provider_count
1415,MEDICAL CENTER BLVD,NaN,WINSTON SALEM,NC,27157,359
579,263 FARMINGTON AVE,NaN,FARMINGTON,CT,06030,118
72,1 MEDICAL CENTER DR,NaN,LEBANON,NH,03756,48
1234,988102 NEBRASKA MEDICAL CTR,NaN,OMAHA,NE,68198,45
1444,ONE HOSPITAL DR,NaN,COLUMBIA,MO,65212,25
...,...,...,...,...,...,...
1377,MADIGAN ARMY MEDICAL CENTER,BUILDING 9040 FITZSIMMONS DRIVE,TACOMA,WA,98431,1
1373,MADIGAN ARMY MEDICAL CENTER,9040 JACKSON AVE,TACOMA,WA,98431,1
1372,MADIGAN ARMY MEDICAL CENTER,9040 FITZSIMMONS DRIVE,TACOMA,WA,98431,1
1371,MADIGAN ARMY MEDICAL CE,9040 JACKSON AVE,TACOMA,WA,98431,1


In [ ]:
# ============================================================
# STAGE 2C-28 — BUILD GEOGRAPHIC ADDRESS KEYS
# ============================================================

unmatched_geocode = (
    census_geocoding_results[
        census_geocoding_results["geocoding_status"] == "unmatched"
    ]
    .copy()
)

# street_2 is intentionally excluded.
# It often contains suite, department, building, clinic,
# organizational, or other non-geographic information.

unmatched_geocode["geo_address_key"] = (
    unmatched_geocode["street_1_norm"].fillna("").astype(str)
    + " | "
    + unmatched_geocode["city_norm"].fillna("").astype(str)
    + " | "
    + unmatched_geocode["state_norm"].fillna("").astype(str)
    + " | "
    + unmatched_geocode["zip_norm"].fillna("").astype(str)
)

print("========== GEOGRAPHIC ADDRESS KEY ANALYSIS ==========")

print(
    "Original unmatched records:",
    len(unmatched_geocode)
)

print(
    "Unique geographic address keys:",
    unmatched_geocode["geo_address_key"].nunique()
)

print(
    "Providers represented:",
    unmatched_geocode["provider_count"].sum()
)

print(
    "Request reduction:",
    round(
        (
            1
            - unmatched_geocode["geo_address_key"].nunique()
            / len(unmatched_geocode)
        )
        * 100,
        2
    ),
    "%"
)

print("\nHighest-volume geographic addresses:")

display(
    unmatched_geocode[
        [
            "street_1_norm",
            "city_norm",
            "state_norm",
            "zip_norm",
            "provider_count"
        ]
    ]
    .groupby(
        [
            "street_1_norm",
            "city_norm",
            "state_norm",
            "zip_norm"
        ],
        dropna=False
    )
    .agg(
        provider_count=("provider_count", "sum"),
        address_variants=("street_1_norm", "size")
    )
    .reset_index()
    .sort_values("provider_count", ascending=False)
    .head(100)
)

========== GEOGRAPHIC ADDRESS KEY ANALYSIS ==========
Original unmatched records: 691
Unique geographic address keys: 508
Providers represented: 1546
Request reduction: 26.48 %

Highest-volume geographic addresses:


,street_1_norm,city_norm,state_norm,zip_norm,provider_count,address_variants
346,MEDICAL CENTER BLVD,WINSTON SALEM,NC,27157,364,6
104,263 FARMINGTON AVE,FARMINGTON,CT,06030,157,34
13,1 MEDICAL CENTER DR,LEBANON,NH,03756,95,44
229,988102 NEBRASKA MEDICAL CTR,OMAHA,NE,68198,45,1
49,1201 BROAD ROCK BLVD,RICHMOND,VA,23249,34,16
...,...,...,...,...,...,...
491,VA SALT LAKE CITY HEALTH CARE SYSTEM,SALT LAKE CITY,UT,84148,2,2
476,USS CARL VINSON CVN-70,FPO AE,VA,09566,1,1
507,WILMINGTON HOSPITAL,WILMINGTON,DE,19899,1,1
0,# L-3539,COLUMBUS,OH,43260,1,1


In [ ]:
# ============================================================
# STAGE 2C-29 — CREATE SECOND-PASS GEOGRAPHIC ADDRESS TABLE
# ============================================================

# One row per geographic address key.
# street_2 is deliberately excluded from the geographic key.

second_pass_addresses = (
    unmatched_geocode
    .groupby(
        [
            "street_1_norm",
            "city_norm",
            "state_norm",
            "zip_norm"
        ],
        dropna=False
    )
    .agg(
        provider_count=("provider_count", "sum"),
        address_variant_count=("provider_count", "size")
    )
    .reset_index()
)

# Recreate the key explicitly
second_pass_addresses["geo_address_key"] = (
    second_pass_addresses["street_1_norm"].fillna("").astype(str)
    + " | "
    + second_pass_addresses["city_norm"].fillna("").astype(str)
    + " | "
    + second_pass_addresses["state_norm"].fillna("").astype(str)
    + " | "
    + second_pass_addresses["zip_norm"].fillna("").astype(str)
)

# Flag potentially non-standard geographic strings.
# This is classification only — no records are removed.

second_pass_addresses["possible_nonstandard_address"] = (
    second_pass_addresses["street_1_norm"]
    .fillna("")
    .str.contains(
        r"\b(FPO|APO|PSC|UNIT|BOX|USS|CVN|HEALTH CARE SYSTEM|"
        r"MEDICAL CENTER|HOSPITAL|ARMY|NAVAL|AIR FORCE)\b",
        case=False,
        regex=True,
        na=False
    )
)

print("========== SECOND-PASS ADDRESS TABLE ==========")

print(
    "Unique geographic addresses:",
    len(second_pass_addresses)
)

print(
    "Providers represented:",
    second_pass_addresses["provider_count"].sum()
)

print(
    "Potentially non-standard/institutional:",
    second_pass_addresses[
        "possible_nonstandard_address"
    ].sum()
)

print(
    "Potentially ordinary:",
    (
        ~second_pass_addresses[
            "possible_nonstandard_address"
        ]
    ).sum()
)

print("\nHighest-volume second-pass addresses:")

display(
    second_pass_addresses
    .sort_values("provider_count", ascending=False)
    .head(100)
)

print("\nPotentially non-standard addresses:")

display(
    second_pass_addresses[
        second_pass_addresses[
            "possible_nonstandard_address"
        ]
    ]
    .sort_values("provider_count", ascending=False)
    .head(100)
)

========== SECOND-PASS ADDRESS TABLE ==========
Unique geographic addresses: 508
Providers represented: 1546
Potentially non-standard/institutional: 110
Potentially ordinary: 398

Highest-volume second-pass addresses:


/tmp/ipykernel_813/2345921995.py:43: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


,street_1_norm,city_norm,state_norm,zip_norm,provider_count,address_variant_count,geo_address_key,possible_nonstandard_address
346,MEDICAL CENTER BLVD,WINSTON SALEM,NC,27157,364,6,MEDICAL CENTER BLVD | WINSTON SALEM | NC | 27157,True
104,263 FARMINGTON AVE,FARMINGTON,CT,06030,157,34,263 FARMINGTON AVE | FARMINGTON | CT | 06030,False
13,1 MEDICAL CENTER DR,LEBANON,NH,03756,95,44,1 MEDICAL CENTER DR | LEBANON | NH | 03756,True
229,988102 NEBRASKA MEDICAL CTR,OMAHA,NE,68198,45,1,988102 NEBRASKA MEDICAL CTR | OMAHA | NE | 68198,False
49,1201 BROAD ROCK BLVD,RICHMOND,VA,23249,34,16,1201 BROAD ROCK BLVD | RICHMOND | VA | 23249,False
...,...,...,...,...,...,...,...,...
491,VA SALT LAKE CITY HEALTH CARE SYSTEM,SALT LAKE CITY,UT,84148,2,2,VA SALT LAKE CITY HEALTH CARE SYSTEM | SALT LA...,True
476,USS CARL VINSON CVN-70,FPO AE,VA,09566,1,1,USS CARL VINSON CVN-70 | FPO AE | VA | 09566,True
507,WILMINGTON HOSPITAL,WILMINGTON,DE,19899,1,1,WILMINGTON HOSPITAL | WILMINGTON | DE | 19899,True
0,# L-3539,COLUMBUS,OH,43260,1,1,# L-3539 | COLUMBUS | OH | 43260,False



Potentially non-standard addresses:


,street_1_norm,city_norm,state_norm,zip_norm,provider_count,address_variant_count,geo_address_key,possible_nonstandard_address
346,MEDICAL CENTER BLVD,WINSTON SALEM,NC,27157,364,6,MEDICAL CENTER BLVD | WINSTON SALEM | NC | 27157,True
13,1 MEDICAL CENTER DR,LEBANON,NH,03756,95,44,1 MEDICAL CENTER DR | LEBANON | NH | 03756,True
370,ONE HOSPITAL DR,COLUMBIA,MO,65212,25,1,ONE HOSPITAL DR | COLUMBIA | MO | 65212,True
11,1 HOSPITAL DR,COLUMBIA,MO,65212,24,1,1 HOSPITAL DR | COLUMBIA | MO | 65212,True
336,MADIGAN ARMY MEDICAL CENTER,TACOMA,WA,98431,16,10,MADIGAN ARMY MEDICAL CENTER | TACOMA | WA | 98431,True
...,...,...,...,...,...,...,...,...
463,UNIVERSITY OF UTAH HOSPITAL CLINICS DEPT OF OB...,SALT LAKE CITY,UT,84132,1,1,UNIVERSITY OF UTAH HOSPITAL CLINICS DEPT OF OB...,True
451,UNIVERSITY OF COLORADO HOSPITAL,DENVER,CO,80262,1,1,UNIVERSITY OF COLORADO HOSPITAL | DENVER | CO ...,True
448,UNIV OF NEBRASKA MEDICAL CENTER COLLEGE OF NUR...,OMAHA,NE,68198,1,1,UNIV OF NEBRASKA MEDICAL CENTER COLLEGE OF NUR...,True
480,VA HOSPITAL -UNIVERSITY DRIVE DIVISON,PITTSBURGH,PA,15240,1,1,VA HOSPITAL -UNIVERSITY DRIVE DIVISON | PITTSB...,True


In [ ]:
# ============================================================
# STAGE 2C-30 — SECOND-PASS ORDINARY ADDRESS SET
# ============================================================

second_pass_ordinary = (
    second_pass_addresses[
        ~second_pass_addresses["possible_nonstandard_address"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("========== SECOND-PASS ORDINARY ADDRESS SET ==========")

print(
    "Second-pass ordinary addresses:",
    len(second_pass_ordinary)
)

print(
    "Providers represented:",
    second_pass_ordinary["provider_count"].sum()
)

print(
    "States represented:",
    second_pass_ordinary["state_norm"].nunique()
)

print("\nTop addresses by provider count:")

display(
    second_pass_ordinary[
        [
            "street_1_norm",
            "city_norm",
            "state_norm",
            "zip_norm",
            "provider_count",
            "address_variant_count"
        ]
    ]
    .sort_values("provider_count", ascending=False)
    .head(100)
)

========== SECOND-PASS ORDINARY ADDRESS SET ==========
Second-pass ordinary addresses: 398
Providers represented: 859
States represented: 46

Top addresses by provider count:


,street_1_norm,city_norm,state_norm,zip_norm,provider_count,address_variant_count
98,263 FARMINGTON AVE,FARMINGTON,CT,06030,157,34
217,988102 NEBRASKA MEDICAL CTR,OMAHA,NE,68198,45,1
43,1201 BROAD ROCK BLVD,RICHMOND,VA,23249,34,16
228,BLDG N46 CAPE SARICHEF,KODIAK,AK,99619,17,1
125,40 DUKE MEDICINE CIR,DURHAM,NC,27710,11,2
...,...,...,...,...,...,...
95,2549 MOMENTUM PL,CHICAGO,IL,60689,1,1
94,2540 NTH GALLOWAY AVE,MESQUITE,TX,75151,1,1
93,254 RT 202-206 NORTH,PLUCKEMIN,NJ,07978,1,1
91,253 BARKLEY MEMORIAL CENTEER,LINCOLN,NE,68583,1,1


In [ ]:
# ============================================================
# STAGE 2C-31 — SECOND-PASS GEOCODING REQUEST TABLE
# ============================================================

second_pass_requests = (
    second_pass_ordinary[
        [
            "street_1_norm",
            "city_norm",
            "state_norm",
            "zip_norm",
            "provider_count",
            "address_variant_count",
            "geo_address_key"
        ]
    ]
    .copy()
)

# Defensive normalization
for col in ["street_1_norm", "city_norm", "state_norm", "zip_norm"]:
    second_pass_requests[col] = (
        second_pass_requests[col]
        .fillna("")
        .astype("string")
        .str.strip()
    )

# Make sure ZIP remains 5-character where applicable
second_pass_requests["zip_norm"] = (
    second_pass_requests["zip_norm"]
    .str.extract(r"(\d{5})", expand=False)
    .fillna("")
)

# Remove accidental duplicate geographic request keys
second_pass_requests = (
    second_pass_requests
    .drop_duplicates(subset=["geo_address_key"])
    .reset_index(drop=True)
)

print("========== SECOND-PASS REQUEST VALIDATION ==========")

print(
    "Unique requests:",
    len(second_pass_requests)
)

print(
    "Providers represented:",
    second_pass_requests["provider_count"].sum()
)

print(
    "Duplicate geographic keys:",
    second_pass_requests["geo_address_key"].duplicated().sum()
)

print(
    "Missing street:",
    (second_pass_requests["street_1_norm"] == "").sum()
)

print(
    "Missing city:",
    (second_pass_requests["city_norm"] == "").sum()
)

print(
    "Missing state:",
    (second_pass_requests["state_norm"] == "").sum()
)

print(
    "Missing ZIP:",
    (second_pass_requests["zip_norm"] == "").sum()
)

print(
    "States:",
    second_pass_requests["state_norm"].nunique()
)

print("\nState distribution:")

display(
    second_pass_requests["state_norm"]
    .value_counts()
    .rename_axis("state")
    .reset_index(name="address_count")
    .head(50)
)

========== SECOND-PASS REQUEST VALIDATION ==========
Unique requests: 398
Providers represented: 859
Duplicate geographic keys: 0
Missing street: 0
Missing city: 0
Missing state: 0
Missing ZIP: 0
States: 46

State distribution:


,state,address_count
0,NC,32
1,TN,27
2,WA,27
3,PA,23
4,CA,22
5,DC,22
6,IL,21
7,OH,15
8,NE,15
9,CT,15


In [ ]:
# ============================================================
# STAGE 2C-32 — SECOND CENSUS GEOCODING PASS
# WITH CHECKPOINTING
# ============================================================

import requests
import pandas as pd
import time
from pathlib import Path

CENSUS_GEOCODER_URL = (
    "https://geocoding.geo.census.gov/geocoder/locations/address"
)

CHECKPOINT_PATH = Path(
    "/content/drive/MyDrive/data/geographic_reference/"
    "nppes_census_geocoding_results_second_pass.csv"
)

# ------------------------------------------------------------
# Load existing checkpoint if present
# ------------------------------------------------------------

if CHECKPOINT_PATH.exists():

    second_pass_results = pd.read_csv(
        CHECKPOINT_PATH,
        dtype="string"
    )

    print(
        "Existing checkpoint loaded:",
        len(second_pass_results),
        "completed requests"
    )

else:

    second_pass_results = pd.DataFrame(
        columns=[
            "geo_address_key",
            "street_1_norm",
            "city_norm",
            "state_norm",
            "zip_norm",
            "provider_count",
            "address_variant_count",
            "geocoding_source",
            "geocoding_status",
            "geocoding_match_count",
            "geocoded_matched_address",
            "provider_latitude",
            "provider_longitude",
            "geocoded_tiger_line_id",
            "geocoded_side"
        ]
    )

    print("No existing checkpoint found.")

# ------------------------------------------------------------
# Determine requests still needing processing
# ------------------------------------------------------------

completed_keys = set(
    second_pass_results["geo_address_key"]
    .dropna()
    .astype(str)
)

requests_to_process = second_pass_requests[
    ~second_pass_requests["geo_address_key"]
    .astype(str)
    .isin(completed_keys)
].copy()

print("\n========== SECOND-PASS GEOCODING ==========")

print(
    "Total requests:",
    len(second_pass_requests)
)

print(
    "Already completed:",
    len(completed_keys)
)

print(
    "Remaining requests:",
    len(requests_to_process)
)

# ------------------------------------------------------------
# Process remaining requests
# ------------------------------------------------------------

new_results = []

for counter, (_, row) in enumerate(
    requests_to_process.iterrows(),
    start=1
):

    params = {
        "street": row["street_1_norm"],
        "city": row["city_norm"],
        "state": row["state_norm"],
        "zip": row["zip_norm"],
        "benchmark": "Public_AR_Current",
        "format": "json"
    }

    result = {
        "geo_address_key": row["geo_address_key"],

        "street_1_norm": row["street_1_norm"],
        "city_norm": row["city_norm"],
        "state_norm": row["state_norm"],
        "zip_norm": row["zip_norm"],

        "provider_count": row["provider_count"],
        "address_variant_count": row["address_variant_count"],

        "geocoding_source": "Census_Geocoder_Second_Pass",
        "geocoding_status": "error",

        "geocoding_match_count": None,
        "geocoded_matched_address": None,

        "provider_latitude": None,
        "provider_longitude": None,

        "geocoded_tiger_line_id": None,
        "geocoded_side": None
    }

    try:

        response = requests.get(
            CENSUS_GEOCODER_URL,
            params=params,
            timeout=30
        )

        response.raise_for_status()

        data = response.json()

        matches = (
            data.get("result", {})
            .get("addressMatches", [])
        )

        result["geocoding_match_count"] = len(matches)

        if matches:

            match = matches[0]

            coordinates = match.get(
                "coordinates",
                {}
            )

            result["geocoding_status"] = "matched"

            result["geocoded_matched_address"] = (
                match.get("matchedAddress")
            )

            result["provider_longitude"] = (
                coordinates.get("x")
            )

            result["provider_latitude"] = (
                coordinates.get("y")
            )

            tiger_line = match.get(
                "tigerLine",
                {}
            )

            result["geocoded_tiger_line_id"] = (
                tiger_line.get("tigerLineId")
            )

            result["geocoded_side"] = (
                tiger_line.get("side")
            )

        else:

            result["geocoding_status"] = "unmatched"

    except Exception as e:

        result["geocoding_status"] = "error"
        result["geocoding_error"] = str(e)

    new_results.append(result)

    # --------------------------------------------------------
    # Checkpoint every 25 requests
    # --------------------------------------------------------

    if (
        counter % 25 == 0
        or counter == len(requests_to_process)
    ):

        batch_df = pd.DataFrame(new_results)

        second_pass_results = pd.concat(
            [
                second_pass_results,
                batch_df
            ],
            ignore_index=True
        )

        second_pass_results.to_csv(
            CHECKPOINT_PATH,
            index=False
        )

        new_results = []

        print(
            f"Checkpoint saved: "
            f"{len(second_pass_results)} / "
            f"{len(second_pass_requests)} requests"
        )

    # Be conservative with request rate
    time.sleep(0.1)

print("\n========== SECOND-PASS COMPLETE ==========")

print(
    "Total results:",
    len(second_pass_results)
)

print(
    "\nStatus distribution:"
)

display(
    second_pass_results[
        "geocoding_status"
    ]
    .value_counts(dropna=False)
)

print(
    "\nProvider coverage:"
)

matched_provider_count = (
    second_pass_results.loc[
        second_pass_results["geocoding_status"] == "matched",
        "provider_count"
    ]
    .sum()
)

print(
    "Providers represented by matched addresses:",
    matched_provider_count
)

print(
    "Checkpoint:",
    CHECKPOINT_PATH
)

No existing checkpoint found.

========== SECOND-PASS GEOCODING ==========
Total requests: 398
Already completed: 0
Remaining requests: 398
Checkpoint saved: 25 / 398 requests
Checkpoint saved: 50 / 398 requests
Checkpoint saved: 75 / 398 requests
Checkpoint saved: 100 / 398 requests
Checkpoint saved: 125 / 398 requests
Checkpoint saved: 150 / 398 requests
Checkpoint saved: 175 / 398 requests
Checkpoint saved: 200 / 398 requests
Checkpoint saved: 225 / 398 requests
Checkpoint saved: 250 / 398 requests
Checkpoint saved: 275 / 398 requests
Checkpoint saved: 300 / 398 requests
Checkpoint saved: 325 / 398 requests
Checkpoint saved: 350 / 398 requests
Checkpoint saved: 375 / 398 requests
Checkpoint saved: 398 / 398 requests

========== SECOND-PASS COMPLETE ==========
Total results: 398

Status distribution:


,count
geocoding_status,
unmatched,398



Provider coverage:
Providers represented by matched addresses: 0
Checkpoint: /content/drive/MyDrive/data/geographic_reference/nppes_census_geocoding_results_second_pass.csv


In [ ]:
# ============================================================
# STAGE 2C-33 — AUDIT SECOND-PASS GEOCODING RESULTS
# ============================================================

print("========== SECOND-PASS RESULT AUDIT ==========")

print("Rows:", len(second_pass_results))

print("\nStatus:")
display(
    second_pass_results["geocoding_status"]
    .value_counts(dropna=False)
)

print("\nMatch-count distribution:")
display(
    second_pass_results["geocoding_match_count"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nSample unresolved requests:")

display(
    second_pass_results[
        [
            "street_1_norm",
            "city_norm",
            "state_norm",
            "zip_norm",
            "geocoding_status",
            "geocoding_match_count"
        ]
    ].head(20)
)

========== SECOND-PASS RESULT AUDIT ==========
Rows: 398

Status:


,count
geocoding_status,
unmatched,398



Match-count distribution:


,count
geocoding_match_count,
0,398



Sample unresolved requests:


,street_1_norm,city_norm,state_norm,zip_norm,geocoding_status,geocoding_match_count
0,# L-3539,COLUMBUS,OH,43260,unmatched,0
1,000 UNIVERSITY DRIVE C,PITTSBURGH,PA,15240,unmatched,0
2,0610 TERRAPIN TRAIL,COLLEGE PARK,MD,20741,unmatched,0
3,1 AMERICAN SQ,INDIANAPOLIS,IN,46282,unmatched,0
4,1 AMERICAN SQ STE 185,INDIANAPOLIS,IN,46282,unmatched,0
5,1 AYERS CIRCLE,PORTSMOUTH,NH,03804,unmatched,0
6,1 AYRES CIRCLE,PORTSMOUTH,NH,03804,unmatched,0
7,1 ELIZABETH PL,DAYTON,OH,45408,unmatched,0
8,1 ELIZABETH PLACE,DAYTON,OH,45408,unmatched,0
9,1 FLETCHER DR,GAINESVILLE,FL,32611,unmatched,0


In [ ]:
# ============================================================
# STAGE 2C-33B — DIRECT CENSUS RESPONSE TEST
# ============================================================

import requests

test_row = second_pass_results.iloc[0]

test_params = {
    "street": test_row["street_1_norm"],
    "city": test_row["city_norm"],
    "state": test_row["state_norm"],
    "zip": test_row["zip_norm"],
    "benchmark": "Public_AR_Current",
    "format": "json"
}

print("========== TEST REQUEST ==========")

print("Street:", test_params["street"])
print("City:", test_params["city"])
print("State:", test_params["state"])
print("ZIP:", test_params["zip"])

response = requests.get(
    CENSUS_GEOCODER_URL,
    params=test_params,
    timeout=30
)

print("\nHTTP status:", response.status_code)
print("Final URL:")
print(response.url)

print("\nResponse preview:")
print(response.text[:2000])

========== TEST REQUEST ==========
Street: # L-3539
City: COLUMBUS
State: OH
ZIP: 43260

HTTP status: 200
Final URL:
https://geocoding.geo.census.gov/geocoder/locations/address?street=%23+L-3539&city=COLUMBUS&state=OH&zip=43260&benchmark=Public_AR_Current&format=json

Response preview:
{"result":{"input":{"address":{"zip":"43260","city":"COLUMBUS","street":"# L-3539","state":"OH"},"benchmark":{"isDefault":true,"benchmarkDescription":"Public Address Ranges - Current Benchmark","id":"4","benchmarkName":"Public_AR_Current"}},"addressMatches":[]}}


In [ ]:
# ============================================================
# STAGE 2C-34 — CORRECTED FIRST-PASS COORDINATE AUDIT
# ============================================================

# Keep only coordinate-bearing first-pass results
coordinate_pairs = (
    matched_first_pass[
        [
            "geo_address_key",
            "street_1_norm",
            "city_norm",
            "state_norm",
            "zip_norm",
            "provider_latitude",
            "provider_longitude"
        ]
    ]
    .dropna(
        subset=[
            "provider_latitude",
            "provider_longitude"
        ]
    )
    .drop_duplicates()
)

# Count distinct coordinate pairs per geographic address
coordinate_counts = (
    coordinate_pairs
    .groupby("geo_address_key")
    .size()
    .sort_values(ascending=False)
)

conflicting_keys = coordinate_counts[
    coordinate_counts > 1
]

print("========== FIRST-PASS COORDINATE CONFLICT AUDIT ==========")

print(
    "Matched first-pass rows:",
    len(matched_first_pass)
)

print(
    "Unique geographic keys:",
    matched_first_pass["geo_address_key"].nunique()
)

print(
    "Unique geographic keys with coordinates:",
    coordinate_pairs["geo_address_key"].nunique()
)

print(
    "Geographic keys with >1 coordinate:",
    len(conflicting_keys)
)

if len(conflicting_keys) > 0:

    print("\n========== COORDINATE CONFLICTS ==========")

    display(
        coordinate_pairs[
            coordinate_pairs["geo_address_key"]
            .isin(conflicting_keys.index)
        ]
        .sort_values(
            [
                "geo_address_key",
                "provider_latitude",
                "provider_longitude"
            ]
        )
        .head(200)
    )

else:

    print(
        "\nNo geographic addresses have conflicting coordinates."
    )

========== FIRST-PASS COORDINATE CONFLICT AUDIT ==========
Matched first-pass rows: 925
Unique geographic keys: 655
Unique geographic keys with coordinates: 655
Geographic keys with >1 coordinate: 0

No geographic addresses have conflicting coordinates.


In [ ]:
# ============================================================
# STAGE 2C-35 — CANONICAL CENSUS GEOGRAPHIC COORDINATES
# ============================================================

# ------------------------------------------------------------
# Build one canonical coordinate per geographic address key
# ------------------------------------------------------------

canonical_census_coordinates = (
    matched_first_pass[
        [
            "geo_address_key",
            "street_1_norm",
            "city_norm",
            "state_norm",
            "zip_norm",
            "provider_latitude",
            "provider_longitude",
            "geocoded_matched_address",
            "geocoded_tiger_line_id",
            "geocoded_side"
        ]
    ]
    .dropna(
        subset=[
            "provider_latitude",
            "provider_longitude"
        ]
    )
    .drop_duplicates(
        subset=["geo_address_key"]
    )
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Add explicit provenance
# ------------------------------------------------------------

canonical_census_coordinates["coordinate_source"] = (
    "Census_Geocoder"
)

canonical_census_coordinates["coordinate_method"] = (
    "Census_Public_AR_Current"
)

canonical_census_coordinates["coordinate_confidence"] = (
    "geocoded_address"
)

print("========== CANONICAL CENSUS COORDINATES ==========")

print(
    "Canonical geographic addresses:",
    len(canonical_census_coordinates)
)

print(
    "Missing latitude:",
    canonical_census_coordinates[
        "provider_latitude"
    ].isna().sum()
)

print(
    "Missing longitude:",
    canonical_census_coordinates[
        "provider_longitude"
    ].isna().sum()
)

print(
    "Duplicate geographic keys:",
    canonical_census_coordinates[
        "geo_address_key"
    ].duplicated().sum()
)

print(
    "Latitude range:",
    (
        canonical_census_coordinates[
            "provider_latitude"
        ].min(),
        canonical_census_coordinates[
            "provider_latitude"
        ].max()
    )
)

print(
    "Longitude range:",
    (
        canonical_census_coordinates[
            "provider_longitude"
        ].min(),
        canonical_census_coordinates[
            "provider_longitude"
        ].max()
    )
)

display(
    canonical_census_coordinates[
        [
            "street_1_norm",
            "city_norm",
            "state_norm",
            "zip_norm",
            "provider_latitude",
            "provider_longitude",
            "coordinate_source",
            "coordinate_confidence"
        ]
    ].head(20)
)

========== CANONICAL CENSUS COORDINATES ==========
Canonical geographic addresses: 655
Missing latitude: 0
Missing longitude: 0
Duplicate geographic keys: 0
Latitude range: (25.761088025072, 48.000059217127)
Longitude range: (-122.91836793628, -69.767950368864)


,street_1_norm,city_norm,state_norm,zip_norm,provider_latitude,provider_longitude,coordinate_source,coordinate_confidence
0,1 GENERAL ST,LAWRENCE,MA,01842,42.709078,-71.147974,Census_Geocoder,geocoded_address
1,1 HANSON PL,BROOKLYN,NY,11243,40.685143,-73.977831,Census_Geocoder,geocoded_address
2,1 HANSON PL STE 710,BROOKLYN,NY,11243,40.685143,-73.977831,Census_Geocoder,geocoded_address
3,1 MED CENTER DR,MORGANTOWN,WV,26507,39.652236,-79.959761,Census_Geocoder,geocoded_address
4,1 MEDICAL CENTER BLVD,WINSTON SALEM,NC,27157,36.090221,-80.271415,Census_Geocoder,geocoded_address
5,1 MEDICAL CENTER DR,WINSTON SALEM,NC,27157,36.090221,-80.271415,Census_Geocoder,geocoded_address
6,1 PARK ST,NEW HAVEN,CT,06504,41.325080,-72.858685,Census_Geocoder,geocoded_address
7,1 PARK ST,NEW HAVEN,CT,06520,41.325080,-72.858685,Census_Geocoder,geocoded_address
8,1 RODGERS DR,MORGANTOWN,WV,26507,39.623787,-79.945900,Census_Geocoder,geocoded_address
9,1 STADIUM DRIVE,MORGANTOWN,WV,26507,39.652236,-79.959761,Census_Geocoder,geocoded_address


In [ ]:
# ============================================================
# STAGE 2C-36 — PROVIDER-LEVEL CENSUS COORDINATE MERGE
# CORRECTED VERSION
# ============================================================

import re
import pandas as pd

provider_geo = geo_ready.copy()


# ------------------------------------------------------------
# 1. Recreate the same normalization logic used for
#    Census geocoding
# ------------------------------------------------------------

def normalize_address_component(value):
    if pd.isna(value):
        return ""

    value = str(value).upper().strip()

    # Normalize whitespace
    value = re.sub(r"\s+", " ", value)

    return value


# ------------------------------------------------------------
# 2. Build provider-level normalized geography fields
# ------------------------------------------------------------

provider_geo["street_1_norm"] = (
    provider_geo[
        "Provider First Line Business Practice Location Address"
    ]
    .map(normalize_address_component)
)

provider_geo["city_norm"] = (
    provider_geo[
        "Provider Business Practice Location Address City Name"
    ]
    .map(normalize_address_component)
)

provider_geo["state_norm"] = (
    provider_geo[
        "Provider Business Practice Location Address State Name"
    ]
    .map(normalize_address_component)
)

# ZIP was already cleaned during the geography eligibility stage
provider_geo["zip_norm"] = (
    provider_geo["zip_clean"]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)


# ------------------------------------------------------------
# 3. Build the geographic address key
#
# IMPORTANT:
# street_2 is intentionally NOT included.
#
# This is consistent with the Census geocoding workflow.
# ------------------------------------------------------------

provider_geo["geo_address_key"] = (
    provider_geo["street_1_norm"]
    + " | "
    + provider_geo["city_norm"]
    + " | "
    + provider_geo["state_norm"]
    + " | "
    + provider_geo["zip_norm"]
)


# ------------------------------------------------------------
# 4. Prepare canonical Census coordinate lookup
# ------------------------------------------------------------

coordinate_lookup = (
    canonical_census_coordinates[
        [
            "geo_address_key",
            "provider_latitude",
            "provider_longitude",
            "coordinate_source",
            "coordinate_method",
            "coordinate_confidence",
            "geocoded_matched_address",
            "geocoded_tiger_line_id",
            "geocoded_side"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 5. Validate canonical lookup
# ------------------------------------------------------------

assert coordinate_lookup["geo_address_key"].notna().all()

assert coordinate_lookup["geo_address_key"].is_unique, (
    "Canonical coordinate table contains duplicate "
    "geographic address keys."
)


# ------------------------------------------------------------
# 6. Merge Census coordinates onto providers
# ------------------------------------------------------------

provider_geo = provider_geo.merge(
    coordinate_lookup,
    on="geo_address_key",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# 7. Explicit coordinate availability
# ------------------------------------------------------------

provider_geo["provider_coordinate_available"] = (
    provider_geo[
        [
            "provider_latitude",
            "provider_longitude"
        ]
    ]
    .notna()
    .all(axis=1)
)


# ------------------------------------------------------------
# 8. Fill provenance for unresolved providers
# ------------------------------------------------------------

provider_geo["coordinate_source"] = (
    provider_geo["coordinate_source"]
    .fillna("unresolved")
)

provider_geo["coordinate_confidence"] = (
    provider_geo["coordinate_confidence"]
    .fillna("unresolved")
)


# ------------------------------------------------------------
# 9. Integrity checks
# ------------------------------------------------------------

print("========== PROVIDER-LEVEL COORDINATE MERGE ==========")

print(
    "Provider rows:",
    len(provider_geo)
)

print(
    "Unique NPIs:",
    provider_geo["NPI"].nunique()
)

print(
    "Duplicate NPIs:",
    provider_geo["NPI"].duplicated().sum()
)

print(
    "Providers with Census coordinates:",
    int(
        provider_geo[
            "provider_coordinate_available"
        ].sum()
    )
)

print(
    "Providers without Census coordinates:",
    int(
        (~provider_geo[
            "provider_coordinate_available"
        ]).sum()
    )
)

print(
    "Coordinate coverage:",
    round(
        provider_geo[
            "provider_coordinate_available"
        ].mean() * 100,
        2
    ),
    "%"
)


# ------------------------------------------------------------
# 10. Coordinate source distribution
# ------------------------------------------------------------

print("\n========== COORDINATE SOURCE ==========")

display(
    provider_geo[
        "coordinate_source"
    ]
    .value_counts(dropna=False)
    .rename_axis("coordinate_source")
    .reset_index(name="provider_count")
)


# ------------------------------------------------------------
# 11. Coordinate confidence distribution
# ------------------------------------------------------------

print("\n========== COORDINATE CONFIDENCE ==========")

display(
    provider_geo[
        "coordinate_confidence"
    ]
    .value_counts(dropna=False)
    .rename_axis("coordinate_confidence")
    .reset_index(name="provider_count")
)


# ------------------------------------------------------------
# 12. Hard integrity assertions
# ------------------------------------------------------------

assert len(provider_geo) == len(geo_ready), (
    "ERROR: Provider row count changed during merge."
)

assert provider_geo["NPI"].is_unique, (
    "ERROR: NPI uniqueness was lost during merge."
)

assert (
    provider_geo["provider_coordinate_available"]
    ==
    (
        provider_geo["provider_latitude"].notna()
        &
        provider_geo["provider_longitude"].notna()
    )
).all()

print(
    "\nRow-count integrity: PASSED"
)

print(
    "NPI uniqueness integrity: PASSED"
)

print(
    "Coordinate completeness integrity: PASSED"
)

========== PROVIDER-LEVEL COORDINATE MERGE ==========
Provider rows: 281478
Unique NPIs: 281478
Duplicate NPIs: 0
Providers with Census coordinates: 2349
Providers without Census coordinates: 279129
Coordinate coverage: 0.83 %

========== COORDINATE SOURCE ==========


,coordinate_source,provider_count
0,unresolved,279129
1,Census_Geocoder,2349



========== COORDINATE CONFIDENCE ==========


,coordinate_confidence,provider_count
0,unresolved,279129
1,geocoded_address,2349



Row-count integrity: PASSED
NPI uniqueness integrity: PASSED
Coordinate completeness integrity: PASSED


In [ ]:
# ============================================================
# STAGE 2C-37 — FINAL GEOGRAPHY LAYER
# ZCTA CENTROID + PRIMARY COUNTY
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Start from the successful provider-level coordinate layer
# ------------------------------------------------------------

be3_geo = provider_geo.copy()


# ============================================================
# 1. REMOVE EXISTING COUNTY FIELDS
# ============================================================
#
# provider_geo inherited county fields from the earlier
# NPPES -> ZCTA/county merge.
#
# We intentionally replace them with the clean Census
# primary-county lookup constructed below.
# ============================================================

existing_county_columns = [
    "county_fips",
    "NAMELSAD_COUNTY_20",
    "county_source",
    "county_available"
]

be3_geo = be3_geo.drop(
    columns=[
        col
        for col in existing_county_columns
        if col in be3_geo.columns
    ]
)


# ============================================================
# 2. ZCTA CENTROID LOOKUP
# ============================================================

zcta_centroid_lookup = (
    zcta_centroids[
        [
            "zcta_clean",
            "centroid_latitude",
            "centroid_longitude"
        ]
    ]
    .copy()
)

assert zcta_centroid_lookup["zcta_clean"].is_unique

print(
    "ZCTA centroid lookup:",
    len(zcta_centroid_lookup),
    "unique ZCTAs"
)


# ============================================================
# 3. PRIMARY COUNTY LOOKUP
# ============================================================
#
# Rebuild directly from the Census relationship table.
#
# Primary county =
# county with the largest ZCTA/county land intersection.
# ============================================================

county_relationship = (
    zcta_county_relationship.copy()
)

county_relationship = county_relationship[
    county_relationship["GEOID_ZCTA5_20"].notna()
    &
    county_relationship["GEOID_COUNTY_20"].notna()
].copy()


# Standardize ZCTA

county_relationship["zcta_clean"] = (
    county_relationship["GEOID_ZCTA5_20"]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)


# Standardize county FIPS

county_relationship["county_fips"] = (
    county_relationship["GEOID_COUNTY_20"]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)


# Ensure land intersection is numeric

county_relationship["AREALAND_PART"] = pd.to_numeric(
    county_relationship["AREALAND_PART"],
    errors="coerce"
)


# Largest land intersection first

county_relationship = county_relationship.sort_values(
    [
        "zcta_clean",
        "AREALAND_PART"
    ],
    ascending=[
        True,
        False
    ],
    na_position="last"
)


# One primary county per ZCTA

county_lookup_final = (
    county_relationship
    .drop_duplicates(
        subset=["zcta_clean"],
        keep="first"
    )
    [
        [
            "zcta_clean",
            "county_fips",
            "NAMELSAD_COUNTY_20"
        ]
    ]
    .copy()
)


assert county_lookup_final["zcta_clean"].is_unique

print(
    "Primary county lookup:",
    len(county_lookup_final),
    "unique ZCTAs"
)


# ============================================================
# 4. ATTACH ZCTA CENTROIDS
# ============================================================

be3_geo = be3_geo.merge(
    zcta_centroid_lookup,
    on="zcta_clean",
    how="left",
    validate="many_to_one"
)


# ============================================================
# 5. ATTACH PRIMARY COUNTY
# ============================================================

be3_geo = be3_geo.merge(
    county_lookup_final,
    on="zcta_clean",
    how="left",
    validate="many_to_one"
)


# ============================================================
# 6. ZCTA CENTROID AVAILABILITY
# ============================================================

be3_geo["zcta_centroid_available"] = (
    be3_geo["centroid_latitude"].notna()
    &
    be3_geo["centroid_longitude"].notna()
)


# ============================================================
# 7. COUNTY AVAILABILITY
# ============================================================

be3_geo["county_available"] = (
    be3_geo["county_fips"].notna()
)


# ============================================================
# 8. ZCTA PROVENANCE
# ============================================================

be3_geo["zcta_source"] = "unresolved"

be3_geo.loc[
    be3_geo["zcta_centroid_available"],
    "zcta_source"
] = "Census_ZCTA_Gazetteer_2020"


# ============================================================
# 9. COUNTY PROVENANCE
# ============================================================

be3_geo["county_source"] = "unresolved"

be3_geo.loc[
    be3_geo["county_available"],
    "county_source"
] = "Census_ZCTA_County_Relationship_2020"


# ============================================================
# 10. OVERALL GEOGRAPHY CLASSIFICATION
# ============================================================

be3_geo["geography_confidence"] = "no_coordinate"

be3_geo.loc[
    be3_geo["zcta_centroid_available"],
    "geography_confidence"
] = "zcta_centroid_only"

be3_geo.loc[
    be3_geo["provider_coordinate_available"],
    "geography_confidence"
] = "provider_coordinate"


# ============================================================
# 11. DIAGNOSTICS
# ============================================================

print("\n========== BE-3 GEOGRAPHY LAYER ==========")

print(
    "Provider rows:",
    len(be3_geo)
)

print(
    "Unique NPIs:",
    be3_geo["NPI"].nunique()
)

print(
    "Duplicate NPIs:",
    be3_geo["NPI"].duplicated().sum()
)

print(
    "\nProviders with provider coordinates:",
    int(
        be3_geo["provider_coordinate_available"].sum()
    )
)

print(
    "Providers with ZCTA centroids:",
    int(
        be3_geo["zcta_centroid_available"].sum()
    )
)

print(
    "Providers with primary county:",
    int(
        be3_geo["county_available"].sum()
    )
)


# ============================================================
# 12. GEOGRAPHY CLASSIFICATION
# ============================================================

print(
    "\n========== GEOGRAPHY CLASSIFICATION =========="
)

display(
    be3_geo[
        "geography_confidence"
    ]
    .value_counts(dropna=False)
    .rename_axis(
        "geography_confidence"
    )
    .reset_index(
        name="provider_count"
    )
)


# ============================================================
# 13. ZCTA SOURCE
# ============================================================

print(
    "\n========== ZCTA SOURCE =========="
)

display(
    be3_geo[
        "zcta_source"
    ]
    .value_counts(dropna=False)
    .rename_axis(
        "zcta_source"
    )
    .reset_index(
        name="provider_count"
    )
)


# ============================================================
# 14. COUNTY SOURCE
# ============================================================

print(
    "\n========== COUNTY SOURCE =========="
)

display(
    be3_geo[
        "county_source"
    ]
    .value_counts(dropna=False)
    .rename_axis(
        "county_source"
    )
    .reset_index(
        name="provider_count"
    )
)


# ============================================================
# 15. FINAL INTEGRITY CHECKS
# ============================================================

assert len(be3_geo) == len(provider_geo), (
    "ERROR: Provider row count changed."
)

assert be3_geo["NPI"].is_unique, (
    "ERROR: NPI uniqueness was lost."
)

assert "county_fips" in be3_geo.columns

assert "NAMELSAD_COUNTY_20" in be3_geo.columns

print(
    "\nRow-count integrity: PASSED"
)

print(
    "NPI uniqueness integrity: PASSED"
)

print(
    "County field integrity: PASSED"
)

ZCTA centroid lookup: 33144 unique ZCTAs
Primary county lookup: 33791 unique ZCTAs

========== BE-3 GEOGRAPHY LAYER ==========
Provider rows: 281478
Unique NPIs: 281478
Duplicate NPIs: 0

Providers with provider coordinates: 2349
Providers with ZCTA centroids: 274994
Providers with primary county: 276704

========== GEOGRAPHY CLASSIFICATION ==========


,geography_confidence,provider_count
0,zcta_centroid_only,274994
1,no_coordinate,4135
2,provider_coordinate,2349



========== ZCTA SOURCE ==========


,zcta_source,provider_count
0,Census_ZCTA_Gazetteer_2020,274994
1,unresolved,6484



========== COUNTY SOURCE ==========


,county_source,provider_count
0,Census_ZCTA_County_Relationship_2020,276704
1,unresolved,4774



Row-count integrity: PASSED
NPI uniqueness integrity: PASSED
County field integrity: PASSED


In [ ]:
# ============================================================
# STAGE 2C-38 — FULL PROVIDER SPATIAL COORDINATE STRATEGY
# CELL 1 — BUILD UNIFIED SPATIAL COORDINATE LAYER
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Start from the validated BE-3 geography layer
# ------------------------------------------------------------

spatial_geo = be3_geo.copy()

print("Input provider rows:", len(spatial_geo))
print("Input unique NPIs:", spatial_geo["NPI"].nunique())


# ============================================================
# 1. PROVIDER-LEVEL COORDINATE VALIDATION
# ============================================================

provider_coord_valid = (
    spatial_geo["provider_latitude"].notna()
    &
    spatial_geo["provider_longitude"].notna()
)

# Basic geographic bounds check
provider_coord_valid &= (
    spatial_geo["provider_latitude"].between(-90, 90)
    &
    spatial_geo["provider_longitude"].between(-180, 180)
)


# ============================================================
# 2. ZCTA CENTROID VALIDATION
# ============================================================

zcta_coord_valid = (
    spatial_geo["centroid_latitude"].notna()
    &
    spatial_geo["centroid_longitude"].notna()
)

zcta_coord_valid &= (
    spatial_geo["centroid_latitude"].between(-90, 90)
    &
    spatial_geo["centroid_longitude"].between(-180, 180)
)


# ============================================================
# 3. CREATE UNIFIED SPATIAL COORDINATES
# ============================================================
#
# Priority:
#
# 1. Provider coordinate
# 2. ZCTA centroid
# 3. Unresolved
#
# IMPORTANT:
# We do NOT overwrite the original provider_latitude /
# provider_longitude fields.
# ============================================================

spatial_geo["spatial_latitude"] = np.nan
spatial_geo["spatial_longitude"] = np.nan

spatial_geo["spatial_coordinate_type"] = "unresolved"
spatial_geo["spatial_coordinate_source"] = "unresolved"
spatial_geo["spatial_coordinate_confidence"] = "unresolved"


# ------------------------------------------------------------
# Priority 1 — Provider coordinate
# ------------------------------------------------------------

provider_mask = provider_coord_valid

spatial_geo.loc[
    provider_mask,
    "spatial_latitude"
] = spatial_geo.loc[
    provider_mask,
    "provider_latitude"
]

spatial_geo.loc[
    provider_mask,
    "spatial_longitude"
] = spatial_geo.loc[
    provider_mask,
    "provider_longitude"
]

spatial_geo.loc[
    provider_mask,
    "spatial_coordinate_type"
] = "provider_coordinate"

spatial_geo.loc[
    provider_mask,
    "spatial_coordinate_source"
] = spatial_geo.loc[
    provider_mask,
    "coordinate_source"
]

spatial_geo.loc[
    provider_mask,
    "spatial_coordinate_confidence"
] = "high"


# ------------------------------------------------------------
# Priority 2 — ZCTA centroid fallback
# ------------------------------------------------------------

zcta_fallback_mask = (
    ~provider_mask
    &
    zcta_coord_valid
)

spatial_geo.loc[
    zcta_fallback_mask,
    "spatial_latitude"
] = spatial_geo.loc[
    zcta_fallback_mask,
    "centroid_latitude"
]

spatial_geo.loc[
    zcta_fallback_mask,
    "spatial_longitude"
] = spatial_geo.loc[
    zcta_fallback_mask,
    "centroid_longitude"
]

spatial_geo.loc[
    zcta_fallback_mask,
    "spatial_coordinate_type"
] = "zcta_centroid"

spatial_geo.loc[
    zcta_fallback_mask,
    "spatial_coordinate_source"
] = "Census_ZCTA_Gazetteer_2020"

spatial_geo.loc[
    zcta_fallback_mask,
    "spatial_coordinate_confidence"
] = "area_level"


# ============================================================
# 4. UNRESOLVED REMAINS UNRESOLVED
# ============================================================

unresolved_mask = (
    spatial_geo["spatial_latitude"].isna()
    |
    spatial_geo["spatial_longitude"].isna()
)

spatial_geo.loc[
    unresolved_mask,
    "spatial_latitude"
] = np.nan

spatial_geo.loc[
    unresolved_mask,
    "spatial_longitude"
] = np.nan

spatial_geo.loc[
    unresolved_mask,
    "spatial_coordinate_type"
] = "unresolved"

spatial_geo.loc[
    unresolved_mask,
    "spatial_coordinate_source"
] = "unresolved"

spatial_geo.loc[
    unresolved_mask,
    "spatial_coordinate_confidence"
] = "unresolved"


# ============================================================
# 5. AVAILABILITY FLAG
# ============================================================

spatial_geo["spatial_coordinate_available"] = (
    spatial_geo["spatial_latitude"].notna()
    &
    spatial_geo["spatial_longitude"].notna()
)


# ============================================================
# 6. INTEGRITY CHECKS
# ============================================================

assert len(spatial_geo) == len(be3_geo), (
    "ERROR: Provider row count changed."
)

assert spatial_geo["NPI"].is_unique, (
    "ERROR: NPI uniqueness was lost."
)

assert (
    spatial_geo["spatial_coordinate_available"]
    ==
    (
        spatial_geo["spatial_latitude"].notna()
        &
        spatial_geo["spatial_longitude"].notna()
    )
).all()


# ============================================================
# 7. SUMMARY
# ============================================================

print("\n========== STAGE 2C-38 — SPATIAL COORDINATE SUMMARY ==========")

summary = (
    spatial_geo["spatial_coordinate_type"]
    .value_counts(dropna=False)
    .rename_axis("spatial_coordinate_type")
    .reset_index(name="provider_count")
)

summary["coverage_percent"] = (
    summary["provider_count"]
    / len(spatial_geo)
    * 100
)

display(summary)


print("\n========== SOURCE SUMMARY ==========")

source_summary = (
    spatial_geo["spatial_coordinate_source"]
    .value_counts(dropna=False)
    .rename_axis("spatial_coordinate_source")
    .reset_index(name="provider_count")
)

source_summary["coverage_percent"] = (
    source_summary["provider_count"]
    / len(spatial_geo)
    * 100
)

display(source_summary)


print("\n========== CONFIDENCE SUMMARY ==========")

confidence_summary = (
    spatial_geo["spatial_coordinate_confidence"]
    .value_counts(dropna=False)
    .rename_axis("spatial_coordinate_confidence")
    .reset_index(name="provider_count")
)

confidence_summary["coverage_percent"] = (
    confidence_summary["provider_count"]
    / len(spatial_geo)
    * 100
)

display(confidence_summary)


# ============================================================
# 8. FINAL COVERAGE
# ============================================================

print(
    "\nProviders with usable spatial coordinates:",
    int(
        spatial_geo["spatial_coordinate_available"].sum()
    )
)

print(
    "Providers without usable spatial coordinates:",
    int(
        (~spatial_geo["spatial_coordinate_available"]).sum()
    )
)

print(
    "Total providers:",
    len(spatial_geo)
)

print("\nStage 2C-38 Cell 1: PASSED")

Input provider rows: 281478
Input unique NPIs: 281478

========== STAGE 2C-38 — SPATIAL COORDINATE SUMMARY ==========


,spatial_coordinate_type,provider_count,coverage_percent
0,zcta_centroid,274994,97.696445
1,unresolved,4135,1.469031
2,provider_coordinate,2349,0.834523



========== SOURCE SUMMARY ==========


,spatial_coordinate_source,provider_count,coverage_percent
0,Census_ZCTA_Gazetteer_2020,274994,97.696445
1,unresolved,4135,1.469031
2,Census_Geocoder,2349,0.834523



========== CONFIDENCE SUMMARY ==========


,spatial_coordinate_confidence,provider_count,coverage_percent
0,area_level,274994,97.696445
1,unresolved,4135,1.469031
2,high,2349,0.834523



Providers with usable spatial coordinates: 277343
Providers without usable spatial coordinates: 4135
Total providers: 281478

Stage 2C-38 Cell 1: PASSED


In [ ]:
# ============================================================
# STAGE 2C-38 — CELL 2
# SPATIAL COORDINATE QUALITY AUDIT
# ============================================================

import numpy as np
import pandas as pd

audit_geo = spatial_geo.copy()

print("Input provider rows:", len(audit_geo))
print("Input unique NPIs:", audit_geo["NPI"].nunique())


# ============================================================
# 1. COORDINATE COMPLETENESS
# ============================================================

lat_present = audit_geo["spatial_latitude"].notna()
lon_present = audit_geo["spatial_longitude"].notna()

both_present = lat_present & lon_present
lat_only = lat_present & ~lon_present
lon_only = ~lat_present & lon_present

print("\n========== COORDINATE COMPLETENESS ==========")

print("Latitude present:", int(lat_present.sum()))
print("Longitude present:", int(lon_present.sum()))
print("Both coordinates present:", int(both_present.sum()))
print("Latitude only:", int(lat_only.sum()))
print("Longitude only:", int(lon_only.sum()))
print("Neither coordinate:", int((~lat_present & ~lon_present).sum()))


# ============================================================
# 2. GLOBAL RANGE CHECK
# ============================================================

valid_range = (
    audit_geo["spatial_latitude"].between(-90, 90)
    &
    audit_geo["spatial_longitude"].between(-180, 180)
)

invalid_range = both_present & ~valid_range

print("\n========== GLOBAL RANGE CHECK ==========")

print(
    "Coordinates within valid geographic range:",
    int((both_present & valid_range).sum())
)

print(
    "Coordinates outside valid geographic range:",
    int(invalid_range.sum())
)


# ============================================================
# 3. U.S.-RELEVANT RANGE CHECK
# ============================================================
#
# This is an AUDIT only.
# We do not remove records based on this check.
#
# The dataset includes territories/special jurisdictions,
# so we use a deliberately broad geographic envelope rather
# than assuming every provider must lie in the contiguous U.S.
# ============================================================

broad_us_range = (
    audit_geo["spatial_latitude"].between(18, 72)
    &
    audit_geo["spatial_longitude"].between(-180, -60)
)

outside_broad_us_range = both_present & ~broad_us_range

print("\n========== BROAD U.S. / TERRITORY RANGE CHECK ==========")

print(
    "Coordinates inside broad U.S./territory envelope:",
    int((both_present & broad_us_range).sum())
)

print(
    "Coordinates outside broad envelope:",
    int(outside_broad_us_range.sum())
)


# ============================================================
# 4. COORDINATE TYPE COUNTS
# ============================================================

print("\n========== COORDINATE TYPE ==========")

type_summary = (
    audit_geo[
        "spatial_coordinate_type"
    ]
    .value_counts(dropna=False)
    .rename_axis("spatial_coordinate_type")
    .reset_index(name="provider_count")
)

type_summary["coverage_percent"] = (
    type_summary["provider_count"]
    / len(audit_geo)
    * 100
)

display(type_summary)


# ============================================================
# 5. DUPLICATE COORDINATE PAIRS
# ============================================================
#
# This is expected to be substantial for ZCTA centroids.
# Multiple providers can legitimately share one ZCTA centroid.
#
# Therefore duplicate coordinates are an AUDIT FEATURE,
# not automatically an error.
# ============================================================

coordinate_pairs = (
    audit_geo[
        both_present
    ]
    [
        [
            "spatial_latitude",
            "spatial_longitude"
        ]
    ]
)

duplicate_coordinate_rows = (
    coordinate_pairs.duplicated(
        keep=False
    )
)

duplicate_coordinate_count = int(
    duplicate_coordinate_rows.sum()
)

unique_coordinate_pairs = int(
    coordinate_pairs.drop_duplicates().shape[0]
)

print("\n========== DUPLICATE COORDINATE AUDIT ==========")

print(
    "Providers with usable coordinates:",
    len(coordinate_pairs)
)

print(
    "Unique coordinate pairs:",
    unique_coordinate_pairs
)

print(
    "Provider rows sharing a coordinate pair:",
    duplicate_coordinate_count
)


# ============================================================
# 6. DUPLICATES BY COORDINATE TYPE
# ============================================================

print("\n========== DUPLICATES BY COORDINATE TYPE ==========")

duplicate_by_type = []

for coord_type, group in audit_geo[
    both_present
].groupby(
    "spatial_coordinate_type"
):

    pair_count = group[
        [
            "spatial_latitude",
            "spatial_longitude"
        ]
    ].drop_duplicates().shape[0]

    provider_count = len(group)

    duplicate_rows = provider_count - pair_count

    duplicate_by_type.append({
        "spatial_coordinate_type": coord_type,
        "provider_count": provider_count,
        "unique_coordinate_pairs": pair_count,
        "duplicate_coordinate_rows": duplicate_rows
    })

duplicate_by_type = pd.DataFrame(
    duplicate_by_type
)

display(duplicate_by_type)


# ============================================================
# 7. PROVIDERS PER ZCTA CENTROID
# ============================================================
#
# This quantifies how heavily the area-level fallback
# represents multiple providers.
# ============================================================

zcta_providers = audit_geo[
    audit_geo["spatial_coordinate_type"] == "zcta_centroid"
].copy()

zcta_centroid_group = (
    zcta_providers
    .groupby(
        [
            "zcta_clean",
            "spatial_latitude",
            "spatial_longitude"
        ],
        dropna=False
    )
    .size()
    .reset_index(
        name="provider_count"
    )
)

print("\n========== ZCTA CENTROID SHARING ==========")

print(
    "ZCTAs represented by fallback coordinates:",
    len(zcta_centroid_group)
)

print(
    "Providers represented by fallback coordinates:",
    len(zcta_providers)
)

if len(zcta_centroid_group) > 0:

    print(
        "Minimum providers per represented ZCTA:",
        int(zcta_centroid_group["provider_count"].min())
    )

    print(
        "Median providers per represented ZCTA:",
        float(zcta_centroid_group["provider_count"].median())
    )

    print(
        "Maximum providers per represented ZCTA:",
        int(zcta_centroid_group["provider_count"].max())
    )


# ============================================================
# 8. TOP SHARED ZCTA CENTROIDS
# ============================================================

print("\n========== TOP SHARED ZCTA CENTROIDS ==========")

display(
    zcta_centroid_group
    .sort_values(
        "provider_count",
        ascending=False
    )
    .head(20)
)


# ============================================================
# 9. PROVIDER COORDINATE DUPLICATE AUDIT
# ============================================================
#
# Exact duplicate provider coordinates can occur when the
# Census geocoder returns the same point for multiple provider
# records at the same address.
# ============================================================

provider_coordinates = audit_geo[
    audit_geo["spatial_coordinate_type"] == "provider_coordinate"
].copy()

provider_coordinate_pairs = (
    provider_coordinates[
        [
            "spatial_latitude",
            "spatial_longitude"
        ]
    ]
)

provider_unique_pairs = (
    provider_coordinate_pairs
    .drop_duplicates()
    .shape[0]
)

print("\n========== PROVIDER COORDINATE AUDIT ==========")

print(
    "Providers with provider coordinates:",
    len(provider_coordinates)
)

print(
    "Unique provider coordinate pairs:",
    provider_unique_pairs
)

print(
    "Provider rows sharing exact provider coordinates:",
    len(provider_coordinates) - provider_unique_pairs
)


# ============================================================
# 10. PROVIDER VS ZCTA COORDINATE OVERLAP
# ============================================================
#
# Check whether any provider-coordinate records also have
# exactly the same coordinates as their ZCTA centroid.
#
# This is informational only.
# ============================================================

both_coordinate_types = (
    audit_geo[
        audit_geo["spatial_coordinate_type"]
        == "provider_coordinate"
    ]
    .copy()
)

both_coordinate_types["provider_equals_zcta_centroid"] = (
    np.isclose(
        both_coordinate_types["provider_latitude"],
        both_coordinate_types["centroid_latitude"],
        equal_nan=False
    )
    &
    np.isclose(
        both_coordinate_types["provider_longitude"],
        both_coordinate_types["centroid_longitude"],
        equal_nan=False
    )
)

print("\n========== PROVIDER / ZCTA OVERLAP ==========")

print(
    "Provider-coordinate records:",
    len(both_coordinate_types)
)

print(
    "Provider coordinates exactly matching ZCTA centroid:",
    int(
        both_coordinate_types[
            "provider_equals_zcta_centroid"
        ].sum()
    )
)


# ============================================================
# 11. FINAL AUDIT INTEGRITY
# ============================================================

assert len(audit_geo) == len(spatial_geo)
assert audit_geo["NPI"].is_unique

assert (
    (
        audit_geo["spatial_coordinate_available"]
        ==
        (
            audit_geo["spatial_latitude"].notna()
            &
            audit_geo["spatial_longitude"].notna()
        )
    )
).all()

print("\n========== FINAL AUDIT ==========")

print("Row-count integrity: PASSED")
print("NPI uniqueness integrity: PASSED")
print("Coordinate availability integrity: PASSED")
print("Stage 2C-38 Cell 2: PASSED")

Input provider rows: 281478
Input unique NPIs: 281478

========== COORDINATE COMPLETENESS ==========
Latitude present: 277343
Longitude present: 277343
Both coordinates present: 277343
Latitude only: 0
Longitude only: 0
Neither coordinate: 4135

========== GLOBAL RANGE CHECK ==========
Coordinates within valid geographic range: 277343
Coordinates outside valid geographic range: 0

========== BROAD U.S. / TERRITORY RANGE CHECK ==========
Coordinates inside broad U.S./territory envelope: 277213
Coordinates outside broad envelope: 130

========== COORDINATE TYPE ==========


,spatial_coordinate_type,provider_count,coverage_percent
0,zcta_centroid,274994,97.696445
1,unresolved,4135,1.469031
2,provider_coordinate,2349,0.834523



========== DUPLICATE COORDINATE AUDIT ==========
Providers with usable coordinates: 277343
Unique coordinate pairs: 16461
Provider rows sharing a coordinate pair: 273873

========== DUPLICATES BY COORDINATE TYPE ==========


,spatial_coordinate_type,provider_count,unique_coordinate_pairs,duplicate_coordinate_rows
0,provider_coordinate,2349,418,1931
1,zcta_centroid,274994,16043,258951



========== ZCTA CENTROID SHARING ==========
ZCTAs represented by fallback coordinates: 16043
Providers represented by fallback coordinates: 274994
Minimum providers per represented ZCTA: 1
Median providers per represented ZCTA: 6.0
Maximum providers per represented ZCTA: 1156

========== TOP SHARED ZCTA CENTROIDS ==========


,zcta_clean,spatial_latitude,spatial_longitude,provider_count
9650,55905,44.055948,-92.525906,1156
15335,95817,38.550547,-121.456373,566
9358,54601,43.806243,-91.140632,566
12644,77030,29.706787,-95.401748,554
366,02115,42.337105,-71.105696,463
365,02114,42.363174,-71.068646,455
12921,78229,29.507055,-98.566621,422
6516,37203,36.149775,-86.789146,404
1590,10021,40.769224,-73.958741,401
4339,27103,36.058230,-80.321496,401



========== PROVIDER COORDINATE AUDIT ==========
Providers with provider coordinates: 2349
Unique provider coordinate pairs: 418
Provider rows sharing exact provider coordinates: 1931

========== PROVIDER / ZCTA OVERLAP ==========
Provider-coordinate records: 2349
Provider coordinates exactly matching ZCTA centroid: 0

========== FINAL AUDIT ==========
Row-count integrity: PASSED
NPI uniqueness integrity: PASSED
Coordinate availability integrity: PASSED
Stage 2C-38 Cell 2: PASSED


In [ ]:
# ============================================================
# STAGE 2C-38 — CELL 3
# FINAL SPATIAL COORDINATE POLICY
# ============================================================

import numpy as np
import pandas as pd

# Work from the audited spatial layer
be3_spatial = spatial_geo.copy()

print("Input provider rows:", len(be3_spatial))
print("Input unique NPIs:", be3_spatial["NPI"].nunique())


# ------------------------------------------------------------
# 1. Preserve the unified spatial coordinate layer
# ------------------------------------------------------------

be3_spatial["spatial_coordinate_available"] = (
    be3_spatial["spatial_latitude"].notna()
    & be3_spatial["spatial_longitude"].notna()
)

assert (
    be3_spatial["spatial_coordinate_available"]
    ==
    (
        be3_spatial["spatial_latitude"].notna()
        & be3_spatial["spatial_longitude"].notna()
    )
).all()


# ------------------------------------------------------------
# 2. Define provider-level coordinate eligibility
#
# Provider-to-provider distance metrics require an actual
# provider/address coordinate. ZCTA centroids are NOT treated
# as provider coordinates.
# ------------------------------------------------------------

be3_spatial["provider_metric_eligible"] = (
    be3_spatial["spatial_coordinate_type"]
    == "provider_coordinate"
)


# ------------------------------------------------------------
# 3. Define area-level coordinate eligibility
#
# Area-level features may use either a provider coordinate or
# a ZCTA centroid.
# ------------------------------------------------------------

be3_spatial["area_metric_eligible"] = (
    be3_spatial["spatial_coordinate_available"]
)


# ------------------------------------------------------------
# 4. Explicitly flag centroid fallback
# ------------------------------------------------------------

be3_spatial["zcta_centroid_fallback"] = (
    be3_spatial["spatial_coordinate_type"]
    == "zcta_centroid"
)


# ------------------------------------------------------------
# 5. Explicitly flag unresolved geography
# ------------------------------------------------------------

be3_spatial["spatial_coordinate_unresolved"] = (
    be3_spatial["spatial_coordinate_type"]
    == "unresolved"
)


# ------------------------------------------------------------
# 6. Define coordinate policy label
# ------------------------------------------------------------

be3_spatial["spatial_metric_policy"] = "unresolved"

be3_spatial.loc[
    be3_spatial["spatial_coordinate_type"] == "provider_coordinate",
    "spatial_metric_policy"
] = "provider_level"

be3_spatial.loc[
    be3_spatial["spatial_coordinate_type"] == "zcta_centroid",
    "spatial_metric_policy"
] = "area_level_fallback"


# ------------------------------------------------------------
# 7. Policy summary
# ------------------------------------------------------------

print("\n========== FINAL SPATIAL COORDINATE POLICY ==========")

policy_summary = (
    be3_spatial[
        [
            "spatial_coordinate_type",
            "spatial_coordinate_source",
            "spatial_coordinate_confidence",
            "provider_metric_eligible",
            "area_metric_eligible",
            "zcta_centroid_fallback",
            "spatial_coordinate_unresolved",
            "spatial_metric_policy"
        ]
    ]
    .value_counts(dropna=False)
    .reset_index(name="provider_count")
)

policy_summary["coverage_percent"] = (
    policy_summary["provider_count"]
    / len(be3_spatial)
    * 100
)

display(policy_summary)


# ------------------------------------------------------------
# 8. Compact eligibility summary
# ------------------------------------------------------------

print("\n========== METRIC ELIGIBILITY ==========")

provider_metric_count = int(
    be3_spatial["provider_metric_eligible"].sum()
)

area_metric_count = int(
    be3_spatial["area_metric_eligible"].sum()
)

unresolved_count = int(
    be3_spatial["spatial_coordinate_unresolved"].sum()
)

print(
    "Provider-level metric eligible:",
    provider_metric_count,
    f"({provider_metric_count / len(be3_spatial) * 100:.2f}%)"
)

print(
    "Area-level metric eligible:",
    area_metric_count,
    f"({area_metric_count / len(be3_spatial) * 100:.2f}%)"
)

print(
    "Spatial coordinate unresolved:",
    unresolved_count,
    f"({unresolved_count / len(be3_spatial) * 100:.2f}%)"
)


# ------------------------------------------------------------
# 9. Verify policy is mutually consistent
# ------------------------------------------------------------

assert (
    be3_spatial["provider_metric_eligible"]
    ==
    (
        be3_spatial["spatial_coordinate_type"]
        == "provider_coordinate"
    )
).all()

assert (
    be3_spatial["zcta_centroid_fallback"]
    ==
    (
        be3_spatial["spatial_coordinate_type"]
        == "zcta_centroid"
    )
).all()

assert (
    be3_spatial["spatial_coordinate_unresolved"]
    ==
    (
        be3_spatial["spatial_coordinate_type"]
        == "unresolved"
    )
).all()

assert (
    be3_spatial["area_metric_eligible"]
    ==
    be3_spatial["spatial_coordinate_available"]
).all()


# ------------------------------------------------------------
# 10. Critical safety check:
# provider-level metrics must NOT contain centroid fallback
# ------------------------------------------------------------

assert not (
    be3_spatial.loc[
        be3_spatial["provider_metric_eligible"],
        "spatial_coordinate_type"
    ]
    == "zcta_centroid"
).any()

assert not (
    be3_spatial.loc[
        be3_spatial["provider_metric_eligible"],
        "spatial_coordinate_type"
    ]
    == "unresolved"
).any()


# ------------------------------------------------------------
# 11. Coordinate range validation for metric-eligible rows
# ------------------------------------------------------------

provider_metric_coords_valid = (
    be3_spatial.loc[
        be3_spatial["provider_metric_eligible"],
        "spatial_latitude"
    ].between(-90, 90)
    &
    be3_spatial.loc[
        be3_spatial["provider_metric_eligible"],
        "spatial_longitude"
    ].between(-180, 180)
)

assert provider_metric_coords_valid.all()


# ------------------------------------------------------------
# 12. Integrity checks
# ------------------------------------------------------------

assert len(be3_spatial) == len(spatial_geo)
assert be3_spatial["NPI"].is_unique

assert (
    be3_spatial["spatial_coordinate_available"]
    ==
    (
        be3_spatial["spatial_latitude"].notna()
        &
        be3_spatial["spatial_longitude"].notna()
    )
).all()

print("\n========== FINAL INTEGRITY CHECK ==========")
print("Row-count integrity: PASSED")
print("NPI uniqueness integrity: PASSED")
print("Provider-metric eligibility integrity: PASSED")
print("Area-metric eligibility integrity: PASSED")
print("Centroid fallback exclusion from provider metrics: PASSED")
print("Coordinate range validation: PASSED")
print("Stage 2C-38 Cell 3: PASSED")

Input provider rows: 281478
Input unique NPIs: 281478

========== FINAL SPATIAL COORDINATE POLICY ==========


,spatial_coordinate_type,spatial_coordinate_source,spatial_coordinate_confidence,provider_metric_eligible,area_metric_eligible,zcta_centroid_fallback,spatial_coordinate_unresolved,spatial_metric_policy,provider_count,coverage_percent
0,zcta_centroid,Census_ZCTA_Gazetteer_2020,area_level,False,True,True,False,area_level_fallback,274994,97.696445
1,unresolved,unresolved,unresolved,False,False,False,True,unresolved,4135,1.469031
2,provider_coordinate,Census_Geocoder,high,True,True,False,False,provider_level,2349,0.834523



========== METRIC ELIGIBILITY ==========
Provider-level metric eligible: 2349 (0.83%)
Area-level metric eligible: 277343 (98.53%)
Spatial coordinate unresolved: 4135 (1.47%)

========== FINAL INTEGRITY CHECK ==========
Row-count integrity: PASSED
NPI uniqueness integrity: PASSED
Provider-metric eligibility integrity: PASSED
Area-metric eligibility integrity: PASSED
Centroid fallback exclusion from provider metrics: PASSED
Coordinate range validation: PASSED
Stage 2C-38 Cell 3: PASSED


In [ ]:
# ============================================================
# STAGE 2C-39 — CELL 1
# BUILD PROVIDER-LEVEL BALLTREE
# ============================================================

import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

# ------------------------------------------------------------
# 1. Select provider-level metric records
# ------------------------------------------------------------

provider_spatial = be3_spatial[
    be3_spatial["provider_metric_eligible"]
].copy()

print("Total BE-3 provider rows:", len(be3_spatial))
print("Provider-level spatial rows:", len(provider_spatial))
print("Expected provider-level rows: 2349")


# ------------------------------------------------------------
# 2. Integrity checks before building the index
# ------------------------------------------------------------

assert len(provider_spatial) == 2349
assert provider_spatial["NPI"].is_unique

assert provider_spatial["spatial_latitude"].notna().all()
assert provider_spatial["spatial_longitude"].notna().all()

assert provider_spatial["spatial_coordinate_type"].eq(
    "provider_coordinate"
).all()

# Valid geographic range
assert provider_spatial["spatial_latitude"].between(-90, 90).all()
assert provider_spatial["spatial_longitude"].between(-180, 180).all()


# ------------------------------------------------------------
# 3. Create coordinate matrix
#
# BallTree haversine requires:
#     latitude  -> radians
#     longitude -> radians
# ------------------------------------------------------------

provider_coordinates_deg = provider_spatial[
    ["spatial_latitude", "spatial_longitude"]
].to_numpy(dtype=float)

provider_coordinates_rad = np.radians(
    provider_coordinates_deg
)

print("\nCoordinate matrix shape:", provider_coordinates_rad.shape)


# ------------------------------------------------------------
# 4. Build BallTree
#
# metric='haversine' returns angular distance in radians.
# Convert to miles/km only after querying.
# ------------------------------------------------------------

provider_balltree = BallTree(
    provider_coordinates_rad,
    metric="haversine"
)

print("\n========== BALLTREE ==========")
print("Index type: sklearn BallTree")
print("Distance metric: haversine")
print("Indexed providers:", provider_balltree.data.shape[0])
print("Coordinate dimensions:", provider_balltree.data.shape[1])


# ------------------------------------------------------------
# 5. Create stable index mapping
#
# BallTree row position must map back to the provider/NPI.
# This mapping will be used by all later spatial features.
# ------------------------------------------------------------

provider_spatial = provider_spatial.reset_index(drop=True)

provider_tree_index = pd.DataFrame({
    "tree_index": np.arange(len(provider_spatial), dtype=np.int32),
    "NPI": provider_spatial["NPI"].astype("string").to_numpy()
})

assert provider_tree_index["tree_index"].is_unique
assert provider_tree_index["NPI"].is_unique
assert len(provider_tree_index) == len(provider_spatial)


# ------------------------------------------------------------
# 6. Store radians on the provider-level table
#
# Keeping these avoids repeatedly converting coordinates in
# later BallTree queries.
# ------------------------------------------------------------

provider_spatial["latitude_rad"] = provider_coordinates_rad[:, 0]
provider_spatial["longitude_rad"] = provider_coordinates_rad[:, 1]
provider_spatial["tree_index"] = np.arange(
    len(provider_spatial),
    dtype=np.int32
)


# ------------------------------------------------------------
# 7. Final integrity checks
# ------------------------------------------------------------

assert len(provider_spatial) == 2349
assert provider_spatial["tree_index"].is_unique
assert provider_spatial["NPI"].is_unique

assert np.isfinite(
    provider_spatial[
        ["latitude_rad", "longitude_rad"]
    ].to_numpy()
).all()

assert provider_balltree.data.shape == (
    len(provider_spatial),
    2
)

print("\n========== FINAL INTEGRITY ==========")
print("Provider-level row count: PASSED")
print("NPI uniqueness: PASSED")
print("Coordinate validity: PASSED")
print("Tree/index mapping: PASSED")
print("BallTree dimensions: PASSED")
print("Stage 2C-39 Cell 1: PASSED")

Total BE-3 provider rows: 281478
Provider-level spatial rows: 2349
Expected provider-level rows: 2349

Coordinate matrix shape: (2349, 2)

========== BALLTREE ==========
Index type: sklearn BallTree
Distance metric: haversine
Indexed providers: 2349
Coordinate dimensions: 2

========== FINAL INTEGRITY ==========
Provider-level row count: PASSED
NPI uniqueness: PASSED
Coordinate validity: PASSED
Tree/index mapping: PASSED
BallTree dimensions: PASSED
Stage 2C-39 Cell 1: PASSED


In [ ]:
# ============================================================
# STAGE 2C-39 — CELL 2
# NEAREST & SECOND-NEAREST PROVIDER DISTANCES
#
# Handles providers sharing identical coordinates.
# ============================================================

import numpy as np
import pandas as pd

EARTH_RADIUS_MILES = 3958.7613

# ------------------------------------------------------------
# 1. Prepare provider-level coordinate table
# ------------------------------------------------------------

provider_spatial = provider_spatial.reset_index(drop=True)

assert len(provider_spatial) == 2349
assert provider_spatial["NPI"].is_unique
assert provider_spatial["tree_index"].is_unique


# ------------------------------------------------------------
# 2. Identify exact coordinate-sharing groups
# ------------------------------------------------------------

provider_spatial["coordinate_group"] = (
    provider_spatial[
        ["spatial_latitude", "spatial_longitude"]
    ]
    .astype(str)
    .agg("|".join, axis=1)
)

coordinate_group_sizes = (
    provider_spatial
    .groupby("coordinate_group")
    .size()
)

provider_spatial["coordinate_group_size"] = (
    provider_spatial["coordinate_group"]
    .map(coordinate_group_sizes)
    .astype(int)
)

print("========== COORDINATE GROUP AUDIT ==========")
print(
    "Provider records:",
    len(provider_spatial)
)

print(
    "Unique coordinate groups:",
    len(coordinate_group_sizes)
)

print(
    "Providers sharing coordinates:",
    int(
        (provider_spatial["coordinate_group_size"] > 1).sum()
    )
)

print(
    "Largest coordinate group:",
    int(coordinate_group_sizes.max())
)


# ------------------------------------------------------------
# 3. Build coordinate-group membership
# ------------------------------------------------------------

coordinate_group_members = (
    provider_spatial
    .groupby("coordinate_group")["tree_index"]
    .apply(list)
    .to_dict()
)


# ------------------------------------------------------------
# 4. Query BallTree
#
# We still use BallTree for geographic ordering.
#
# Query enough neighbors to capture ordinary nearby
# providers. Exact-coordinate groups are handled separately.
# ------------------------------------------------------------

K_QUERY = min(50, len(provider_spatial))

query_coordinates = provider_spatial[
    ["latitude_rad", "longitude_rad"]
].to_numpy()

distances_rad, indices = provider_balltree.query(
    query_coordinates,
    k=K_QUERY
)

distances_miles = (
    distances_rad * EARTH_RADIUS_MILES
)

print("\n========== BALLTREE QUERY ==========")
print("Query matrix shape:", distances_miles.shape)


# ------------------------------------------------------------
# 5. Calculate nearest / second-nearest providers
# ------------------------------------------------------------

nearest_indices = np.full(
    len(provider_spatial),
    -1,
    dtype=np.int32
)

second_nearest_indices = np.full(
    len(provider_spatial),
    -1,
    dtype=np.int32
)

nearest_distances = np.full(
    len(provider_spatial),
    np.nan,
    dtype=float
)

second_nearest_distances = np.full(
    len(provider_spatial),
    np.nan,
    dtype=float
)


for row_num in range(len(provider_spatial)):

    own_tree_index = int(
        provider_spatial.iloc[row_num]["tree_index"]
    )

    own_group = provider_spatial.iloc[
        row_num
    ]["coordinate_group"]

    same_coordinate_members = (
        coordinate_group_members[own_group]
    )

    # --------------------------------------------------------
    # A. Other providers at EXACT same coordinate
    # --------------------------------------------------------

    same_coordinate_others = [
        idx
        for idx in same_coordinate_members
        if idx != own_tree_index
    ]

    # --------------------------------------------------------
    # B. If there are >= 2 other providers at the same
    # coordinate, both nearest neighbors are zero miles.
    # --------------------------------------------------------

    if len(same_coordinate_others) >= 2:

        nearest_indices[row_num] = (
            same_coordinate_others[0]
        )

        second_nearest_indices[row_num] = (
            same_coordinate_others[1]
        )

        nearest_distances[row_num] = 0.0
        second_nearest_distances[row_num] = 0.0

        continue

    # --------------------------------------------------------
    # C. If exactly one other provider shares the coordinate,
    # it is necessarily the nearest provider at 0 miles.
    # We then find the next geographic provider using BallTree.
    # --------------------------------------------------------

    if len(same_coordinate_others) == 1:

        nearest_indices[row_num] = (
            same_coordinate_others[0]
        )

        nearest_distances[row_num] = 0.0

        # Find the closest BallTree result that:
        #   1. is not self
        #   2. is not the exact-coordinate neighbor
        excluded = {
            own_tree_index,
            same_coordinate_others[0]
        }

        found_second = False

        for candidate_idx, candidate_distance in zip(
            indices[row_num],
            distances_miles[row_num]
        ):

            candidate_idx = int(candidate_idx)

            if candidate_idx not in excluded:
                second_nearest_indices[row_num] = (
                    candidate_idx
                )
                second_nearest_distances[row_num] = (
                    float(candidate_distance)
                )
                found_second = True
                break

        if not found_second:
            raise ValueError(
                "Could not find second-nearest provider "
                f"for tree_index={own_tree_index}"
            )

        continue

    # --------------------------------------------------------
    # D. No other provider shares exact coordinate.
    #
    # Find first two BallTree results that are not self.
    # --------------------------------------------------------

    found = []

    for candidate_idx, candidate_distance in zip(
        indices[row_num],
        distances_miles[row_num]
    ):

        candidate_idx = int(candidate_idx)

        if candidate_idx == own_tree_index:
            continue

        found.append(
            (candidate_idx, float(candidate_distance))
        )

        if len(found) == 2:
            break

    if len(found) < 2:
        raise ValueError(
            "Could not find two other providers for "
            f"tree_index={own_tree_index}"
        )

    nearest_indices[row_num] = found[0][0]
    nearest_distances[row_num] = found[0][1]

    second_nearest_indices[row_num] = found[1][0]
    second_nearest_distances[row_num] = found[1][1]


# ------------------------------------------------------------
# 6. Validate all providers received two neighbors
# ------------------------------------------------------------

assert (nearest_indices >= 0).all()
assert (second_nearest_indices >= 0).all()

assert np.isfinite(nearest_distances).all()
assert np.isfinite(second_nearest_distances).all()


# ------------------------------------------------------------
# 7. Map tree indices back to NPIs
# ------------------------------------------------------------

tree_to_npi = (
    provider_spatial
    .set_index("tree_index")["NPI"]
)

nearest_npi = tree_to_npi.loc[
    nearest_indices
].to_numpy()

second_nearest_npi = tree_to_npi.loc[
    second_nearest_indices
].to_numpy()


# ------------------------------------------------------------
# 8. Build feature table
# ------------------------------------------------------------

nearest_features = pd.DataFrame({
    "NPI": provider_spatial["NPI"].to_numpy(),

    "nearest_provider_npi":
        nearest_npi,

    "nearest_provider_distance_miles":
        nearest_distances,

    "second_nearest_provider_npi":
        second_nearest_npi,

    "second_nearest_provider_distance_miles":
        second_nearest_distances
})


# ------------------------------------------------------------
# 9. Critical validation:
# provider cannot be its own neighbor
# ------------------------------------------------------------

assert (
    nearest_features["NPI"]
    !=
    nearest_features["nearest_provider_npi"]
).all()

assert (
    nearest_features["NPI"]
    !=
    nearest_features["second_nearest_provider_npi"]
).all()


# ------------------------------------------------------------
# 10. Distance ordering
# ------------------------------------------------------------

assert (
    nearest_features[
        "nearest_provider_distance_miles"
    ]
    <=
    nearest_features[
        "second_nearest_provider_distance_miles"
    ]
).all()

assert (
    nearest_features[
        "nearest_provider_distance_miles"
    ] >= 0
).all()

assert (
    nearest_features[
        "second_nearest_provider_distance_miles"
    ] >= 0
).all()


# ------------------------------------------------------------
# 11. Distance summary
# ------------------------------------------------------------

print("\n========== DISTANCE SUMMARY ==========")

display(
    nearest_features[
        [
            "nearest_provider_distance_miles",
            "second_nearest_provider_distance_miles"
        ]
    ].describe()
)


# ------------------------------------------------------------
# 12. Zero-distance audit
# ------------------------------------------------------------

nearest_zero = (
    nearest_features[
        "nearest_provider_distance_miles"
    ] == 0
)

second_zero = (
    nearest_features[
        "second_nearest_provider_distance_miles"
    ] == 0
)

print("\n========== ZERO-DISTANCE AUDIT ==========")

print(
    "Providers with nearest provider at 0 miles:",
    int(nearest_zero.sum())
)

print(
    "Providers with second-nearest provider at 0 miles:",
    int(second_zero.sum())
)


# ------------------------------------------------------------
# 13. Attach features to full BE-3 layer
# ------------------------------------------------------------

be3_spatial = be3_spatial.drop(
    columns=[
        "nearest_provider_npi",
        "nearest_provider_distance_miles",
        "second_nearest_provider_npi",
        "second_nearest_provider_distance_miles"
    ],
    errors="ignore"
)

be3_spatial = be3_spatial.merge(
    nearest_features,
    on="NPI",
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# 14. Coverage
# ------------------------------------------------------------

nearest_available = (
    be3_spatial[
        "nearest_provider_distance_miles"
    ].notna()
)

second_nearest_available = (
    be3_spatial[
        "second_nearest_provider_distance_miles"
    ].notna()
)

print("\n========== FULL BE-3 COVERAGE ==========")

print(
    "Nearest-provider distance available:",
    int(nearest_available.sum())
)

print(
    "Second-nearest-provider distance available:",
    int(second_nearest_available.sum())
)

assert nearest_available.sum() == 2349
assert second_nearest_available.sum() == 2349


# ------------------------------------------------------------
# 15. Non-provider-coordinate records remain NaN
# ------------------------------------------------------------

non_provider_rows = (
    ~be3_spatial["provider_metric_eligible"]
)

assert (
    be3_spatial.loc[
        non_provider_rows,
        "nearest_provider_distance_miles"
    ].isna()
).all()

assert (
    be3_spatial.loc[
        non_provider_rows,
        "second_nearest_provider_distance_miles"
    ].isna()
).all()


# ------------------------------------------------------------
# 16. Final integrity
# ------------------------------------------------------------

assert len(be3_spatial) == 281478
assert be3_spatial["NPI"].is_unique

print("\n========== FINAL INTEGRITY ==========")
print("Coordinate-group handling: PASSED")
print("Nearest-neighbor calculation: PASSED")
print("Second-nearest-neighbor calculation: PASSED")
print("Self-NPI exclusion: PASSED")
print("Distance ordering: PASSED")
print("Full BE-3 coverage: PASSED")
print("Non-provider rows remain NaN: PASSED")
print("Full BE-3 row count: PASSED")
print("Full BE-3 NPI uniqueness: PASSED")
print("Stage 2C-39 Cell 2: PASSED")

========== COORDINATE GROUP AUDIT ==========
Provider records: 2349
Unique coordinate groups: 418
Providers sharing coordinates: 2063
Largest coordinate group: 564

========== BALLTREE QUERY ==========
Query matrix shape: (2349, 50)

========== DISTANCE SUMMARY ==========


,nearest_provider_distance_miles,second_nearest_provider_distance_miles
count,2349.000000,2349.000000
mean,2.674375,4.802776
std,14.364552,20.527230
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
max,234.855418,262.103672



========== ZERO-DISTANCE AUDIT ==========
Providers with nearest provider at 0 miles: 2063
Providers with second-nearest provider at 0 miles: 1969

========== FULL BE-3 COVERAGE ==========
Nearest-provider distance available: 2349
Second-nearest-provider distance available: 2349

========== FINAL INTEGRITY ==========
Coordinate-group handling: PASSED
Nearest-neighbor calculation: PASSED
Second-nearest-neighbor calculation: PASSED
Self-NPI exclusion: PASSED
Distance ordering: PASSED
Full BE-3 coverage: PASSED
Non-provider rows remain NaN: PASSED
Full BE-3 row count: PASSED
Full BE-3 NPI uniqueness: PASSED
Stage 2C-39 Cell 2: PASSED


In [ ]:
# ============================================================
# STAGE 2C-39 — CELL 3
# PROVIDER DENSITY BY RADIUS
# ============================================================

import numpy as np
import pandas as pd

EARTH_RADIUS_MILES = 3958.7613

DENSITY_RADII_MILES = [5, 10, 25, 50, 100]

# ------------------------------------------------------------
# 1. Confirm provider-level spatial population
# ------------------------------------------------------------

assert len(provider_spatial) == 2349
assert provider_spatial["NPI"].is_unique

provider_coordinates_rad = provider_spatial[
    ["latitude_rad", "longitude_rad"]
].to_numpy()

print("Provider-level records:", len(provider_spatial))
print("Density radii:", DENSITY_RADII_MILES)


# ------------------------------------------------------------
# 2. Convert radii from miles to radians
# ------------------------------------------------------------

DENSITY_RADII_RAD = {
    radius_miles: radius_miles / EARTH_RADIUS_MILES
    for radius_miles in DENSITY_RADII_MILES
}


# ------------------------------------------------------------
# 3. Calculate provider counts within each radius
#
# query_radius includes the provider itself.
# Therefore subtract 1 from every count.
# ------------------------------------------------------------

density_features = pd.DataFrame({
    "NPI": provider_spatial["NPI"].to_numpy()
})

for radius_miles in DENSITY_RADII_MILES:

    radius_rad = DENSITY_RADII_RAD[radius_miles]

    neighbor_indices = provider_balltree.query_radius(
        provider_coordinates_rad,
        r=radius_rad
    )

    provider_counts = np.array(
        [
            len(neighbors) - 1
            for neighbors in neighbor_indices
        ],
        dtype=np.int32
    )

    density_features[
        f"provider_count_{radius_miles}mi"
    ] = provider_counts


# ------------------------------------------------------------
# 4. Basic density summary
# ------------------------------------------------------------

print("\n========== PROVIDER DENSITY SUMMARY ==========")

density_columns = [
    f"provider_count_{r}mi"
    for r in DENSITY_RADII_MILES
]

display(
    density_features[density_columns].describe()
)


# ------------------------------------------------------------
# 5. Density monotonicity validation
#
# Counts should never decrease as radius increases.
# ------------------------------------------------------------

for smaller, larger in zip(
    DENSITY_RADII_MILES[:-1],
    DENSITY_RADII_MILES[1:]
):

    smaller_col = f"provider_count_{smaller}mi"
    larger_col = f"provider_count_{larger}mi"

    assert (
        density_features[smaller_col]
        <=
        density_features[larger_col]
    ).all()


# ------------------------------------------------------------
# 6. Non-negative validation
# ------------------------------------------------------------

for column in density_columns:

    assert (
        density_features[column] >= 0
    ).all()

    assert pd.api.types.is_integer_dtype(
        density_features[column]
    )


# ------------------------------------------------------------
# 7. Attach density features to full BE-3 layer
#
# Only provider-coordinate records receive these metrics.
# ZCTA-centroid and unresolved records remain NaN.
# ------------------------------------------------------------

be3_spatial = be3_spatial.drop(
    columns=density_columns,
    errors="ignore"
)

be3_spatial = be3_spatial.merge(
    density_features,
    on="NPI",
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# 8. Coverage validation
# ------------------------------------------------------------

print("\n========== FULL BE-3 DENSITY COVERAGE ==========")

for column in density_columns:

    available = int(
        be3_spatial[column].notna().sum()
    )

    print(
        f"{column}: {available}"
    )

    assert available == 2349


# ------------------------------------------------------------
# 9. Verify non-provider-coordinate rows remain NaN
# ------------------------------------------------------------

non_provider_rows = (
    ~be3_spatial["provider_metric_eligible"]
)

for column in density_columns:

    assert (
        be3_spatial.loc[
            non_provider_rows,
            column
        ].isna()
    ).all()


# ------------------------------------------------------------
# 10. Verify no self-count contamination
#
# Because self is explicitly removed, a provider with no
# other provider inside a radius should have count = 0.
# ------------------------------------------------------------

print("\n========== ZERO-DENSITY COUNTS ==========")

for column in density_columns:

    zero_count = int(
        (
            density_features[column] == 0
        ).sum()
    )

    print(
        f"{column}: {zero_count} providers with zero neighbors"
    )


# ------------------------------------------------------------
# 11. Full-layer integrity
# ------------------------------------------------------------

assert len(be3_spatial) == 281478
assert be3_spatial["NPI"].is_unique

print("\n========== FINAL INTEGRITY ==========")
print("Density calculation: PASSED")
print("Radius monotonicity: PASSED")
print("Non-negative counts: PASSED")
print("Provider-level coverage: PASSED")
print("Non-provider rows remain NaN: PASSED")
print("Full BE-3 row count: PASSED")
print("Full BE-3 NPI uniqueness: PASSED")
print("Stage 2C-39 Cell 3: PASSED")

Provider-level records: 2349
Density radii: [5, 10, 25, 50, 100]

========== PROVIDER DENSITY SUMMARY ==========


,provider_count_5mi,provider_count_10mi,provider_count_25mi,provider_count_50mi,provider_count_100mi
count,2349.000000,2349.000000,2349.000000,2349.000000,2349.000000
mean,205.228608,206.252022,208.603661,226.352490,282.319285
std,218.354234,217.563661,216.838632,214.885979,230.415334
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,18.000000,25.000000,30.000000,49.000000,71.000000
50%,80.000000,82.000000,82.000000,118.000000,208.000000
75%,267.000000,267.000000,268.000000,325.000000,604.000000
max,564.000000,564.000000,565.000000,565.000000,717.000000



========== FULL BE-3 DENSITY COVERAGE ==========
provider_count_5mi: 2349
provider_count_10mi: 2349
provider_count_25mi: 2349
provider_count_50mi: 2349
provider_count_100mi: 2349

========== ZERO-DENSITY COUNTS ==========
provider_count_5mi: 138 providers with zero neighbors
provider_count_10mi: 115 providers with zero neighbors
provider_count_25mi: 73 providers with zero neighbors
provider_count_50mi: 48 providers with zero neighbors
provider_count_100mi: 15 providers with zero neighbors

========== FINAL INTEGRITY ==========
Density calculation: PASSED
Radius monotonicity: PASSED
Non-negative counts: PASSED
Provider-level coverage: PASSED
Non-provider rows remain NaN: PASSED
Full BE-3 row count: PASSED
Full BE-3 NPI uniqueness: PASSED
Stage 2C-39 Cell 3: PASSED


In [ ]:
# ============================================================
# STAGE 2C-39 — CELL 4
# SAME-TAXONOMY PROVIDER DENSITY BY RADIUS
# ============================================================

import numpy as np
import pandas as pd

EARTH_RADIUS_MILES = 3958.7613

TAXONOMY_DENSITY_RADII_MILES = [5, 10, 25, 50, 100]

# ------------------------------------------------------------
# 1. Identify the primary taxonomy field
# ------------------------------------------------------------

TAXONOMY_COL = "Healthcare Provider Taxonomy Code_1"

assert TAXONOMY_COL in provider_spatial.columns

print("Provider-level records:", len(provider_spatial))
print("Taxonomy field:", TAXONOMY_COL)
print("Radii:", TAXONOMY_DENSITY_RADII_MILES)


# ------------------------------------------------------------
# 2. Prepare taxonomy values
#
# Missing taxonomy values are retained as missing and will
# not participate in same-taxonomy matching.
# ------------------------------------------------------------

provider_taxonomy = (
    provider_spatial[TAXONOMY_COL]
    .astype("string")
    .str.strip()
)

provider_taxonomy = provider_taxonomy.replace(
    {
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA
    }
)

print("\n========== TAXONOMY COVERAGE ==========")

print(
    "Providers with taxonomy:",
    int(provider_taxonomy.notna().sum())
)

print(
    "Providers missing taxonomy:",
    int(provider_taxonomy.isna().sum())
)

print(
    "Unique taxonomies:",
    int(provider_taxonomy.dropna().nunique())
)


# ------------------------------------------------------------
# 3. Convert provider coordinates to radians
# ------------------------------------------------------------

provider_coordinates_rad = provider_spatial[
    ["latitude_rad", "longitude_rad"]
].to_numpy()

assert provider_coordinates_rad.shape == (2349, 2)


# ------------------------------------------------------------
# 4. Query the BallTree once at the largest radius
#
# All smaller-radius counts can be derived from the same
# neighbor lists.
# ------------------------------------------------------------

MAX_RADIUS_MILES = max(TAXONOMY_DENSITY_RADII_MILES)

MAX_RADIUS_RAD = (
    MAX_RADIUS_MILES / EARTH_RADIUS_MILES
)

neighbor_indices_by_provider = provider_balltree.query_radius(
    provider_coordinates_rad,
    r=MAX_RADIUS_RAD
)

print("\n========== BALLTREE QUERY ==========")
print("Maximum radius:", MAX_RADIUS_MILES, "miles")
print(
    "Providers queried:",
    len(neighbor_indices_by_provider)
)


# ------------------------------------------------------------
# 5. Calculate same-taxonomy counts
#
# For each provider:
#   - inspect providers within 100 miles
#   - exclude self
#   - require same taxonomy
#   - apply each radius independently
# ------------------------------------------------------------

same_taxonomy_features = pd.DataFrame({
    "NPI": provider_spatial["NPI"].to_numpy()
})

taxonomy_array = provider_taxonomy.to_numpy()

for radius_miles in TAXONOMY_DENSITY_RADII_MILES:

    radius_rad = (
        radius_miles / EARTH_RADIUS_MILES
    )

    radius_column = (
        f"same_taxonomy_count_{radius_miles}mi"
    )

    counts = np.zeros(
        len(provider_spatial),
        dtype=np.int32
    )

    for provider_idx, neighbor_indices in enumerate(
        neighbor_indices_by_provider
    ):

        if pd.isna(taxonomy_array[provider_idx]):
            counts[provider_idx] = 0
            continue

        provider_lat = provider_coordinates_rad[
            provider_idx, 0
        ]

        provider_lon = provider_coordinates_rad[
            provider_idx, 1
        ]

        # Haversine angular distance from the current provider
        # to every returned neighbor.
        neighbor_coordinates = provider_coordinates_rad[
            neighbor_indices
        ]

        lat1 = provider_lat
        lon1 = provider_lon

        lat2 = neighbor_coordinates[:, 0]
        lon2 = neighbor_coordinates[:, 1]

        dlat = lat2 - lat1
        dlon = lon2 - lon1

        a = (
            np.sin(dlat / 2.0) ** 2
            +
            np.cos(lat1)
            * np.cos(lat2)
            * np.sin(dlon / 2.0) ** 2
        )

        angular_distances = 2.0 * np.arcsin(
            np.sqrt(np.clip(a, 0, 1))
        )

        # Radius filter.
        within_radius = (
            angular_distances <= radius_rad + 1e-12
        )

        candidate_indices = neighbor_indices[
            within_radius
        ]

        # Exclude self explicitly.
        candidate_indices = candidate_indices[
            candidate_indices != provider_idx
        ]

        # Same taxonomy only.
        same_taxonomy = (
            taxonomy_array[candidate_indices]
            == taxonomy_array[provider_idx]
        )

        # Missing taxonomy never counts.
        valid_taxonomy = pd.notna(
            taxonomy_array[candidate_indices]
        )

        counts[provider_idx] = int(
            np.sum(
                same_taxonomy & valid_taxonomy
            )
        )

    same_taxonomy_features[
        radius_column
    ] = counts


# ------------------------------------------------------------
# 6. Summary statistics
# ------------------------------------------------------------

taxonomy_density_columns = [
    f"same_taxonomy_count_{r}mi"
    for r in TAXONOMY_DENSITY_RADII_MILES
]

print("\n========== SAME-TAXONOMY DENSITY SUMMARY ==========")

display(
    same_taxonomy_features[
        taxonomy_density_columns
    ].describe()
)


# ------------------------------------------------------------
# 7. Radius monotonicity validation
# ------------------------------------------------------------

for smaller, larger in zip(
    TAXONOMY_DENSITY_RADII_MILES[:-1],
    TAXONOMY_DENSITY_RADII_MILES[1:]
):

    smaller_col = (
        f"same_taxonomy_count_{smaller}mi"
    )

    larger_col = (
        f"same_taxonomy_count_{larger}mi"
    )

    assert (
        same_taxonomy_features[smaller_col]
        <=
        same_taxonomy_features[larger_col]
    ).all()


# ------------------------------------------------------------
# 8. Non-negative integer validation
# ------------------------------------------------------------

for column in taxonomy_density_columns:

    assert (
        same_taxonomy_features[column] >= 0
    ).all()

    assert pd.api.types.is_integer_dtype(
        same_taxonomy_features[column]
    )


# ------------------------------------------------------------
# 9. Attach features to full BE-3 layer
#
# Only provider-coordinate records receive these metrics.
# ------------------------------------------------------------

be3_spatial = be3_spatial.drop(
    columns=taxonomy_density_columns,
    errors="ignore"
)

be3_spatial = be3_spatial.merge(
    same_taxonomy_features,
    on="NPI",
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# 10. Coverage validation
# ------------------------------------------------------------

print("\n========== FULL BE-3 SAME-TAXONOMY COVERAGE ==========")

for column in taxonomy_density_columns:

    available = int(
        be3_spatial[column].notna().sum()
    )

    print(
        f"{column}: {available}"
    )

    assert available == 2349


# ------------------------------------------------------------
# 11. Verify non-provider rows remain NaN
# ------------------------------------------------------------

non_provider_rows = (
    ~be3_spatial["provider_metric_eligible"]
)

for column in taxonomy_density_columns:

    assert (
        be3_spatial.loc[
            non_provider_rows,
            column
        ].isna()
    ).all()


# ------------------------------------------------------------
# 12. Zero-density audit
# ------------------------------------------------------------

print("\n========== ZERO SAME-TAXONOMY DENSITY ==========")

for column in taxonomy_density_columns:

    zero_count = int(
        (
            same_taxonomy_features[column] == 0
        ).sum()
    )

    print(
        f"{column}: "
        f"{zero_count} providers with zero same-taxonomy neighbors"
    )


# ------------------------------------------------------------
# 13. Full BE-3 integrity
# ------------------------------------------------------------

assert len(be3_spatial) == 281478
assert be3_spatial["NPI"].is_unique

print("\n========== FINAL INTEGRITY ==========")
print("Same-taxonomy density calculation: PASSED")
print("Radius monotonicity: PASSED")
print("Non-negative counts: PASSED")
print("Provider-level coverage: PASSED")
print("Non-provider rows remain NaN: PASSED")
print("Full BE-3 row count: PASSED")
print("Full BE-3 NPI uniqueness: PASSED")
print("Stage 2C-39 Cell 4: PASSED")

Provider-level records: 2349
Taxonomy field: Healthcare Provider Taxonomy Code_1
Radii: [5, 10, 25, 50, 100]

========== TAXONOMY COVERAGE ==========
Providers with taxonomy: 2349
Providers missing taxonomy: 0
Unique taxonomies: 231

========== BALLTREE QUERY ==========
Maximum radius: 100 miles
Providers queried: 2349

========== SAME-TAXONOMY DENSITY SUMMARY ==========


,same_taxonomy_count_5mi,same_taxonomy_count_10mi,same_taxonomy_count_25mi,same_taxonomy_count_50mi,same_taxonomy_count_100mi
count,2349.000000,2349.000000,2349.00000,2349.000000,2349.000000
mean,10.213708,10.234142,10.36441,11.003831,12.316731
std,17.351717,17.342394,17.46859,17.703369,18.378154
min,0.000000,0.000000,0.00000,0.000000,0.000000
25%,0.000000,0.000000,0.00000,0.000000,1.000000
50%,3.000000,3.000000,3.00000,4.000000,5.000000
75%,11.000000,11.000000,11.00000,12.000000,14.000000
max,69.000000,69.000000,70.00000,70.000000,70.000000



========== FULL BE-3 SAME-TAXONOMY COVERAGE ==========
same_taxonomy_count_5mi: 2349
same_taxonomy_count_10mi: 2349
same_taxonomy_count_25mi: 2349
same_taxonomy_count_50mi: 2349
same_taxonomy_count_100mi: 2349

========== ZERO SAME-TAXONOMY DENSITY ==========
same_taxonomy_count_5mi: 719 providers with zero same-taxonomy neighbors
same_taxonomy_count_10mi: 703 providers with zero same-taxonomy neighbors
same_taxonomy_count_25mi: 676 providers with zero same-taxonomy neighbors
same_taxonomy_count_50mi: 626 providers with zero same-taxonomy neighbors
same_taxonomy_count_100mi: 547 providers with zero same-taxonomy neighbors

========== FINAL INTEGRITY ==========
Same-taxonomy density calculation: PASSED
Radius monotonicity: PASSED
Non-negative counts: PASSED
Provider-level coverage: PASSED
Non-provider rows remain NaN: PASSED
Full BE-3 row count: PASSED
Full BE-3 NPI uniqueness: PASSED
Stage 2C-39 Cell 4: PASSED


In [ ]:
# ============================================================
# STAGE 2C-39 — CELL 5
# CONSOLIDATE + AUDIT PROVIDER-LEVEL SPATIAL FEATURES
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Define the expected provider-level spatial features
# ------------------------------------------------------------

nearest_features = [
    "nearest_provider_npi",
    "nearest_provider_distance_miles",
    "second_nearest_provider_npi",
    "second_nearest_provider_distance_miles",
]

provider_density_features = [
    "provider_count_5mi",
    "provider_count_10mi",
    "provider_count_25mi",
    "provider_count_50mi",
    "provider_count_100mi",
]

same_taxonomy_density_features = [
    "same_taxonomy_count_5mi",
    "same_taxonomy_count_10mi",
    "same_taxonomy_count_25mi",
    "same_taxonomy_count_50mi",
    "same_taxonomy_count_100mi",
]

provider_spatial_features = (
    nearest_features
    + provider_density_features
    + same_taxonomy_density_features
)

print("Expected provider-level spatial features:")
for feature in provider_spatial_features:
    print(" -", feature)


# ------------------------------------------------------------
# 2. Confirm every expected feature exists
# ------------------------------------------------------------

missing_features = [
    feature
    for feature in provider_spatial_features
    if feature not in be3_spatial.columns
]

assert len(missing_features) == 0, (
    f"Missing expected spatial features: {missing_features}"
)

print("\nFeature existence check: PASSED")


# ------------------------------------------------------------
# 3. Confirm BE-3 base integrity
# ------------------------------------------------------------

assert len(be3_spatial) == 281478
assert be3_spatial["NPI"].is_unique

print("BE-3 row count:", len(be3_spatial))
print("NPI uniqueness: PASSED")


# ------------------------------------------------------------
# 4. Confirm provider-level eligibility population
# ------------------------------------------------------------

provider_eligible_mask = (
    be3_spatial["provider_metric_eligible"]
)

provider_eligible_count = int(
    provider_eligible_mask.sum()
)

non_provider_count = int(
    (~provider_eligible_mask).sum()
)

assert provider_eligible_count == 2349
assert non_provider_count == 279129

print("\n========== PROVIDER-LEVEL POPULATION ==========")
print("Provider-metric eligible:", provider_eligible_count)
print("Non-provider-metric rows:", non_provider_count)


# ------------------------------------------------------------
# 5. Check nearest-provider features
# ------------------------------------------------------------

print("\n========== NEAREST-PROVIDER FEATURES ==========")

for feature in nearest_features:

    provider_available = int(
        be3_spatial.loc[
            provider_eligible_mask,
            feature
        ].notna().sum()
    )

    non_provider_missing = int(
        be3_spatial.loc[
            ~provider_eligible_mask,
            feature
        ].isna().sum()
    )

    assert provider_available == 2349
    assert non_provider_missing == 279129

    print(
        f"{feature}: "
        f"{provider_available}/2349 provider rows populated"
    )


# ------------------------------------------------------------
# 6. Validate nearest-distance ordering
# ------------------------------------------------------------

nearest_distance = (
    be3_spatial.loc[
        provider_eligible_mask,
        "nearest_provider_distance_miles"
    ]
)

second_distance = (
    be3_spatial.loc[
        provider_eligible_mask,
        "second_nearest_provider_distance_miles"
    ]
)

assert (
    nearest_distance <= second_distance
).all()

assert (
    nearest_distance >= 0
).all()

assert (
    second_distance >= 0
).all()

print("\nNearest-distance ordering: PASSED")
print("Non-negative distances: PASSED")


# ------------------------------------------------------------
# 7. Check provider density features
# ------------------------------------------------------------

print("\n========== PROVIDER DENSITY FEATURES ==========")

for feature in provider_density_features:

    values = be3_spatial.loc[
        provider_eligible_mask,
        feature
    ]

    assert values.notna().all()
    assert (values >= 0).all()

    print(
        f"{feature}: "
        f"min={int(values.min())}, "
        f"median={int(values.median())}, "
        f"max={int(values.max())}"
    )

    # Non-provider rows must remain unavailable.
    assert (
        be3_spatial.loc[
            ~provider_eligible_mask,
            feature
        ].isna()
    ).all()


# ------------------------------------------------------------
# 8. Validate provider-density monotonicity
# ------------------------------------------------------------

for smaller, larger in zip(
    [5, 10, 25, 50],
    [10, 25, 50, 100]
):

    smaller_feature = (
        f"provider_count_{smaller}mi"
    )

    larger_feature = (
        f"provider_count_{larger}mi"
    )

    assert (
        be3_spatial.loc[
            provider_eligible_mask,
            smaller_feature
        ]
        <=
        be3_spatial.loc[
            provider_eligible_mask,
            larger_feature
        ]
    ).all()

print("\nProvider-density monotonicity: PASSED")


# ------------------------------------------------------------
# 9. Check same-taxonomy density features
# ------------------------------------------------------------

print("\n========== SAME-TAXONOMY DENSITY FEATURES ==========")

for feature in same_taxonomy_density_features:

    values = be3_spatial.loc[
        provider_eligible_mask,
        feature
    ]

    assert values.notna().all()
    assert (values >= 0).all()

    print(
        f"{feature}: "
        f"min={int(values.min())}, "
        f"median={int(values.median())}, "
        f"max={int(values.max())}"
    )

    # Non-provider rows must remain unavailable.
    assert (
        be3_spatial.loc[
            ~provider_eligible_mask,
            feature
        ].isna()
    ).all()


# ------------------------------------------------------------
# 10. Validate same-taxonomy density monotonicity
# ------------------------------------------------------------

for smaller, larger in zip(
    [5, 10, 25, 50],
    [10, 25, 50, 100]
):

    smaller_feature = (
        f"same_taxonomy_count_{smaller}mi"
    )

    larger_feature = (
        f"same_taxonomy_count_{larger}mi"
    )

    assert (
        be3_spatial.loc[
            provider_eligible_mask,
            smaller_feature
        ]
        <=
        be3_spatial.loc[
            provider_eligible_mask,
            larger_feature
        ]
    ).all()

print("\nSame-taxonomy-density monotonicity: PASSED")


# ------------------------------------------------------------
# 11. Verify all provider-level features have identical
#     provider-level coverage
# ------------------------------------------------------------

print("\n========== FEATURE COVERAGE AUDIT ==========")

coverage_rows = []

for feature in provider_spatial_features:

    provider_non_null = int(
        be3_spatial.loc[
            provider_eligible_mask,
            feature
        ].notna().sum()
    )

    provider_null = int(
        be3_spatial.loc[
            provider_eligible_mask,
            feature
        ].isna().sum()
    )

    non_provider_non_null = int(
        be3_spatial.loc[
            ~provider_eligible_mask,
            feature
        ].notna().sum()
    )

    coverage_rows.append({
        "feature": feature,
        "provider_rows": provider_eligible_count,
        "provider_non_null": provider_non_null,
        "provider_null": provider_null,
        "non_provider_non_null": non_provider_non_null,
    })

coverage_audit = pd.DataFrame(coverage_rows)

display(coverage_audit)

assert (
    coverage_audit["provider_non_null"] == 2349
).all()

assert (
    coverage_audit["provider_null"] == 0
).all()

assert (
    coverage_audit["non_provider_non_null"] == 0
).all()

print("Coverage consistency: PASSED")


# ------------------------------------------------------------
# 12. Confirm spatial policy is preserved
# ------------------------------------------------------------

print("\n========== SPATIAL POLICY AUDIT ==========")

policy_counts = (
    be3_spatial["spatial_metric_policy"]
    .value_counts(dropna=False)
)

display(policy_counts)

assert (
    policy_counts.get("provider_level", 0)
    == 2349
)

assert (
    policy_counts.get("area_level_fallback", 0)
    == 274994
)

assert (
    policy_counts.get("unresolved", 0)
    == 4135
)

print("Spatial policy counts: PASSED")


# ------------------------------------------------------------
# 13. Create the explicit provider-level feature subset
#
# This is the clean BE-3 provider-level spatial feature table.
# ------------------------------------------------------------

be3_provider_spatial_features = be3_spatial.loc[
    provider_eligible_mask,
    [
        "NPI",
        "spatial_latitude",
        "spatial_longitude",
        "spatial_coordinate_type",
        "spatial_coordinate_source",
        "spatial_coordinate_confidence",
        "provider_metric_eligible",
    ]
    + provider_spatial_features
].copy()

assert len(be3_provider_spatial_features) == 2349
assert be3_provider_spatial_features["NPI"].is_unique

print("\n========== PROVIDER FEATURE TABLE ==========")
print(
    "Rows:",
    len(be3_provider_spatial_features)
)

print(
    "Columns:",
    len(be3_provider_spatial_features.columns)
)

display(
    be3_provider_spatial_features.head()
)


# ------------------------------------------------------------
# 14. Final feature inventory
# ------------------------------------------------------------

print("\n========== FINAL PROVIDER-LEVEL FEATURE INVENTORY ==========")

for i, feature in enumerate(
    provider_spatial_features,
    start=1
):
    print(f"{i:02d}. {feature}")


# ------------------------------------------------------------
# 15. Final integrity checks
# ------------------------------------------------------------

assert len(be3_spatial) == 281478
assert be3_spatial["NPI"].is_unique

assert len(be3_provider_spatial_features) == 2349
assert be3_provider_spatial_features["NPI"].is_unique

assert (
    be3_provider_spatial_features[
        provider_spatial_features
    ].notna().all().all()
)

print("\n============================================================")
print("STAGE 2C-39 CELL 5: PASSED")
print("============================================================")
print("Full BE-3 rows:", len(be3_spatial))
print("Provider-level spatial rows:", len(be3_provider_spatial_features))
print("Provider-level feature count:", len(provider_spatial_features))
print("Nearest-provider features: PASSED")
print("Provider-density features: PASSED")
print("Same-taxonomy-density features: PASSED")
print("Coverage consistency: PASSED")
print("Spatial policy preservation: PASSED")
print("NPI uniqueness: PASSED")

Expected provider-level spatial features:
 - nearest_provider_npi
 - nearest_provider_distance_miles
 - second_nearest_provider_npi
 - second_nearest_provider_distance_miles
 - provider_count_5mi
 - provider_count_10mi
 - provider_count_25mi
 - provider_count_50mi
 - provider_count_100mi
 - same_taxonomy_count_5mi
 - same_taxonomy_count_10mi
 - same_taxonomy_count_25mi
 - same_taxonomy_count_50mi
 - same_taxonomy_count_100mi

Feature existence check: PASSED
BE-3 row count: 281478
NPI uniqueness: PASSED

========== PROVIDER-LEVEL POPULATION ==========
Provider-metric eligible: 2349
Non-provider-metric rows: 279129

========== NEAREST-PROVIDER FEATURES ==========
nearest_provider_npi: 2349/2349 provider rows populated
nearest_provider_distance_miles: 2349/2349 provider rows populated
second_nearest_provider_npi: 2349/2349 provider rows populated
second_nearest_provider_distance_miles: 2349/2349 provider rows populated

Nearest-distance ordering: PASSED
Non-negative distances: PASSED

===

,feature,provider_rows,provider_non_null,provider_null,non_provider_non_null
0,nearest_provider_npi,2349,2349,0,0
1,nearest_provider_distance_miles,2349,2349,0,0
2,second_nearest_provider_npi,2349,2349,0,0
3,second_nearest_provider_distance_miles,2349,2349,0,0
4,provider_count_5mi,2349,2349,0,0
5,provider_count_10mi,2349,2349,0,0
6,provider_count_25mi,2349,2349,0,0
7,provider_count_50mi,2349,2349,0,0
8,provider_count_100mi,2349,2349,0,0
9,same_taxonomy_count_5mi,2349,2349,0,0


Coverage consistency: PASSED

========== SPATIAL POLICY AUDIT ==========


,count
spatial_metric_policy,
area_level_fallback,274994
unresolved,4135
provider_level,2349


Spatial policy counts: PASSED

========== PROVIDER FEATURE TABLE ==========
Rows: 2349
Columns: 21


,NPI,spatial_latitude,spatial_longitude,spatial_coordinate_type,spatial_coordinate_source,spatial_coordinate_confidence,provider_metric_eligible,nearest_provider_npi,nearest_provider_distance_miles,second_nearest_provider_npi,...,provider_count_5mi,provider_count_10mi,provider_count_25mi,provider_count_50mi,provider_count_100mi,same_taxonomy_count_5mi,same_taxonomy_count_10mi,same_taxonomy_count_25mi,same_taxonomy_count_50mi,same_taxonomy_count_100mi
18,1437197597,37.763598,-122.457630,provider_coordinate,Census_Geocoder,high,True,1114984648,0.0,1013972298,...,202.0,202.0,203.0,205.0,208.0,1.0,1.0,1.0,1.0,1.0
130,1609810977,33.592230,-101.899077,provider_coordinate,Census_Geocoder,high,True,1053397208,0.0,1306822242,...,47.0,48.0,48.0,49.0,49.0,0.0,0.0,0.0,0.0,0.0
308,1063491280,35.112140,-89.893928,provider_coordinate,Census_Geocoder,high,True,1689789893,0.0,1225017445,...,12.0,15.0,15.0,15.0,15.0,4.0,4.0,4.0,4.0,4.0
370,1114984648,37.763598,-122.457630,provider_coordinate,Census_Geocoder,high,True,1437197597,0.0,1013972298,...,202.0,202.0,203.0,205.0,208.0,69.0,69.0,70.0,70.0,70.0
511,1669470001,42.898510,-78.865003,provider_coordinate,Census_Geocoder,high,True,1962485268,0.0,1255396685,...,118.0,118.0,118.0,118.0,149.0,2.0,2.0,2.0,2.0,2.0



========== FINAL PROVIDER-LEVEL FEATURE INVENTORY ==========
01. nearest_provider_npi
02. nearest_provider_distance_miles
03. second_nearest_provider_npi
04. second_nearest_provider_distance_miles
05. provider_count_5mi
06. provider_count_10mi
07. provider_count_25mi
08. provider_count_50mi
09. provider_count_100mi
10. same_taxonomy_count_5mi
11. same_taxonomy_count_10mi
12. same_taxonomy_count_25mi
13. same_taxonomy_count_50mi
14. same_taxonomy_count_100mi

STAGE 2C-39 CELL 5: PASSED
Full BE-3 rows: 281478
Provider-level spatial rows: 2349
Provider-level feature count: 14
Nearest-provider features: PASSED
Provider-density features: PASSED
Same-taxonomy-density features: PASSED
Coverage consistency: PASSED
Spatial policy preservation: PASSED
NPI uniqueness: PASSED


In [ ]:
# ============================================================
# STAGE 2C-40 — CELL 1
# BUILD + VALIDATE AREA-LEVEL / ZCTA SPATIAL LAYER
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Identify area-level fallback population
# ------------------------------------------------------------

area_mask = (
    be3_spatial["area_metric_eligible"]
    &
    be3_spatial["zcta_centroid_fallback"]
)

area_spatial = be3_spatial.loc[
    area_mask
].copy()

print("========== AREA-LEVEL POPULATION ==========")
print("Area-level fallback rows:", len(area_spatial))

assert len(area_spatial) == 274994
assert area_spatial["NPI"].is_unique


# ------------------------------------------------------------
# 2. Required ZCTA fields
# ------------------------------------------------------------

required_zcta_columns = [
    "zcta_clean",
    "spatial_latitude",
    "spatial_longitude",
    "spatial_coordinate_type",
    "spatial_coordinate_source",
    "spatial_coordinate_confidence",
]

missing_zcta_columns = [
    col
    for col in required_zcta_columns
    if col not in area_spatial.columns
]

assert len(missing_zcta_columns) == 0, (
    f"Missing ZCTA spatial columns: {missing_zcta_columns}"
)

print("Required ZCTA fields: PASSED")


# ------------------------------------------------------------
# 3. Confirm every area-level row is using a ZCTA centroid
# ------------------------------------------------------------

assert (
    area_spatial["spatial_coordinate_type"]
    == "zcta_centroid"
).all()

assert (
    area_spatial["spatial_coordinate_source"]
    == "Census_ZCTA_Gazetteer_2020"
).all()

print("ZCTA centroid policy: PASSED")


# ------------------------------------------------------------
# 4. Validate ZCTA identifiers
# ------------------------------------------------------------

area_spatial["zcta_clean"] = (
    area_spatial["zcta_clean"]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)

assert area_spatial["zcta_clean"].notna().all()

assert (
    area_spatial["zcta_clean"]
    .str.fullmatch(r"\d{5}")
).all()

print("ZCTA format: PASSED")


# ------------------------------------------------------------
# 5. Validate centroid coordinates
# ------------------------------------------------------------

assert area_spatial[
    "spatial_latitude"
].notna().all()

assert area_spatial[
    "spatial_longitude"
].notna().all()

assert area_spatial[
    "spatial_latitude"
].between(-90, 90).all()

assert area_spatial[
    "spatial_longitude"
].between(-180, 180).all()

print("Coordinate validity: PASSED")


# ------------------------------------------------------------
# 6. Check ZCTA cardinality
# ------------------------------------------------------------

unique_zctas = (
    area_spatial["zcta_clean"]
    .nunique()
)

print("\n========== ZCTA COVERAGE ==========")
print("Unique ZCTAs represented:", unique_zctas)

assert unique_zctas > 0


# ------------------------------------------------------------
# 7. Confirm all providers in the area layer have a
#    corresponding spatial coordinate
# ------------------------------------------------------------

assert (
    area_spatial[
        [
            "spatial_latitude",
            "spatial_longitude"
        ]
    ]
    .notna()
    .all()
    .all()
)

print("Area coordinate coverage: PASSED")


# ------------------------------------------------------------
# 8. Examine provider concentration by ZCTA
# ------------------------------------------------------------

zcta_provider_counts = (
    area_spatial
    .groupby("zcta_clean")["NPI"]
    .nunique()
    .sort_values(ascending=False)
)

print("\n========== PROVIDERS PER ZCTA ==========")
print(
    zcta_provider_counts.describe()
)

print(
    "\nMaximum providers represented by one ZCTA:",
    int(zcta_provider_counts.max())
)


# ------------------------------------------------------------
# 9. Verify all ZCTA providers sharing a ZCTA use the same
#    centroid coordinate
# ------------------------------------------------------------

centroid_coordinate_counts = (
    area_spatial
    .groupby("zcta_clean")[
        [
            "spatial_latitude",
            "spatial_longitude"
        ]
    ]
    .nunique()
)

assert (
    centroid_coordinate_counts[
        "spatial_latitude"
    ] <= 1
).all()

assert (
    centroid_coordinate_counts[
        "spatial_longitude"
    ] <= 1
).all()

print(
    "One centroid coordinate per ZCTA: PASSED"
)


# ------------------------------------------------------------
# 10. Create a clean ZCTA centroid lookup
# ------------------------------------------------------------

zcta_centroid_layer = (
    area_spatial[
        [
            "zcta_clean",
            "spatial_latitude",
            "spatial_longitude",
            "spatial_coordinate_source",
            "spatial_coordinate_confidence",
        ]
    ]
    .drop_duplicates(
        subset=["zcta_clean"]
    )
    .sort_values("zcta_clean")
    .reset_index(drop=True)
)

assert (
    zcta_centroid_layer["zcta_clean"].is_unique
)

print("\n========== ZCTA CENTROID LAYER ==========")
print(
    "Unique ZCTA centroids:",
    len(zcta_centroid_layer)
)

display(
    zcta_centroid_layer.head()
)


# ------------------------------------------------------------
# 11. Final integrity
# ------------------------------------------------------------

assert len(area_spatial) == 274994
assert area_spatial["NPI"].is_unique

print("\n============================================================")
print("STAGE 2C-40 CELL 1: PASSED")
print("============================================================")
print("Area-level providers:", len(area_spatial))
print("Unique ZCTAs:", len(zcta_centroid_layer))
print("Coordinate type: zcta_centroid")
print("Coordinate source: Census_ZCTA_Gazetteer_2020")
print("NPI uniqueness: PASSED")

========== AREA-LEVEL POPULATION ==========
Area-level fallback rows: 274994
Required ZCTA fields: PASSED
ZCTA centroid policy: PASSED
ZCTA format: PASSED
Coordinate validity: PASSED

========== ZCTA COVERAGE ==========
Unique ZCTAs represented: 16043
Area coordinate coverage: PASSED

========== PROVIDERS PER ZCTA ==========
count    16043.000000
mean        17.141058
std         31.947378
min          1.000000
25%          2.000000
50%          6.000000
75%         19.000000
max       1156.000000
Name: NPI, dtype: float64

Maximum providers represented by one ZCTA: 1156
One centroid coordinate per ZCTA: PASSED

========== ZCTA CENTROID LAYER ==========
Unique ZCTA centroids: 16043


,zcta_clean,spatial_latitude,spatial_longitude,spatial_coordinate_source,spatial_coordinate_confidence
0,00601,18.180555,-66.749961,Census_ZCTA_Gazetteer_2020,area_level
1,00602,18.361945,-67.175597,Census_ZCTA_Gazetteer_2020,area_level
2,00603,18.455183,-67.119887,Census_ZCTA_Gazetteer_2020,area_level
3,00606,18.158327,-66.932928,Census_ZCTA_Gazetteer_2020,area_level
4,00610,18.294032,-67.127156,Census_ZCTA_Gazetteer_2020,area_level



STAGE 2C-40 CELL 1: PASSED
Area-level providers: 274994
Unique ZCTAs: 16043
Coordinate type: zcta_centroid
Coordinate source: Census_ZCTA_Gazetteer_2020
NPI uniqueness: PASSED


In [ ]:
# ============================================================
# STAGE 2C-40 — CELL 2
# BUILD ZCTA CENTROID BALLTREE
# ============================================================

import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

EARTH_RADIUS_MILES = 3958.7613

# ------------------------------------------------------------
# 1. Prepare unique ZCTA centroid coordinates
# ------------------------------------------------------------

zcta_tree = zcta_centroid_layer.copy()

zcta_tree = zcta_tree.sort_values(
    "zcta_clean"
).reset_index(drop=True)

assert zcta_tree["zcta_clean"].is_unique

assert zcta_tree[
    ["spatial_latitude", "spatial_longitude"]
].notna().all().all()

# ------------------------------------------------------------
# 2. Convert coordinates to radians
# ------------------------------------------------------------

zcta_coordinates_rad = np.radians(
    zcta_tree[
        [
            "spatial_latitude",
            "spatial_longitude"
        ]
    ].to_numpy(dtype=float)
)

assert zcta_coordinates_rad.shape == (
    len(zcta_tree),
    2
)

# ------------------------------------------------------------
# 3. Build BallTree
# ------------------------------------------------------------

zcta_balltree = BallTree(
    zcta_coordinates_rad,
    metric="haversine"
)

# ------------------------------------------------------------
# 4. Stable tree index
# ------------------------------------------------------------

zcta_tree["tree_index"] = np.arange(
    len(zcta_tree),
    dtype=np.int32
)

zcta_tree["latitude_rad"] = (
    zcta_coordinates_rad[:, 0]
)

zcta_tree["longitude_rad"] = (
    zcta_coordinates_rad[:, 1]
)

# ------------------------------------------------------------
# 5. Create index → ZCTA mapping
# ------------------------------------------------------------

zcta_tree_index_to_zcta = (
    zcta_tree[
        [
            "tree_index",
            "zcta_clean"
        ]
    ]
    .set_index("tree_index")[
        "zcta_clean"
    ]
    .to_dict()
)

assert len(zcta_tree_index_to_zcta) == len(zcta_tree)

# ------------------------------------------------------------
# 6. Integrity checks
# ------------------------------------------------------------

assert zcta_balltree.data.shape == (
    len(zcta_tree),
    2
)

assert zcta_tree["tree_index"].is_unique

assert (
    zcta_tree["tree_index"].min() == 0
)

assert (
    zcta_tree["tree_index"].max()
    == len(zcta_tree) - 1
)

assert np.isfinite(
    zcta_coordinates_rad
).all()

# ------------------------------------------------------------
# 7. Display summary
# ------------------------------------------------------------

print("========== ZCTA BALLTREE ==========")
print(
    "Unique ZCTAs:",
    len(zcta_tree)
)

print(
    "BallTree shape:",
    zcta_balltree.data.shape
)

print(
    "Metric: Haversine"
)

print(
    "Earth radius:",
    EARTH_RADIUS_MILES,
    "miles"
)

print(
    "Coordinate range — latitude:",
    float(zcta_tree["spatial_latitude"].min()),
    "to",
    float(zcta_tree["spatial_latitude"].max())
)

print(
    "Coordinate range — longitude:",
    float(zcta_tree["spatial_longitude"].min()),
    "to",
    float(zcta_tree["spatial_longitude"].max())
)

print(
    "Tree/index mapping: PASSED"
)

print(
    "Coordinate validity: PASSED"
)

print("\n============================================================")
print("STAGE 2C-40 CELL 2: PASSED")
print("============================================================")

========== ZCTA BALLTREE ==========
Unique ZCTAs: 16043
BallTree shape: (16043, 2)
Metric: Haversine
Earth radius: 3958.7613 miles
Coordinate range — latitude: -14.323296 to 71.253861
Coordinate range — longitude: -170.750333 to 145.754396
Tree/index mapping: PASSED
Coordinate validity: PASSED

STAGE 2C-40 CELL 2: PASSED


In [ ]:
# ============================================================
# STAGE 2C-40 — CELL 3
# AREA-LEVEL PROVIDER DENSITY BY ZCTA
# ============================================================

AREA_PROVIDER_DENSITY_RADII_MILES = [
    5, 10, 25, 50, 100
]

# ------------------------------------------------------------
# 1. Aggregate provider counts by ZCTA
# ------------------------------------------------------------

providers_per_zcta = (
    area_spatial
    .groupby("zcta_clean")["NPI"]
    .nunique()
    .rename("area_provider_count")
)

print("========== AREA PROVIDERS BY ZCTA ==========")
print("ZCTAs with providers:", len(providers_per_zcta))
print("Total area-level providers:", int(providers_per_zcta.sum()))

assert providers_per_zcta.sum() == 274994


# ------------------------------------------------------------
# 2. Attach provider counts to the ZCTA tree
# ------------------------------------------------------------

zcta_tree["area_provider_count"] = (
    zcta_tree["zcta_clean"]
    .map(providers_per_zcta)
    .fillna(0)
    .astype(np.int32)
)

assert (
    zcta_tree["area_provider_count"] >= 0
).all()

assert (
    zcta_tree["area_provider_count"].sum()
    == 274994
)

print("Provider-count attachment: PASSED")


# ------------------------------------------------------------
# 3. Query ZCTA neighbors at the largest radius
# ------------------------------------------------------------

MAX_RADIUS_MILES = max(
    AREA_PROVIDER_DENSITY_RADII_MILES
)

MAX_RADIUS_RAD = (
    MAX_RADIUS_MILES / EARTH_RADIUS_MILES
)

zcta_neighbor_indices = (
    zcta_balltree.query_radius(
        zcta_coordinates_rad,
        r=MAX_RADIUS_RAD
    )
)

assert len(zcta_neighbor_indices) == len(zcta_tree)

print("\n========== ZCTA NEIGHBOR QUERY ==========")
print("Maximum radius:", MAX_RADIUS_MILES, "miles")
print("ZCTAs queried:", len(zcta_neighbor_indices))


# ------------------------------------------------------------
# 4. Calculate area-level provider density
#
# Each ZCTA receives the total number of providers located
# in ZCTAs whose centroids fall within the requested radius.
#
# The focal ZCTA is intentionally included because these are
# area-level provider availability counts.
# ------------------------------------------------------------

for radius_miles in AREA_PROVIDER_DENSITY_RADII_MILES:

    radius_rad = (
        radius_miles / EARTH_RADIUS_MILES
    )

    feature_name = (
        f"area_provider_count_{radius_miles}mi"
    )

    counts = np.zeros(
        len(zcta_tree),
        dtype=np.int32
    )

    for zcta_idx, neighbor_indices in enumerate(
        zcta_neighbor_indices
    ):

        neighbor_coordinates = (
            zcta_coordinates_rad[
                neighbor_indices
            ]
        )

        lat1 = zcta_coordinates_rad[
            zcta_idx, 0
        ]

        lon1 = zcta_coordinates_rad[
            zcta_idx, 1
        ]

        lat2 = neighbor_coordinates[:, 0]
        lon2 = neighbor_coordinates[:, 1]

        dlat = lat2 - lat1
        dlon = lon2 - lon1

        a = (
            np.sin(dlat / 2.0) ** 2
            +
            np.cos(lat1)
            * np.cos(lat2)
            * np.sin(dlon / 2.0) ** 2
        )

        angular_distances = (
            2.0
            * np.arcsin(
                np.sqrt(
                    np.clip(a, 0, 1)
                )
            )
        )

        within_radius = (
            angular_distances
            <= radius_rad + 1e-12
        )

        valid_neighbor_indices = (
            neighbor_indices[
                within_radius
            ]
        )

        counts[zcta_idx] = int(
            zcta_tree.loc[
                valid_neighbor_indices,
                "area_provider_count"
            ].sum()
        )

    zcta_tree[feature_name] = counts


# ------------------------------------------------------------
# 5. Validate resulting ZCTA features
# ------------------------------------------------------------

area_provider_density_features = [
    f"area_provider_count_{r}mi"
    for r in AREA_PROVIDER_DENSITY_RADII_MILES
]

print("\n========== AREA-LEVEL DENSITY SUMMARY ==========")

display(
    zcta_tree[
        area_provider_density_features
    ].describe()
)


# ------------------------------------------------------------
# 6. Non-negative validation
# ------------------------------------------------------------

for feature in area_provider_density_features:

    assert (
        zcta_tree[feature] >= 0
    ).all()


# ------------------------------------------------------------
# 7. Radius monotonicity
# ------------------------------------------------------------

for smaller, larger in zip(
    [5, 10, 25, 50],
    [10, 25, 50, 100]
):

    smaller_feature = (
        f"area_provider_count_{smaller}mi"
    )

    larger_feature = (
        f"area_provider_count_{larger}mi"
    )

    assert (
        zcta_tree[smaller_feature]
        <=
        zcta_tree[larger_feature]
    ).all()

print("\nRadius monotonicity: PASSED")


# ------------------------------------------------------------
# 8. Every represented ZCTA should have all features
# ------------------------------------------------------------

for feature in area_provider_density_features:

    assert (
        zcta_tree[feature].notna().all()
    )

print("ZCTA feature coverage: PASSED")


# ------------------------------------------------------------
# 9. Attach ZCTA-level features back to area providers
# ------------------------------------------------------------

area_feature_lookup = zcta_tree[
    [
        "zcta_clean"
    ] + area_provider_density_features
].copy()

area_spatial = area_spatial.drop(
    columns=area_provider_density_features,
    errors="ignore"
)

area_spatial = area_spatial.merge(
    area_feature_lookup,
    on="zcta_clean",
    how="left",
    validate="many_to_one"
)

assert len(area_spatial) == 274994
assert area_spatial["NPI"].is_unique

# All area providers should have the features.
for feature in area_provider_density_features:

    assert (
        area_spatial[feature].notna().all()
    )

print("Provider-level attachment: PASSED")


# ------------------------------------------------------------
# 10. Attach features to the full BE-3 layer
# ------------------------------------------------------------

be3_spatial = be3_spatial.drop(
    columns=area_provider_density_features,
    errors="ignore"
)

be3_spatial = be3_spatial.merge(
    area_spatial[
        [
            "NPI"
        ] + area_provider_density_features
    ],
    on="NPI",
    how="left",
    validate="one_to_one"
)

assert len(be3_spatial) == 281478
assert be3_spatial["NPI"].is_unique


# ------------------------------------------------------------
# 11. Confirm only area-level fallback rows receive
#     these features
# ------------------------------------------------------------

area_feature_mask = (
    be3_spatial["spatial_metric_policy"]
    == "area_level_fallback"
)

unresolved_mask = (
    be3_spatial["spatial_metric_policy"]
    == "unresolved"
)

provider_mask = (
    be3_spatial["spatial_metric_policy"]
    == "provider_level"
)

for feature in area_provider_density_features:

    assert (
        be3_spatial.loc[
            area_feature_mask,
            feature
        ].notna()
    ).all()

    assert (
        be3_spatial.loc[
            unresolved_mask,
            feature
        ].isna()
    ).all()

    assert (
        be3_spatial.loc[
            provider_mask,
            feature
        ].isna()
    ).all()


# ------------------------------------------------------------
# 12. Final integrity
# ------------------------------------------------------------

print("\n========== FULL BE-3 AREA FEATURE COVERAGE ==========")

for feature in area_provider_density_features:

    available = int(
        be3_spatial[feature].notna().sum()
    )

    print(
        f"{feature}: {available}"
    )

    assert available == 274994

print("\n============================================================")
print("STAGE 2C-40 CELL 3: PASSED")
print("============================================================")
print("Unique ZCTAs:", len(zcta_tree))
print("Area-level providers:", len(area_spatial))
print("Full BE-3 rows:", len(be3_spatial))
print("Area-density features:", len(area_provider_density_features))
print("Provider-level rows excluded: PASSED")
print("Unresolved rows excluded: PASSED")
print("Radius monotonicity: PASSED")
print("NPI uniqueness: PASSED")

========== AREA PROVIDERS BY ZCTA ==========
ZCTAs with providers: 16043
Total area-level providers: 274994
Provider-count attachment: PASSED

========== ZCTA NEIGHBOR QUERY ==========
Maximum radius: 100 miles
ZCTAs queried: 16043

========== AREA-LEVEL DENSITY SUMMARY ==========


,area_provider_count_5mi,area_provider_count_10mi,area_provider_count_25mi,area_provider_count_50mi,area_provider_count_100mi
count,16043.000000,16043.000000,16043.000000,16043.000000,16043.000000
mean,128.769744,376.632799,1287.506763,2828.339525,6802.813252
std,275.303035,674.594118,1902.712504,3388.912840,6657.370537
min,1.000000,1.000000,1.000000,1.000000,1.000000
25%,3.000000,11.000000,119.000000,562.500000,2321.500000
50%,21.000000,101.000000,524.000000,1532.000000,4841.000000
75%,142.000000,476.500000,1697.500000,3809.500000,8387.000000
max,3028.000000,5543.000000,11176.000000,16593.000000,28008.000000



Radius monotonicity: PASSED
ZCTA feature coverage: PASSED
Provider-level attachment: PASSED

========== FULL BE-3 AREA FEATURE COVERAGE ==========
area_provider_count_5mi: 274994
area_provider_count_10mi: 274994
area_provider_count_25mi: 274994
area_provider_count_50mi: 274994
area_provider_count_100mi: 274994

STAGE 2C-40 CELL 3: PASSED
Unique ZCTAs: 16043
Area-level providers: 274994
Full BE-3 rows: 281478
Area-density features: 5
Provider-level rows excluded: PASSED
Unresolved rows excluded: PASSED
Radius monotonicity: PASSED
NPI uniqueness: PASSED


In [ ]:
# ============================================================
# STAGE 2C-40 — CELL 4
# AREA-LEVEL SAME-TAXONOMY DENSITY
# OPTIMIZED IMPLEMENTATION
# ============================================================

from scipy.sparse import csr_matrix


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

AREA_SAME_TAXONOMY_RADII_MILES = [
    5, 10, 25, 50, 100
]

TAXONOMY_COL = "Healthcare Provider Taxonomy Code_1"

AREA_SAME_TAXONOMY_FEATURES = [
    f"area_same_taxonomy_count_{r}mi"
    for r in AREA_SAME_TAXONOMY_RADII_MILES
]


# ------------------------------------------------------------
# 2. Validate area-level taxonomy data
# ------------------------------------------------------------

area_spatial[TAXONOMY_COL] = (
    area_spatial[TAXONOMY_COL]
    .astype("string")
    .str.strip()
)

assert area_spatial[TAXONOMY_COL].notna().all()

print("========== AREA TAXONOMY COVERAGE ==========")
print("Area-level providers:", len(area_spatial))
print(
    "Providers with taxonomy:",
    area_spatial[TAXONOMY_COL].notna().sum()
)
print(
    "Unique taxonomies:",
    area_spatial[TAXONOMY_COL].nunique()
)


# ------------------------------------------------------------
# 3. Aggregate providers by ZCTA + taxonomy
# ------------------------------------------------------------

zcta_taxonomy_counts = (
    area_spatial
    .groupby(
        ["zcta_clean", TAXONOMY_COL],
        as_index=False
    )["NPI"]
    .nunique()
    .rename(
        columns={
            "NPI": "taxonomy_provider_count"
        }
    )
)

print("\n========== ZCTA + TAXONOMY AGGREGATION ==========")
print(
    "ZCTA-taxonomy groups:",
    len(zcta_taxonomy_counts)
)
print(
    "Unique ZCTAs:",
    zcta_taxonomy_counts["zcta_clean"].nunique()
)
print(
    "Unique taxonomies:",
    zcta_taxonomy_counts[TAXONOMY_COL].nunique()
)

assert (
    zcta_taxonomy_counts["zcta_clean"].nunique()
    == len(zcta_tree)
)


# ------------------------------------------------------------
# 4. Create stable integer IDs
#
# This lets us perform the calculation using sparse matrices
# rather than Python nested loops.
# ------------------------------------------------------------

zcta_to_id = pd.Series(
    np.arange(len(zcta_tree)),
    index=zcta_tree["zcta_clean"]
)

taxonomy_values = (
    zcta_taxonomy_counts[TAXONOMY_COL]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

taxonomy_to_id = {
    taxonomy: idx
    for idx, taxonomy
    in enumerate(taxonomy_values)
}

zcta_taxonomy_counts["zcta_id"] = (
    zcta_taxonomy_counts["zcta_clean"]
    .map(zcta_to_id)
    .astype(np.int32)
)

zcta_taxonomy_counts["taxonomy_id"] = (
    zcta_taxonomy_counts[TAXONOMY_COL]
    .map(taxonomy_to_id)
    .astype(np.int32)
)

assert zcta_taxonomy_counts["zcta_id"].notna().all()
assert zcta_taxonomy_counts["taxonomy_id"].notna().all()

N_ZCTA = len(zcta_tree)
N_TAXONOMIES = len(taxonomy_values)

print("\n========== INTEGER MAPPINGS ==========")
print("ZCTAs:", N_ZCTA)
print("Taxonomies:", N_TAXONOMIES)
print("ZCTA-taxonomy groups:", len(zcta_taxonomy_counts))


# ------------------------------------------------------------
# 5. Build ZCTA × taxonomy provider-count matrix
#
# Rows    = ZCTAs
# Columns = taxonomies
# Values  = provider counts
# ------------------------------------------------------------

zcta_taxonomy_matrix = csr_matrix(
    (
        zcta_taxonomy_counts[
            "taxonomy_provider_count"
        ].astype(np.int32),
        (
            zcta_taxonomy_counts["zcta_id"],
            zcta_taxonomy_counts["taxonomy_id"]
        )
    ),
    shape=(
        N_ZCTA,
        N_TAXONOMIES
    ),
    dtype=np.int32
)

print("\n========== ZCTA × TAXONOMY MATRIX ==========")
print(
    "Matrix shape:",
    zcta_taxonomy_matrix.shape
)
print(
    "Non-zero entries:",
    zcta_taxonomy_matrix.nnz
)

assert (
    zcta_taxonomy_matrix.nnz
    == len(zcta_taxonomy_counts)
)


# ------------------------------------------------------------
# 6. Build neighbor matrices for each radius
#
# Neighbor matrix:
#
#     row = focal ZCTA
#     col = neighboring ZCTA
#
# value = 1 when within radius
#
# Focal ZCTA is intentionally included.
# ------------------------------------------------------------

zcta_neighbor_matrices = {}

for radius_miles in AREA_SAME_TAXONOMY_RADII_MILES:

    radius_rad = (
        radius_miles
        / EARTH_RADIUS_MILES
    )

    neighbor_indices = (
        zcta_balltree.query_radius(
            zcta_coordinates_rad,
            r=radius_rad
        )
    )

    row_indices = []
    col_indices = []

    for focal_idx, neighbors in enumerate(
        neighbor_indices
    ):
        row_indices.extend(
            [focal_idx] * len(neighbors)
        )
        col_indices.extend(
            neighbors.tolist()
        )

    data = np.ones(
        len(row_indices),
        dtype=np.int8
    )

    neighbor_matrix = csr_matrix(
        (
            data,
            (
                row_indices,
                col_indices
            )
        ),
        shape=(
            N_ZCTA,
            N_ZCTA
        ),
        dtype=np.int8
    )

    zcta_neighbor_matrices[
        radius_miles
    ] = neighbor_matrix

    print(
        f"{radius_miles}mi neighbor matrix:",
        neighbor_matrix.shape,
        "| edges:",
        neighbor_matrix.nnz
    )


# ------------------------------------------------------------
# 7. Sparse matrix multiplication
#
# Neighbor matrix:
#       ZCTA × ZCTA
#
# Taxonomy matrix:
#       ZCTA × TAXONOMY
#
# Product:
#       ZCTA × TAXONOMY
#
# Result = number of providers of each taxonomy within
# the requested radius around each focal ZCTA.
# ------------------------------------------------------------

zcta_taxonomy_density_matrices = {}

for radius_miles in AREA_SAME_TAXONOMY_RADII_MILES:

    print(
        f"\nCalculating {radius_miles}mi "
        "same-taxonomy density..."
    )

    density_matrix = (
        zcta_neighbor_matrices[
            radius_miles
        ].astype(np.int32)
        @
        zcta_taxonomy_matrix
    )

    zcta_taxonomy_density_matrices[
        radius_miles
    ] = density_matrix

    print(
        "Result shape:",
        density_matrix.shape,
        "| non-zero entries:",
        density_matrix.nnz
    )


# ------------------------------------------------------------
# 8. Map density values back to each provider
#
# Every provider gets the value corresponding to:
#
#     focal provider's ZCTA
#     focal provider's taxonomy
# ------------------------------------------------------------

provider_zcta_ids = (
    area_spatial["zcta_clean"]
    .map(zcta_to_id)
    .astype(np.int32)
    .to_numpy()
)

provider_taxonomy_ids = (
    area_spatial[TAXONOMY_COL]
    .map(taxonomy_to_id)
    .astype(np.int32)
    .to_numpy()
)

assert np.all(
    provider_zcta_ids >= 0
)

assert np.all(
    provider_taxonomy_ids >= 0
)


for radius_miles in AREA_SAME_TAXONOMY_RADII_MILES:

    feature_name = (
        f"area_same_taxonomy_count_{radius_miles}mi"
    )

    density_matrix = (
        zcta_taxonomy_density_matrices[
            radius_miles
        ]
    )

    # Sparse matrix lookup
    values = np.asarray(
        density_matrix[
            provider_zcta_ids,
            provider_taxonomy_ids
        ]
    ).ravel()

    area_spatial[
        feature_name
    ] = values.astype(np.int32)

    print(
        f"{feature_name}: attached"
    )


# ------------------------------------------------------------
# 9. Summary
# ------------------------------------------------------------

print(
    "\n========== AREA SAME-TAXONOMY DENSITY SUMMARY =========="
)

display(
    area_spatial[
        AREA_SAME_TAXONOMY_FEATURES
    ].describe()
)


# ------------------------------------------------------------
# 10. Basic validation
# ------------------------------------------------------------

for feature in AREA_SAME_TAXONOMY_FEATURES:

    assert (
        area_spatial[feature] >= 0
    ).all()

    assert (
        area_spatial[feature].notna().all()
    )

print(
    "Non-negative counts: PASSED"
)


# ------------------------------------------------------------
# 11. Radius monotonicity
# ------------------------------------------------------------

for smaller, larger in zip(
    [5, 10, 25, 50],
    [10, 25, 50, 100]
):

    smaller_feature = (
        f"area_same_taxonomy_count_{smaller}mi"
    )

    larger_feature = (
        f"area_same_taxonomy_count_{larger}mi"
    )

    assert (
        area_spatial[smaller_feature]
        <= area_spatial[larger_feature]
    ).all()

print(
    "Radius monotonicity: PASSED"
)


# ------------------------------------------------------------
# 12. Attach to full BE-3
# ------------------------------------------------------------

be3_spatial = be3_spatial.drop(
    columns=AREA_SAME_TAXONOMY_FEATURES,
    errors="ignore"
)

be3_spatial = be3_spatial.merge(
    area_spatial[
        ["NPI"] + AREA_SAME_TAXONOMY_FEATURES
    ],
    on="NPI",
    how="left",
    validate="one_to_one"
)

assert len(be3_spatial) == 281478
assert be3_spatial["NPI"].is_unique


# ------------------------------------------------------------
# 13. Spatial policy validation
# ------------------------------------------------------------

area_level_mask = (
    be3_spatial["spatial_metric_policy"]
    == "area_level_fallback"
)

provider_level_mask = (
    be3_spatial["spatial_metric_policy"]
    == "provider_level"
)

unresolved_mask = (
    be3_spatial["spatial_metric_policy"]
    == "unresolved"
)

for feature in AREA_SAME_TAXONOMY_FEATURES:

    assert (
        be3_spatial.loc[
            area_level_mask,
            feature
        ].notna()
    ).all()

    assert (
        be3_spatial.loc[
            provider_level_mask,
            feature
        ].isna()
    ).all()

    assert (
        be3_spatial.loc[
            unresolved_mask,
            feature
        ].isna()
    ).all()

print(
    "Spatial policy exclusion: PASSED"
)


# ------------------------------------------------------------
# 14. Coverage
# ------------------------------------------------------------

print(
    "\n========== FULL BE-3 AREA SAME-TAXONOMY COVERAGE =========="
)

for feature in AREA_SAME_TAXONOMY_FEATURES:

    available = int(
        be3_spatial[feature]
        .notna()
        .sum()
    )

    print(
        f"{feature}: {available}"
    )

    assert available == 274994


# ------------------------------------------------------------
# 15. Final integrity
# ------------------------------------------------------------

assert len(area_spatial) == 274994
assert area_spatial["NPI"].is_unique

assert len(be3_spatial) == 281478
assert be3_spatial["NPI"].is_unique

print("\n============================================================")
print("STAGE 2C-40 CELL 4: PASSED")
print("============================================================")
print("Area-level providers:", len(area_spatial))
print("Unique ZCTAs:", N_ZCTA)
print("Unique taxonomies:", N_TAXONOMIES)
print(
    "ZCTA-taxonomy groups:",
    len(zcta_taxonomy_counts)
)
print(
    "Area same-taxonomy features:",
    len(AREA_SAME_TAXONOMY_FEATURES)
)
print("Full BE-3 rows:", len(be3_spatial))
print("Area-level coverage: PASSED")
print("Provider-level rows excluded: PASSED")
print("Unresolved rows excluded: PASSED")
print("Radius monotonicity: PASSED")
print("NPI uniqueness: PASSED")

========== AREA TAXONOMY COVERAGE ==========
Area-level providers: 274994
Providers with taxonomy: 274994
Unique taxonomies: 720

========== ZCTA + TAXONOMY AGGREGATION ==========
ZCTA-taxonomy groups: 160068
Unique ZCTAs: 16043
Unique taxonomies: 720

========== INTEGER MAPPINGS ==========
ZCTAs: 16043
Taxonomies: 720
ZCTA-taxonomy groups: 160068

========== ZCTA × TAXONOMY MATRIX ==========
Matrix shape: (16043, 720)
Non-zero entries: 160068
5mi neighbor matrix: (16043, 16043) | edges: 77583
10mi neighbor matrix: (16043, 16043) | edges: 236575
25mi neighbor matrix: (16043, 16043) | edges: 921287
50mi neighbor matrix: (16043, 16043) | edges: 2252601
100mi neighbor matrix: (16043, 16043) | edges: 5844427

Calculating 5mi same-taxonomy density...
Result shape: (16043, 720) | non-zero entries: 574954

Calculating 10mi same-taxonomy density...
Result shape: (16043, 720) | non-zero entries: 1093653

Calculating 25mi same-taxonomy density...
Result shape: (16043, 720) | non-zero entries: 21

,area_same_taxonomy_count_5mi,area_same_taxonomy_count_10mi,area_same_taxonomy_count_25mi,area_same_taxonomy_count_50mi,area_same_taxonomy_count_100mi
count,274994.000000,274994.000000,274994.000000,274994.000000,274994.000000
mean,9.278079,18.510986,46.057609,85.132992,178.112192
std,21.743202,40.468303,94.166127,155.898356,288.782976
min,1.000000,1.000000,1.000000,1.000000,1.000000
25%,1.000000,2.000000,4.000000,9.000000,22.000000
50%,3.000000,6.000000,15.000000,31.000000,77.000000
75%,9.000000,18.000000,47.000000,92.000000,203.000000
max,382.000000,551.000000,1058.000000,1428.000000,2474.000000


Non-negative counts: PASSED
Radius monotonicity: PASSED
Spatial policy exclusion: PASSED

========== FULL BE-3 AREA SAME-TAXONOMY COVERAGE ==========
area_same_taxonomy_count_5mi: 274994
area_same_taxonomy_count_10mi: 274994
area_same_taxonomy_count_25mi: 274994
area_same_taxonomy_count_50mi: 274994
area_same_taxonomy_count_100mi: 274994

STAGE 2C-40 CELL 4: PASSED
Area-level providers: 274994
Unique ZCTAs: 16043
Unique taxonomies: 720
ZCTA-taxonomy groups: 160068
Area same-taxonomy features: 5
Full BE-3 rows: 281478
Area-level coverage: PASSED
Provider-level rows excluded: PASSED
Unresolved rows excluded: PASSED
Radius monotonicity: PASSED
NPI uniqueness: PASSED


In [ ]:
# ============================================================
# STAGE 2C-40 — CELL 5
# FINAL SPATIAL FEATURE CONSOLIDATION & AUDIT
# ============================================================

# ------------------------------------------------------------
# 1. Define expected feature groups
# ------------------------------------------------------------

PROVIDER_SPATIAL_FEATURES = [
    "nearest_provider_npi",
    "nearest_provider_distance_miles",
    "second_nearest_provider_npi",
    "second_nearest_provider_distance_miles",

    "provider_count_5mi",
    "provider_count_10mi",
    "provider_count_25mi",
    "provider_count_50mi",
    "provider_count_100mi",

    "same_taxonomy_count_5mi",
    "same_taxonomy_count_10mi",
    "same_taxonomy_count_25mi",
    "same_taxonomy_count_50mi",
    "same_taxonomy_count_100mi",
]

AREA_SPATIAL_FEATURES = [
    "area_provider_count_5mi",
    "area_provider_count_10mi",
    "area_provider_count_25mi",
    "area_provider_count_50mi",
    "area_provider_count_100mi",

    "area_same_taxonomy_count_5mi",
    "area_same_taxonomy_count_10mi",
    "area_same_taxonomy_count_25mi",
    "area_same_taxonomy_count_50mi",
    "area_same_taxonomy_count_100mi",
]

ALL_SPATIAL_FEATURES = (
    PROVIDER_SPATIAL_FEATURES
    + AREA_SPATIAL_FEATURES
)

print("========== SPATIAL FEATURE INVENTORY ==========")
print(
    "Provider-level features:",
    len(PROVIDER_SPATIAL_FEATURES)
)
print(
    "Area-level features:",
    len(AREA_SPATIAL_FEATURES)
)
print(
    "Total spatial features:",
    len(ALL_SPATIAL_FEATURES)
)


# ------------------------------------------------------------
# 2. Verify all expected features exist
# ------------------------------------------------------------

missing_features = [
    feature
    for feature in ALL_SPATIAL_FEATURES
    if feature not in be3_spatial.columns
]

assert len(missing_features) == 0, (
    f"Missing spatial features: {missing_features}"
)

print(
    "Feature existence: PASSED"
)


# ------------------------------------------------------------
# 3. Core BE-3 integrity
# ------------------------------------------------------------

assert len(be3_spatial) == 281478
assert be3_spatial["NPI"].is_unique

print("\n========== CORE BE-3 INTEGRITY ==========")
print(
    "BE-3 rows:",
    len(be3_spatial)
)
print(
    "Unique NPI:",
    be3_spatial["NPI"].nunique()
)
print(
    "Row count: PASSED"
)
print(
    "NPI uniqueness: PASSED"
)


# ------------------------------------------------------------
# 4. Spatial policy counts
# ------------------------------------------------------------

policy_counts = (
    be3_spatial[
        "spatial_metric_policy"
    ]
    .value_counts(dropna=False)
)

print("\n========== SPATIAL METRIC POLICY ==========")
display(
    policy_counts.to_frame(
        "provider_count"
    )
)

assert (
    policy_counts.get(
        "provider_level",
        0
    ) == 2349
)

assert (
    policy_counts.get(
        "area_level_fallback",
        0
    ) == 274994
)

assert (
    policy_counts.get(
        "unresolved",
        0
    ) == 4135
)

assert (
    policy_counts.sum()
    == 281478
)

print(
    "Spatial policy counts: PASSED"
)


# ------------------------------------------------------------
# 5. Provider-level feature coverage
# ------------------------------------------------------------

provider_mask = (
    be3_spatial[
        "provider_metric_eligible"
    ]
)

non_provider_mask = (
    ~provider_mask
)

print(
    "\n========== PROVIDER-LEVEL FEATURE COVERAGE =========="
)

for feature in PROVIDER_SPATIAL_FEATURES:

    provider_available = int(
        be3_spatial.loc[
            provider_mask,
            feature
        ].notna().sum()
    )

    non_provider_available = int(
        be3_spatial.loc[
            non_provider_mask,
            feature
        ].notna().sum()
    )

    print(
        f"{feature}: "
        f"provider={provider_available} | "
        f"non_provider={non_provider_available}"
    )

    assert (
        provider_available == 2349
    )

    assert (
        non_provider_available == 0
    )

print(
    "Provider-level coverage/exclusion: PASSED"
)


# ------------------------------------------------------------
# 6. Area-level feature coverage
# ------------------------------------------------------------

area_mask = (
    be3_spatial[
        "spatial_metric_policy"
    ]
    == "area_level_fallback"
)

not_area_mask = (
    ~area_mask
)

print(
    "\n========== AREA-LEVEL FEATURE COVERAGE =========="
)

for feature in AREA_SPATIAL_FEATURES:

    area_available = int(
        be3_spatial.loc[
            area_mask,
            feature
        ].notna().sum()
    )

    non_area_available = int(
        be3_spatial.loc[
            not_area_mask,
            feature
        ].notna().sum()
    )

    print(
        f"{feature}: "
        f"area={area_available} | "
        f"non_area={non_area_available}"
    )

    assert (
        area_available == 274994
    )

    assert (
        non_area_available == 0
    )

print(
    "Area-level coverage/exclusion: PASSED"
)


# ------------------------------------------------------------
# 7. Unresolved rows must have no spatial metric features
# ------------------------------------------------------------

unresolved_mask = (
    be3_spatial[
        "spatial_metric_policy"
    ]
    == "unresolved"
)

unresolved_feature_values = (
    be3_spatial.loc[
        unresolved_mask,
        ALL_SPATIAL_FEATURES
    ]
)

assert (
    unresolved_feature_values
    .notna()
    .sum()
    .sum()
    == 0
)

print(
    "Unresolved rows excluded from all spatial metrics: PASSED"
)


# ------------------------------------------------------------
# 8. Provider-level and area-level metrics must not overlap
# ------------------------------------------------------------

provider_overlap = (
    be3_spatial.loc[
        provider_mask,
        AREA_SPATIAL_FEATURES
    ]
    .notna()
    .sum()
    .sum()
)

area_overlap = (
    be3_spatial.loc[
        area_mask,
        PROVIDER_SPATIAL_FEATURES
    ]
    .notna()
    .sum()
    .sum()
)

assert provider_overlap == 0
assert area_overlap == 0

print(
    "Provider/area metric separation: PASSED"
)


# ------------------------------------------------------------
# 9. Validate density monotonicity
# ------------------------------------------------------------

provider_density_features = [
    "provider_count_5mi",
    "provider_count_10mi",
    "provider_count_25mi",
    "provider_count_50mi",
    "provider_count_100mi",
]

provider_taxonomy_features = [
    "same_taxonomy_count_5mi",
    "same_taxonomy_count_10mi",
    "same_taxonomy_count_25mi",
    "same_taxonomy_count_50mi",
    "same_taxonomy_count_100mi",
]

area_density_features = [
    "area_provider_count_5mi",
    "area_provider_count_10mi",
    "area_provider_count_25mi",
    "area_provider_count_50mi",
    "area_provider_count_100mi",
]

area_taxonomy_features = [
    "area_same_taxonomy_count_5mi",
    "area_same_taxonomy_count_10mi",
    "area_same_taxonomy_count_25mi",
    "area_same_taxonomy_count_50mi",
    "area_same_taxonomy_count_100mi",
]


def assert_monotonic(features, mask):

    for smaller, larger in zip(
        features[:-1],
        features[1:]
    ):

        comparison = (
            be3_spatial.loc[
                mask,
                smaller
            ]
            <=
            be3_spatial.loc[
                mask,
                larger
            ]
        )

        assert comparison.all()


assert_monotonic(
    provider_density_features,
    provider_mask
)

assert_monotonic(
    provider_taxonomy_features,
    provider_mask
)

assert_monotonic(
    area_density_features,
    area_mask
)

assert_monotonic(
    area_taxonomy_features,
    area_mask
)

print(
    "Radius monotonicity across all density features: PASSED"
)


# ------------------------------------------------------------
# 10. Non-negative count validation
# ------------------------------------------------------------

count_features = (
    provider_density_features
    + provider_taxonomy_features
    + area_density_features
    + area_taxonomy_features
)

for feature in count_features:

    assert (
        be3_spatial[feature]
        .dropna()
        >= 0
    ).all()

print(
    "Non-negative spatial counts: PASSED"
)


# ------------------------------------------------------------
# 11. Nearest-distance validation
# ------------------------------------------------------------

distance_features = [
    "nearest_provider_distance_miles",
    "second_nearest_provider_distance_miles",
]

for feature in distance_features:

    assert (
        be3_spatial.loc[
            provider_mask,
            feature
        ].notna().all()
    )

    assert (
        be3_spatial.loc[
            provider_mask,
            feature
        ] >= 0
    ).all()

    assert (
        be3_spatial.loc[
            non_provider_mask,
            feature
        ].isna().all()
    )

print(
    "Nearest-distance validation: PASSED"
)


# ------------------------------------------------------------
# 12. Nearest vs second-nearest ordering
# ------------------------------------------------------------

assert (
    be3_spatial.loc[
        provider_mask,
        "nearest_provider_distance_miles"
    ]
    <=
    be3_spatial.loc[
        provider_mask,
        "second_nearest_provider_distance_miles"
    ]
).all()

print(
    "Nearest/second-nearest ordering: PASSED"
)


# ------------------------------------------------------------
# 13. Coordinate-policy consistency
# ------------------------------------------------------------

assert (
    (
        be3_spatial[
            "spatial_coordinate_type"
        ]
        == "provider_coordinate"
    )
    ==
    provider_mask
).all()

assert (
    (
        be3_spatial[
            "spatial_coordinate_type"
        ]
    )
    .eq("zcta_centroid")
    ==
    area_mask
).all()

assert (
    (
        be3_spatial[
            "spatial_coordinate_type"
        ]
    )
    .eq("unresolved")
    ==
    unresolved_mask
).all()

print(
    "Coordinate/policy consistency: PASSED"
)


# ------------------------------------------------------------
# 14. Final feature coverage table
# ------------------------------------------------------------

coverage_rows = []

for feature in ALL_SPATIAL_FEATURES:

    coverage_rows.append(
        {
            "feature": feature,
            "non_null_count": int(
                be3_spatial[feature]
                .notna()
                .sum()
            ),
            "coverage_pct": round(
                100
                * be3_spatial[feature]
                .notna()
                .mean(),
                4
            )
        }
    )

spatial_feature_coverage = pd.DataFrame(
    coverage_rows
)

print(
    "\n========== FINAL SPATIAL FEATURE COVERAGE =========="
)

display(
    spatial_feature_coverage
)


# ------------------------------------------------------------
# 15. Final spatial feature layer
# ------------------------------------------------------------

be3_spatial_features = be3_spatial[
    [
        "NPI",
        "spatial_latitude",
        "spatial_longitude",
        "spatial_coordinate_type",
        "spatial_coordinate_source",
        "spatial_coordinate_confidence",
        "spatial_metric_policy",
    ]
    + ALL_SPATIAL_FEATURES
].copy()

assert (
    len(be3_spatial_features)
    == 281478
)

assert (
    be3_spatial_features["NPI"]
    .is_unique
)


# ------------------------------------------------------------
# 16. Final dimensions
# ------------------------------------------------------------

print(
    "\n========== FINAL BE-3 SPATIAL FEATURE LAYER =========="
)

print(
    "Rows:",
    len(be3_spatial_features)
)

print(
    "Columns:",
    len(be3_spatial_features.columns)
)

print(
    "Spatial features:",
    len(ALL_SPATIAL_FEATURES)
)

print(
    "Provider-level spatial features:",
    len(PROVIDER_SPATIAL_FEATURES)
)

print(
    "Area-level spatial features:",
    len(AREA_SPATIAL_FEATURES)
)


# ------------------------------------------------------------
# 17. Final assertions
# ------------------------------------------------------------

assert (
    len(ALL_SPATIAL_FEATURES)
    == 24
)

assert (
    len(be3_spatial_features)
    == 281478
)

assert (
    be3_spatial_features["NPI"]
    .nunique()
    == 281478
)

assert (
    be3_spatial_features["NPI"]
    .notna()
    .all()
)


print("\n============================================================")
print("STAGE 2C-40 CELL 5: PASSED")
print("============================================================")
print("Full BE-3 rows:", len(be3_spatial_features))
print("Spatial feature columns:", len(ALL_SPATIAL_FEATURES))
print("Provider-level features:", len(PROVIDER_SPATIAL_FEATURES))
print("Area-level features:", len(AREA_SPATIAL_FEATURES))
print("Provider-level rows:", int(provider_mask.sum()))
print("Area-level fallback rows:", int(area_mask.sum()))
print("Unresolved rows:", int(unresolved_mask.sum()))
print("Feature separation: PASSED")
print("Coverage validation: PASSED")
print("Radius monotonicity: PASSED")
print("Spatial policy validation: PASSED")
print("NPI uniqueness: PASSED")
print("============================================================")

========== SPATIAL FEATURE INVENTORY ==========
Provider-level features: 14
Area-level features: 10
Total spatial features: 24
Feature existence: PASSED

========== CORE BE-3 INTEGRITY ==========
BE-3 rows: 281478
Unique NPI: 281478
Row count: PASSED
NPI uniqueness: PASSED

========== SPATIAL METRIC POLICY ==========


,provider_count
spatial_metric_policy,
area_level_fallback,274994
unresolved,4135
provider_level,2349


Spatial policy counts: PASSED

========== PROVIDER-LEVEL FEATURE COVERAGE ==========
nearest_provider_npi: provider=2349 | non_provider=0
nearest_provider_distance_miles: provider=2349 | non_provider=0
second_nearest_provider_npi: provider=2349 | non_provider=0
second_nearest_provider_distance_miles: provider=2349 | non_provider=0
provider_count_5mi: provider=2349 | non_provider=0
provider_count_10mi: provider=2349 | non_provider=0
provider_count_25mi: provider=2349 | non_provider=0
provider_count_50mi: provider=2349 | non_provider=0
provider_count_100mi: provider=2349 | non_provider=0
same_taxonomy_count_5mi: provider=2349 | non_provider=0
same_taxonomy_count_10mi: provider=2349 | non_provider=0
same_taxonomy_count_25mi: provider=2349 | non_provider=0
same_taxonomy_count_50mi: provider=2349 | non_provider=0
same_taxonomy_count_100mi: provider=2349 | non_provider=0
Provider-level coverage/exclusion: PASSED

========== AREA-LEVEL FEATURE COVERAGE ==========
area_provider_count_5mi: area

,feature,non_null_count,coverage_pct
0,nearest_provider_npi,2349,0.8345
1,nearest_provider_distance_miles,2349,0.8345
2,second_nearest_provider_npi,2349,0.8345
3,second_nearest_provider_distance_miles,2349,0.8345
4,provider_count_5mi,2349,0.8345
5,provider_count_10mi,2349,0.8345
6,provider_count_25mi,2349,0.8345
7,provider_count_50mi,2349,0.8345
8,provider_count_100mi,2349,0.8345
9,same_taxonomy_count_5mi,2349,0.8345



========== FINAL BE-3 SPATIAL FEATURE LAYER ==========
Rows: 281478
Columns: 31
Spatial features: 24
Provider-level spatial features: 14
Area-level spatial features: 10

STAGE 2C-40 CELL 5: PASSED
Full BE-3 rows: 281478
Spatial feature columns: 24
Provider-level features: 14
Area-level features: 10
Provider-level rows: 2349
Area-level fallback rows: 274994
Unresolved rows: 4135
Feature separation: PASSED
Coverage validation: PASSED
Radius monotonicity: PASSED
Spatial policy validation: PASSED
NPI uniqueness: PASSED


In [ ]:
# ============================================================
# STAGE 2C-41 — CELL 1
# BUILD FINAL BE-3 OUTPUT LAYER
# ============================================================

import pandas as pd
import numpy as np

print("========== STAGE 2C-41 CELL 1 ==========")
print("Building final BE-3 output layer...")


# ------------------------------------------------------------
# 1. Confirm required source layers exist
# ------------------------------------------------------------

required_objects = {
    "be3_spatial_features": be3_spatial_features,
    "be3_spatial": be3_spatial,
}

for name, obj in required_objects.items():
    assert obj is not None, f"Missing required object: {name}"

print("Required BE-3 layers: PASSED")


# ------------------------------------------------------------
# 2. Provider/geographic fields to preserve
# ------------------------------------------------------------

BE3_BASE_COLUMNS = [
    "NPI",
    "Entity Type Code",
    "Provider Organization Name (Legal Business Name)",
    "Provider Last Name (Legal Name)",
    "Provider First Name",
    "Provider First Line Business Practice Location Address",
    "Provider Second Line Business Practice Location Address",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address Postal Code",
    "Provider Business Practice Location Address Country Code (If outside U.S.)",
    "Healthcare Provider Taxonomy Code_1",
    "Healthcare Provider Primary Taxonomy Switch_1",
]


# ------------------------------------------------------------
# 3. Spatial/geographic fields to preserve
# ------------------------------------------------------------

BE3_GEOGRAPHIC_COLUMNS = [
    "zcta_clean",
    "county_fips",
    "spatial_latitude",
    "spatial_longitude",
    "spatial_coordinate_type",
    "spatial_coordinate_source",
    "spatial_coordinate_confidence",
    "spatial_coordinate_available",
    "provider_metric_eligible",
    "area_metric_eligible",
    "zcta_centroid_fallback",
    "spatial_coordinate_unresolved",
    "spatial_metric_policy",
]


# ------------------------------------------------------------
# 4. Final 24 spatial features
# ------------------------------------------------------------

PROVIDER_SPATIAL_FEATURES = [
    "nearest_provider_npi",
    "nearest_provider_distance_miles",
    "second_nearest_provider_npi",
    "second_nearest_provider_distance_miles",
    "provider_count_5mi",
    "provider_count_10mi",
    "provider_count_25mi",
    "provider_count_50mi",
    "provider_count_100mi",
    "same_taxonomy_count_5mi",
    "same_taxonomy_count_10mi",
    "same_taxonomy_count_25mi",
    "same_taxonomy_count_50mi",
    "same_taxonomy_count_100mi",
]

AREA_SPATIAL_FEATURES = [
    "area_provider_count_5mi",
    "area_provider_count_10mi",
    "area_provider_count_25mi",
    "area_provider_count_50mi",
    "area_provider_count_100mi",
    "area_same_taxonomy_count_5mi",
    "area_same_taxonomy_count_10mi",
    "area_same_taxonomy_count_25mi",
    "area_same_taxonomy_count_50mi",
    "area_same_taxonomy_count_100mi",
]

ALL_SPATIAL_FEATURES = (
    PROVIDER_SPATIAL_FEATURES
    + AREA_SPATIAL_FEATURES
)

assert len(PROVIDER_SPATIAL_FEATURES) == 14
assert len(AREA_SPATIAL_FEATURES) == 10
assert len(ALL_SPATIAL_FEATURES) == 24

print("Spatial feature inventory: PASSED")


# ------------------------------------------------------------
# 5. Identify columns actually available
# ------------------------------------------------------------

available_base = [
    c for c in BE3_BASE_COLUMNS
    if c in be3_spatial_features.columns
]

available_geo = [
    c for c in BE3_GEOGRAPHIC_COLUMNS
    if c in be3_spatial_features.columns
]

missing_base = [
    c for c in BE3_BASE_COLUMNS
    if c not in be3_spatial_features.columns
]

missing_geo = [
    c for c in BE3_GEOGRAPHIC_COLUMNS
    if c not in be3_spatial_features.columns
]

missing_spatial = [
    c for c in ALL_SPATIAL_FEATURES
    if c not in be3_spatial_features.columns
]

assert len(missing_spatial) == 0, missing_spatial

print("Spatial columns available: PASSED")

if missing_base:
    print(
        "Note: base columns not present in current spatial layer:"
    )
    print(missing_base)

if missing_geo:
    print(
        "Note: geographic columns not present in current spatial layer:"
    )
    print(missing_geo)


# ------------------------------------------------------------
# 6. Build final BE-3 spatial output
# ------------------------------------------------------------

FINAL_BE3_COLUMNS = (
    ["NPI"]
    + [
        c for c in available_base
        if c != "NPI"
    ]
    + available_geo
    + ALL_SPATIAL_FEATURES
)

# Remove accidental duplicates while preserving order
FINAL_BE3_COLUMNS = list(
    dict.fromkeys(FINAL_BE3_COLUMNS)
)

be3_final = (
    be3_spatial_features[
        FINAL_BE3_COLUMNS
    ]
    .copy()
)


# ------------------------------------------------------------
# 7. Core integrity checks
# ------------------------------------------------------------

assert len(be3_final) == 281478

assert be3_final["NPI"].notna().all()

assert be3_final["NPI"].nunique() == 281478

assert be3_final["NPI"].is_unique


# ------------------------------------------------------------
# 8. Spatial policy integrity
# ------------------------------------------------------------

assert (
    be3_final["spatial_metric_policy"]
    .value_counts()
    .to_dict()
    == {
        "area_level_fallback": 274994,
        "unresolved": 4135,
        "provider_level": 2349,
    }
)


# ------------------------------------------------------------
# 9. Provider-level feature isolation
# ------------------------------------------------------------

provider_mask = (
    be3_final["spatial_metric_policy"]
    == "provider_level"
)

area_mask = (
    be3_final["spatial_metric_policy"]
    == "area_level_fallback"
)

unresolved_mask = (
    be3_final["spatial_metric_policy"]
    == "unresolved"
)

for feature in PROVIDER_SPATIAL_FEATURES:

    assert (
        be3_final.loc[
            provider_mask,
            feature
        ]
        .notna()
        .all()
    )

    assert (
        be3_final.loc[
            ~provider_mask,
            feature
        ]
        .isna()
        .all()
    )


# ------------------------------------------------------------
# 10. Area-level feature isolation
# ------------------------------------------------------------

for feature in AREA_SPATIAL_FEATURES:

    assert (
        be3_final.loc[
            area_mask,
            feature
        ]
        .notna()
        .all()
    )

    assert (
        be3_final.loc[
            ~area_mask,
            feature
        ]
        .isna()
        .all()
    )


# ------------------------------------------------------------
# 11. Unresolved rows have no spatial metrics
# ------------------------------------------------------------

assert (
    be3_final.loc[
        unresolved_mask,
        ALL_SPATIAL_FEATURES
    ]
    .notna()
    .sum()
    .sum()
    == 0
)


# ------------------------------------------------------------
# 12. Final dimensions
# ------------------------------------------------------------

print("\n========== FINAL BE-3 OUTPUT LAYER ==========")

print(
    "Rows:",
    len(be3_final)
)

print(
    "Columns:",
    len(be3_final.columns)
)

print(
    "Spatial feature columns:",
    len(ALL_SPATIAL_FEATURES)
)

print(
    "Provider-level features:",
    len(PROVIDER_SPATIAL_FEATURES)
)

print(
    "Area-level features:",
    len(AREA_SPATIAL_FEATURES)
)

print(
    "Provider-level rows:",
    int(provider_mask.sum())
)

print(
    "Area-level fallback rows:",
    int(area_mask.sum())
)

print(
    "Unresolved rows:",
    int(unresolved_mask.sum())
)


# ------------------------------------------------------------
# 13. Final column inventory
# ------------------------------------------------------------

print("\n========== FINAL COLUMN INVENTORY ==========")

for i, column in enumerate(be3_final.columns, start=1):
    print(f"{i:02d}. {column}")


# ------------------------------------------------------------
# 14. Final PASS
# ------------------------------------------------------------

print("\n============================================================")
print("STAGE 2C-41 CELL 1: PASSED")
print("============================================================")
print("Final BE-3 rows: 281478")
print("Unique NPI: 281478")
print("Spatial features: 24")
print("Provider-level features: 14")
print("Area-level features: 10")
print("Provider-level rows: 2349")
print("Area-level fallback rows: 274994")
print("Unresolved rows: 4135")
print("Provider/area feature separation: PASSED")
print("Unresolved spatial exclusion: PASSED")
print("NPI uniqueness: PASSED")
print("============================================================")

========== STAGE 2C-41 CELL 1 ==========
Building final BE-3 output layer...
Required BE-3 layers: PASSED
Spatial feature inventory: PASSED
Spatial columns available: PASSED
Note: base columns not present in current spatial layer:
['Entity Type Code', 'Provider Organization Name (Legal Business Name)', 'Provider Last Name (Legal Name)', 'Provider First Name', 'Provider First Line Business Practice Location Address', 'Provider Second Line Business Practice Location Address', 'Provider Business Practice Location Address City Name', 'Provider Business Practice Location Address State Name', 'Provider Business Practice Location Address Postal Code', 'Provider Business Practice Location Address Country Code (If outside U.S.)', 'Healthcare Provider Taxonomy Code_1', 'Healthcare Provider Primary Taxonomy Switch_1']
Note: geographic columns not present in current spatial layer:
['zcta_clean', 'county_fips', 'spatial_coordinate_available', 'provider_metric_eligible', 'area_metric_eligible', 'zct

In [ ]:
# ============================================================
# STAGE 2C-41 — CELL 2
# RE-ATTACH BE-3 PROVIDER + GEOGRAPHY FIELDS
# ============================================================

print("========== STAGE 2C-41 CELL 2 ==========")
print("Re-attaching provider and geography fields...")


# ------------------------------------------------------------
# 1. Find the original BE-3 provider layer
# ------------------------------------------------------------

# geo_ready is the 281,478-row U.S. geography-eligible
# provider layer created earlier in BE-3.

assert "geo_ready" in globals(), (
    "geo_ready is not available. "
    "Run the earlier BE-3 geography preparation cells first."
)

assert len(geo_ready) == 281478
assert geo_ready["NPI"].is_unique

print("Original BE-3 provider layer: PASSED")


# ------------------------------------------------------------
# 2. Provider fields required for final BE-3
# ------------------------------------------------------------

PROVIDER_FIELDS_FINAL = [
    "NPI",
    "Entity Type Code",
    "Provider Organization Name (Legal Business Name)",
    "Provider Last Name (Legal Name)",
    "Provider First Name",
    "Provider First Line Business Practice Location Address",
    "Provider Second Line Business Practice Location Address",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address Postal Code",
    "Provider Business Practice Location Address Country Code (If outside U.S.)",
    "Healthcare Provider Taxonomy Code_1",
    "Healthcare Provider Primary Taxonomy Switch_1",
    "zip_clean",
    "zcta_clean",
    "county_fips",
]


missing_provider_fields = [
    c for c in PROVIDER_FIELDS_FINAL
    if c not in geo_ready.columns
]

assert len(missing_provider_fields) == 0, (
    f"Missing provider/geography fields: "
    f"{missing_provider_fields}"
)

print("Required provider/geography fields: PASSED")


# ------------------------------------------------------------
# 3. Select clean provider layer
# ------------------------------------------------------------

provider_final = (
    geo_ready[
        PROVIDER_FIELDS_FINAL
    ]
    .copy()
)

assert len(provider_final) == 281478
assert provider_final["NPI"].is_unique


# ------------------------------------------------------------
# 4. Prevent accidental duplicate columns
# ------------------------------------------------------------

spatial_columns_to_add = [
    c for c in be3_final.columns
    if c != "NPI"
]

duplicate_columns = sorted(
    set(spatial_columns_to_add)
    & set(provider_final.columns)
)

assert len(duplicate_columns) == 0, (
    f"Unexpected overlapping columns: {duplicate_columns}"
)

print("Column-overlap check: PASSED")


# ------------------------------------------------------------
# 5. Merge provider/geography + spatial layer
# ------------------------------------------------------------

be3_final_complete = provider_final.merge(
    be3_final,
    on="NPI",
    how="left",
    validate="one_to_one",
    suffixes=("", "_spatial")
)


# ------------------------------------------------------------
# 6. Core merge validation
# ------------------------------------------------------------

assert len(be3_final_complete) == 281478

assert (
    be3_final_complete["NPI"]
    .is_unique
)

assert (
    be3_final_complete["NPI"]
    .notna()
    .all()
)

print("NPI one-to-one merge: PASSED")


# ------------------------------------------------------------
# 7. Spatial feature coverage after merge
# ------------------------------------------------------------

for feature in ALL_SPATIAL_FEATURES:

    assert (
        be3_final_complete[feature]
        .equals(
            be3_final[feature]
        )
    )


print("Spatial feature preservation: PASSED")


# ------------------------------------------------------------
# 8. Validate spatial policy after merge
# ------------------------------------------------------------

policy_counts = (
    be3_final_complete[
        "spatial_metric_policy"
    ]
    .value_counts()
    .to_dict()
)

assert (
    policy_counts["provider_level"]
    == 2349
)

assert (
    policy_counts["area_level_fallback"]
    == 274994
)

assert (
    policy_counts["unresolved"]
    == 4135
)

print("Spatial policy preservation: PASSED")


# ------------------------------------------------------------
# 9. Validate taxonomy availability
# ------------------------------------------------------------

taxonomy_present = (
    be3_final_complete[
        "Healthcare Provider Taxonomy Code_1"
    ]
    .notna()
)

print(
    "Primary taxonomy populated:",
    int(taxonomy_present.sum()),
    "/",
    len(be3_final_complete)
)


# ------------------------------------------------------------
# 10. Validate geography fields
# ------------------------------------------------------------

print("\n========== GEOGRAPHY COVERAGE ==========")

print(
    "ZIP populated:",
    int(
        be3_final_complete["zip_clean"]
        .notna()
        .sum()
    )
)

print(
    "ZCTA populated:",
    int(
        be3_final_complete["zcta_clean"]
        .notna()
        .sum()
    )
)

print(
    "County FIPS populated:",
    int(
        be3_final_complete["county_fips"]
        .notna()
        .sum()
    )
)


# ------------------------------------------------------------
# 11. Final column inventory
# ------------------------------------------------------------

print("\n========== FINAL COMPLETE BE-3 LAYER ==========")

print(
    "Rows:",
    len(be3_final_complete)
)

print(
    "Columns:",
    len(be3_final_complete.columns)
)

print(
    "Spatial features:",
    len(ALL_SPATIAL_FEATURES)
)

print(
    "Provider-level spatial features:",
    len(PROVIDER_SPATIAL_FEATURES)
)

print(
    "Area-level spatial features:",
    len(AREA_SPATIAL_FEATURES)
)


print("\n========== FINAL COLUMN INVENTORY ==========")

for i, column in enumerate(
    be3_final_complete.columns,
    start=1
):
    print(
        f"{i:02d}. {column}"
    )


# ------------------------------------------------------------
# 12. Final PASS
# ------------------------------------------------------------

print("\n============================================================")
print("STAGE 2C-41 CELL 2: PASSED")
print("============================================================")
print("Full BE-3 rows: 281478")
print("Unique NPI: 281478")
print("Provider/geography fields restored: PASSED")
print("Spatial features preserved: 24")
print("Provider-level features: 14")
print("Area-level features: 10")
print("NPI one-to-one merge: PASSED")
print("Spatial policy preserved: PASSED")
print("Taxonomy/geography fields available: PASSED")
print("============================================================")

========== STAGE 2C-41 CELL 2 ==========
Re-attaching provider and geography fields...
Original BE-3 provider layer: PASSED
Required provider/geography fields: PASSED
Column-overlap check: PASSED
NPI one-to-one merge: PASSED
Spatial feature preservation: PASSED
Spatial policy preservation: PASSED
Primary taxonomy populated: 281478 / 281478

========== GEOGRAPHY COVERAGE ==========
ZIP populated: 281478
ZCTA populated: 281478
County FIPS populated: 276704

========== FINAL COMPLETE BE-3 LAYER ==========
Rows: 281478
Columns: 46
Spatial features: 24
Provider-level spatial features: 14
Area-level spatial features: 10

========== FINAL COLUMN INVENTORY ==========
01. NPI
02. Entity Type Code
03. Provider Organization Name (Legal Business Name)
04. Provider Last Name (Legal Name)
05. Provider First Name
06. Provider First Line Business Practice Location Address
07. Provider Second Line Business Practice Location Address
08. Provider Business Practice Location Address City Name
09. Provider 

In [ ]:
# ============================================================
# STAGE 2C-41 — CELL 3
# FINAL BE-3 DATA QUALITY AUDIT
# ============================================================

print("========== STAGE 2C-41 CELL 3 ==========")
print("Running final BE-3 data-quality audit...")


# ------------------------------------------------------------
# 1. Core dimensions
# ------------------------------------------------------------

assert len(be3_final_complete) == 281478
assert be3_final_complete["NPI"].is_unique
assert be3_final_complete["NPI"].notna().all()

print("Row count: PASSED")
print("NPI uniqueness: PASSED")


# ------------------------------------------------------------
# 2. Duplicate column names
# ------------------------------------------------------------

duplicate_columns = (
    be3_final_complete.columns[
        be3_final_complete.columns.duplicated()
    ]
    .tolist()
)

assert len(duplicate_columns) == 0, duplicate_columns

print("Duplicate column names: PASSED")


# ------------------------------------------------------------
# 3. Expected spatial feature inventory
# ------------------------------------------------------------

assert len(PROVIDER_SPATIAL_FEATURES) == 14
assert len(AREA_SPATIAL_FEATURES) == 10
assert len(ALL_SPATIAL_FEATURES) == 24

missing_spatial_features = [
    c for c in ALL_SPATIAL_FEATURES
    if c not in be3_final_complete.columns
]

assert len(missing_spatial_features) == 0

print("Spatial feature inventory: PASSED")


# ------------------------------------------------------------
# 4. ZIP validation
# ------------------------------------------------------------

zip_series = (
    be3_final_complete["zip_clean"]
    .astype("string")
    .str.strip()
)

assert zip_series.notna().all()

assert (
    zip_series.str.fullmatch(r"\d{5}")
    .all()
)

print("ZIP format: PASSED")


# ------------------------------------------------------------
# 5. ZCTA validation
# ------------------------------------------------------------

zcta_series = (
    be3_final_complete["zcta_clean"]
    .astype("string")
    .str.strip()
)

assert zcta_series.notna().all()

assert (
    zcta_series.str.fullmatch(r"\d{5}")
    .all()
)

print("ZCTA format: PASSED")


# ------------------------------------------------------------
# 6. County FIPS validation
# ------------------------------------------------------------

county_series = (
    be3_final_complete["county_fips"]
    .astype("string")
    .str.strip()
)

county_present = (
    be3_final_complete["county_fips"]
    .notna()
)

if county_present.any():

    assert (
        county_series[county_present]
        .str.fullmatch(r"\d{5}")
        .all()
    )

print(
    "County FIPS format: PASSED"
)

print(
    "County FIPS populated:",
    int(county_present.sum()),
    "/",
    len(be3_final_complete)
)


# ------------------------------------------------------------
# 7. Coordinate completeness
# ------------------------------------------------------------

lat_present = (
    be3_final_complete["spatial_latitude"]
    .notna()
)

lon_present = (
    be3_final_complete["spatial_longitude"]
    .notna()
)

assert (
    lat_present == lon_present
).all()

print("Coordinate pair completeness: PASSED")


# ------------------------------------------------------------
# 8. Coordinate range validation
# ------------------------------------------------------------

valid_lat = (
    be3_final_complete["spatial_latitude"]
    .dropna()
    .between(-90, 90)
    .all()
)

valid_lon = (
    be3_final_complete["spatial_longitude"]
    .dropna()
    .between(-180, 180)
    .all()
)

assert valid_lat
assert valid_lon

print("Coordinate validity: PASSED")


# ------------------------------------------------------------
# 9. Spatial policy counts
# ------------------------------------------------------------

policy_counts = (
    be3_final_complete[
        "spatial_metric_policy"
    ]
    .value_counts()
)

assert (
    policy_counts["provider_level"]
    == 2349
)

assert (
    policy_counts["area_level_fallback"]
    == 274994
)

assert (
    policy_counts["unresolved"]
    == 4135
)

print("Spatial policy counts: PASSED")


# ------------------------------------------------------------
# 10. Provider-level spatial feature isolation
# ------------------------------------------------------------

provider_mask = (
    be3_final_complete[
        "spatial_metric_policy"
    ]
    == "provider_level"
)

area_mask = (
    be3_final_complete[
        "spatial_metric_policy"
    ]
    == "area_level_fallback"
)

unresolved_mask = (
    be3_final_complete[
        "spatial_metric_policy"
    ]
    == "unresolved"
)

for feature in PROVIDER_SPATIAL_FEATURES:

    assert (
        be3_final_complete.loc[
            provider_mask,
            feature
        ]
        .notna()
        .all()
    )

    assert (
        be3_final_complete.loc[
            ~provider_mask,
            feature
        ]
        .isna()
        .all()
    )

print(
    "Provider-level feature isolation: PASSED"
)


# ------------------------------------------------------------
# 11. Area-level spatial feature isolation
# ------------------------------------------------------------

for feature in AREA_SPATIAL_FEATURES:

    assert (
        be3_final_complete.loc[
            area_mask,
            feature
        ]
        .notna()
        .all()
    )

    assert (
        be3_final_complete.loc[
            ~area_mask,
            feature
        ]
        .isna()
        .all()
    )

print(
    "Area-level feature isolation: PASSED"
)


# ------------------------------------------------------------
# 12. Unresolved rows must have no spatial metrics
# ------------------------------------------------------------

unresolved_non_null = (
    be3_final_complete.loc[
        unresolved_mask,
        ALL_SPATIAL_FEATURES
    ]
    .notna()
    .sum()
    .sum()
)

assert unresolved_non_null == 0

print(
    "Unresolved spatial exclusion: PASSED"
)


# ------------------------------------------------------------
# 13. Provider density monotonicity
# ------------------------------------------------------------

provider_density_features = [
    "provider_count_5mi",
    "provider_count_10mi",
    "provider_count_25mi",
    "provider_count_50mi",
    "provider_count_100mi",
]

provider_density_values = (
    be3_final_complete.loc[
        provider_mask,
        provider_density_features
    ]
)

assert (
    provider_density_values.diff(axis=1)
    .iloc[:, 1:]
    .ge(0)
    .all()
    .all()
)

print(
    "Provider density monotonicity: PASSED"
)


# ------------------------------------------------------------
# 14. Area density monotonicity
# ------------------------------------------------------------

area_density_features = [
    "area_provider_count_5mi",
    "area_provider_count_10mi",
    "area_provider_count_25mi",
    "area_provider_count_50mi",
    "area_provider_count_100mi",
]

area_density_values = (
    be3_final_complete.loc[
        area_mask,
        area_density_features
    ]
)

assert (
    area_density_values.diff(axis=1)
    .iloc[:, 1:]
    .ge(0)
    .all()
    .all()
)

print(
    "Area density monotonicity: PASSED"
)


# ------------------------------------------------------------
# 15. Same-taxonomy density monotonicity
# ------------------------------------------------------------

provider_taxonomy_density = [
    "same_taxonomy_count_5mi",
    "same_taxonomy_count_10mi",
    "same_taxonomy_count_25mi",
    "same_taxonomy_count_50mi",
    "same_taxonomy_count_100mi",
]

area_taxonomy_density = [
    "area_same_taxonomy_count_5mi",
    "area_same_taxonomy_count_10mi",
    "area_same_taxonomy_count_25mi",
    "area_same_taxonomy_count_50mi",
    "area_same_taxonomy_count_100mi",
]

assert (
    be3_final_complete.loc[
        provider_mask,
        provider_taxonomy_density
    ]
    .diff(axis=1)
    .iloc[:, 1:]
    .ge(0)
    .all()
    .all()
)

assert (
    be3_final_complete.loc[
        area_mask,
        area_taxonomy_density
    ]
    .diff(axis=1)
    .iloc[:, 1:]
    .ge(0)
    .all()
    .all()
)

print(
    "Same-taxonomy density monotonicity: PASSED"
)


# ------------------------------------------------------------
# 16. Nearest / second-nearest ordering
# ------------------------------------------------------------

nearest = (
    be3_final_complete.loc[
        provider_mask,
        "nearest_provider_distance_miles"
    ]
)

second_nearest = (
    be3_final_complete.loc[
        provider_mask,
        "second_nearest_provider_distance_miles"
    ]
)

assert (
    nearest <= second_nearest
).all()

print(
    "Nearest/second-nearest ordering: PASSED"
)


# ------------------------------------------------------------
# 17. Non-negative count validation
# ------------------------------------------------------------

count_features = (
    provider_density_features
    + provider_taxonomy_density
    + area_density_features
    + area_taxonomy_density
)

for feature in count_features:

    assert (
        be3_final_complete[feature]
        .dropna()
        >= 0
    ).all()

print(
    "Non-negative spatial counts: PASSED"
)


# ------------------------------------------------------------
# 18. Final missingness summary
# ------------------------------------------------------------

audit_summary = pd.DataFrame({
    "column": be3_final_complete.columns,
    "non_null_count": [
        int(
            be3_final_complete[c]
            .notna()
            .sum()
        )
        for c in be3_final_complete.columns
    ],
})

audit_summary["missing_count"] = (
    len(be3_final_complete)
    - audit_summary["non_null_count"]
)

audit_summary["coverage_pct"] = (
    100
    * audit_summary["non_null_count"]
    / len(be3_final_complete)
).round(4)


# ------------------------------------------------------------
# 19. Final schema dimensions
# ------------------------------------------------------------

print("\n========== FINAL SCHEMA ==========")

print(
    "Rows:",
    len(be3_final_complete)
)

print(
    "Columns:",
    len(be3_final_complete.columns)
)

print(
    "Spatial features:",
    len(ALL_SPATIAL_FEATURES)
)

print(
    "Provider-level features:",
    len(PROVIDER_SPATIAL_FEATURES)
)

print(
    "Area-level features:",
    len(AREA_SPATIAL_FEATURES)
)


# ------------------------------------------------------------
# 20. Display missingness summary
# ------------------------------------------------------------

print("\n========== FINAL MISSINGNESS SUMMARY ==========")

display(
    audit_summary
)


# ------------------------------------------------------------
# 21. Final PASS
# ------------------------------------------------------------

print("\n============================================================")
print("STAGE 2C-41 CELL 3: PASSED")
print("============================================================")
print("Full BE-3 rows: 281478")
print("Unique NPI: 281478")
print("Final columns: 46")
print("Spatial features: 24")
print("Provider-level features: 14")
print("Area-level features: 10")
print("ZIP validation: PASSED")
print("ZCTA validation: PASSED")
print("County FIPS validation: PASSED")
print("Coordinate validation: PASSED")
print("Spatial policy validation: PASSED")
print("Provider/area separation: PASSED")
print("Density monotonicity: PASSED")
print("Nearest-distance validation: PASSED")
print("Non-negative counts: PASSED")
print("============================================================")

========== STAGE 2C-41 CELL 3 ==========
Running final BE-3 data-quality audit...
Row count: PASSED
NPI uniqueness: PASSED
Duplicate column names: PASSED
Spatial feature inventory: PASSED
ZIP format: PASSED
ZCTA format: PASSED
County FIPS format: PASSED
County FIPS populated: 276704 / 281478
Coordinate pair completeness: PASSED
Coordinate validity: PASSED
Spatial policy counts: PASSED
Provider-level feature isolation: PASSED
Area-level feature isolation: PASSED
Unresolved spatial exclusion: PASSED
Provider density monotonicity: PASSED
Area density monotonicity: PASSED
Same-taxonomy density monotonicity: PASSED
Nearest/second-nearest ordering: PASSED
Non-negative spatial counts: PASSED

========== FINAL SCHEMA ==========
Rows: 281478
Columns: 46
Spatial features: 24
Provider-level features: 14
Area-level features: 10

========== FINAL MISSINGNESS SUMMARY ==========


,column,non_null_count,missing_count,coverage_pct
0,NPI,281478,0,100.0000
1,Entity Type Code,281478,0,100.0000
2,Provider Organization Name (Legal Business Name),47003,234475,16.6986
3,Provider Last Name (Legal Name),234472,47006,83.3003
4,Provider First Name,234475,47003,83.3014
5,Provider First Line Business Practice Location...,281478,0,100.0000
6,Provider Second Line Business Practice Locatio...,94501,186977,33.5731
7,Provider Business Practice Location Address Ci...,281478,0,100.0000
8,Provider Business Practice Location Address St...,281478,0,100.0000
9,Provider Business Practice Location Address Po...,281478,0,100.0000



STAGE 2C-41 CELL 3: PASSED
Full BE-3 rows: 281478
Unique NPI: 281478
Final columns: 46
Spatial features: 24
Provider-level features: 14
Area-level features: 10
ZIP validation: PASSED
ZCTA validation: PASSED
County FIPS validation: PASSED
Coordinate validation: PASSED
Spatial policy validation: PASSED
Provider/area separation: PASSED
Density monotonicity: PASSED
Nearest-distance validation: PASSED
Non-negative counts: PASSED


In [ ]:
# ============================================================
# STAGE 2C-41 — CELL 4
# FINAL BE-3 PRODUCTION EXPORT
# ============================================================

from pathlib import Path
import json
import pandas as pd

print("========== STAGE 2C-41 CELL 4 ==========")
print("Preparing final BE-3 production exports...")


# ------------------------------------------------------------
# 1. Confirm final layer
# ------------------------------------------------------------

assert "be3_final_complete" in globals()

assert len(be3_final_complete) == 281478
assert be3_final_complete["NPI"].is_unique

assert len(ALL_SPATIAL_FEATURES) == 24
assert len(PROVIDER_SPATIAL_FEATURES) == 14
assert len(AREA_SPATIAL_FEATURES) == 10

print("Final BE-3 layer: PASSED")


# ------------------------------------------------------------
# 2. Define output directory
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/data/be3_final"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Output directory:",
    OUTPUT_DIR
)


# ------------------------------------------------------------
# 3. Define output paths
# ------------------------------------------------------------

PARQUET_PATH = (
    OUTPUT_DIR
    / "be3_final_281478.parquet"
)

CSV_PATH = (
    OUTPUT_DIR
    / "be3_final_281478.csv"
)

SCHEMA_PATH = (
    OUTPUT_DIR
    / "be3_final_schema.json"
)


# ------------------------------------------------------------
# 4. Final defensive copy
# ------------------------------------------------------------

be3_export = (
    be3_final_complete
    .copy()
)


# ------------------------------------------------------------
# 5. Final export integrity checks
# ------------------------------------------------------------

assert len(be3_export) == 281478
assert be3_export["NPI"].is_unique
assert be3_export["NPI"].notna().all()

assert (
    len([
        c for c in be3_export.columns
        if c in ALL_SPATIAL_FEATURES
    ])
    == 24
)

print("Export integrity checks: PASSED")


# ------------------------------------------------------------
# 6. Export Parquet
# ------------------------------------------------------------

be3_export.to_parquet(
    PARQUET_PATH,
    index=False
)

print(
    "Parquet written:",
    PARQUET_PATH
)


# ------------------------------------------------------------
# 7. Export CSV
# ------------------------------------------------------------

be3_export.to_csv(
    CSV_PATH,
    index=False
)

print(
    "CSV written:",
    CSV_PATH
)


# ------------------------------------------------------------
# 8. Build schema manifest
# ------------------------------------------------------------

schema_manifest = {
    "dataset": "BE-3 Final Geospatial Provider Layer",
    "stage": "2C-41",
    "status": "production_ready",
    "row_count": int(len(be3_export)),
    "column_count": int(len(be3_export.columns)),
    "primary_key": "NPI",
    "unique_npi_count": int(
        be3_export["NPI"].nunique()
    ),

    "spatial_feature_count": 24,

    "provider_level_spatial_feature_count": 14,
    "area_level_spatial_feature_count": 10,

    "provider_level_rows": 2349,
    "area_level_fallback_rows": 274994,
    "unresolved_rows": 4135,

    "county_fips_populated": int(
        be3_export["county_fips"].notna().sum()
    ),

    "county_fips_missing": int(
        be3_export["county_fips"].isna().sum()
    ),

    "spatial_coordinates_populated": int(
        be3_export["spatial_latitude"].notna().sum()
    ),

    "spatial_coordinates_missing": int(
        be3_export["spatial_latitude"].isna().sum()
    ),

    "provider_level_features": PROVIDER_SPATIAL_FEATURES,

    "area_level_features": AREA_SPATIAL_FEATURES,

    "all_spatial_features": ALL_SPATIAL_FEATURES,

    "columns": [
        {
            "position": i,
            "name": column,
            "dtype": str(
                be3_export[column].dtype
            )
        }
        for i, column in enumerate(
            be3_export.columns,
            start=1
        )
    ],

    "spatial_metric_policy": {
        "provider_level": 2349,
        "area_level_fallback": 274994,
        "unresolved": 4135
    },

    "geography_policy": {
        "county_mapping": (
            "2020 Census ZCTA-to-county relationship; "
            "primary county selected by largest AREALAND_PART"
        ),
        "unmatched_county_fips": 4774,
        "unresolved_spatial_rows": 4135
    },

    "coordinate_policy": {
        "provider_coordinates": (
            "Census geocoder matched provider addresses"
        ),
        "area_coordinates": (
            "2020 Census ZCTA Gazetteer centroid fallback"
        ),
        "provider_metrics_use": (
            "provider-level coordinates only"
        ),
        "area_metrics_use": (
            "ZCTA centroid fallback coordinates"
        )
    },

    "files": {
        "parquet": str(PARQUET_PATH),
        "csv": str(CSV_PATH),
        "schema": str(SCHEMA_PATH)
    }
}


# ------------------------------------------------------------
# 9. Write schema manifest
# ------------------------------------------------------------

with open(
    SCHEMA_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        schema_manifest,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Schema manifest written:",
    SCHEMA_PATH
)


# ------------------------------------------------------------
# 10. Confirm files exist
# ------------------------------------------------------------

assert PARQUET_PATH.exists()
assert CSV_PATH.exists()
assert SCHEMA_PATH.exists()

print("\n========== EXPORTED FILES ==========")

for path in [
    PARQUET_PATH,
    CSV_PATH,
    SCHEMA_PATH
]:

    size_mb = (
        path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"{path.name}: "
        f"{size_mb:.2f} MB"
    )


# ------------------------------------------------------------
# 11. Final PASS
# ------------------------------------------------------------

print("\n============================================================")
print("STAGE 2C-41 CELL 4: PASSED")
print("============================================================")
print("BE-3 production layer exported successfully")
print("Rows: 281478")
print("Columns: 46")
print("Spatial features: 24")
print("Provider-level features: 14")
print("Area-level features: 10")
print("Parquet: PASSED")
print("CSV: PASSED")
print("Schema manifest: PASSED")
print("============================================================")

========== STAGE 2C-41 CELL 4 ==========
Preparing final BE-3 production exports...
Final BE-3 layer: PASSED
Output directory: /content/drive/MyDrive/data/be3_final
Export integrity checks: PASSED
Parquet written: /content/drive/MyDrive/data/be3_final/be3_final_281478.parquet
CSV written: /content/drive/MyDrive/data/be3_final/be3_final_281478.csv
Schema manifest written: /content/drive/MyDrive/data/be3_final/be3_final_schema.json

========== EXPORTED FILES ==========
be3_final_281478.parquet: 19.32 MB
be3_final_281478.csv: 73.68 MB
be3_final_schema.json: 0.01 MB

STAGE 2C-41 CELL 4: PASSED
BE-3 production layer exported successfully
Rows: 281478
Columns: 46
Spatial features: 24
Provider-level features: 14
Area-level features: 10
Parquet: PASSED
CSV: PASSED
Schema manifest: PASSED


In [ ]:
# ============================================================
# STAGE 2C-41 — CELL 5
# BE-3 → ML-1 HANDOFF PREPARATION
# ============================================================

from pathlib import Path
import json
import pandas as pd

print("========== STAGE 2C-41 CELL 5 ==========")
print("Preparing BE-3 → ML-1 handoff...")


# ------------------------------------------------------------
# 1. Confirm source
# ------------------------------------------------------------

assert "be3_final_complete" in globals()

be3_ml1_source = be3_final_complete.copy()

assert len(be3_ml1_source) == 281478
assert be3_ml1_source["NPI"].is_unique
assert be3_ml1_source["NPI"].notna().all()

print("Source BE-3 layer: PASSED")


# ------------------------------------------------------------
# 2. Define ML-1 feature groups
# ------------------------------------------------------------

ML1_PROVIDER_LEVEL_FEATURES = [
    "nearest_provider_distance_miles",
    "second_nearest_provider_distance_miles",
    "provider_count_5mi",
    "provider_count_10mi",
    "provider_count_25mi",
    "provider_count_50mi",
    "provider_count_100mi",
    "same_taxonomy_count_5mi",
    "same_taxonomy_count_10mi",
    "same_taxonomy_count_25mi",
    "same_taxonomy_count_50mi",
    "same_taxonomy_count_100mi",
]

ML1_AREA_LEVEL_FEATURES = [
    "area_provider_count_5mi",
    "area_provider_count_10mi",
    "area_provider_count_25mi",
    "area_provider_count_50mi",
    "area_provider_count_100mi",
    "area_same_taxonomy_count_5mi",
    "area_same_taxonomy_count_10mi",
    "area_same_taxonomy_count_25mi",
    "area_same_taxonomy_count_50mi",
    "area_same_taxonomy_count_100mi",
]

ML1_SPATIAL_FEATURES = (
    ML1_PROVIDER_LEVEL_FEATURES
    + ML1_AREA_LEVEL_FEATURES
)


# ------------------------------------------------------------
# 3. Define identifiers / geography / metadata
# ------------------------------------------------------------

ML1_IDENTIFIER_COLUMNS = [
    "NPI"
]

ML1_PROVIDER_CONTEXT_COLUMNS = [
    "Entity Type Code",
    "Provider Organization Name (Legal Business Name)",
    "Provider Last Name (Legal Name)",
    "Provider First Name",
    "Healthcare Provider Taxonomy Code_1",
    "Healthcare Provider Primary Taxonomy Switch_1",
]

ML1_GEOGRAPHY_COLUMNS = [
    "zip_clean",
    "zcta_clean",
    "county_fips",
]

ML1_SPATIAL_METADATA_COLUMNS = [
    "spatial_latitude",
    "spatial_longitude",
    "spatial_coordinate_type",
    "spatial_coordinate_source",
    "spatial_coordinate_confidence",
    "spatial_metric_policy",
]


# ------------------------------------------------------------
# 4. Define address columns separately
# ------------------------------------------------------------

ML1_ADDRESS_COLUMNS = [
    "Provider First Line Business Practice Location Address",
    "Provider Second Line Business Practice Location Address",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address Postal Code",
    "Provider Business Practice Location Address Country Code (If outside U.S.)",
]


# ------------------------------------------------------------
# 5. Create the ML-1 handoff dataframe
# ------------------------------------------------------------

ML1_HANDOFF_COLUMNS = (
    ML1_IDENTIFIER_COLUMNS
    + ML1_PROVIDER_CONTEXT_COLUMNS
    + ML1_GEOGRAPHY_COLUMNS
    + ML1_SPATIAL_METADATA_COLUMNS
    + ML1_SPATIAL_FEATURES
)

# Defensive column validation
missing_handoff_columns = [
    c
    for c in ML1_HANDOFF_COLUMNS
    if c not in be3_ml1_source.columns
]

assert not missing_handoff_columns, (
    f"Missing ML-1 columns: {missing_handoff_columns}"
)

be3_ml1_handoff = (
    be3_ml1_source[
        ML1_HANDOFF_COLUMNS
    ]
    .copy()
)

assert len(be3_ml1_handoff) == 281478
assert be3_ml1_handoff["NPI"].is_unique


# ------------------------------------------------------------
# 6. Validate spatial feature inventory
# ------------------------------------------------------------

assert len(ML1_PROVIDER_LEVEL_FEATURES) == 12
assert len(ML1_AREA_LEVEL_FEATURES) == 10
assert len(ML1_SPATIAL_FEATURES) == 22

print("ML-1 spatial feature inventory: PASSED")


# ------------------------------------------------------------
# 7. Why only 22 spatial features here?
# ------------------------------------------------------------

# The two nearest-provider NPI columns are identifiers for
# reference providers rather than numerical ML features.
#
# They remain available in the full BE-3 production layer but
# are intentionally NOT included in the numerical ML-1 feature
# matrix.

ML1_REFERENCE_COLUMNS = [
    "nearest_provider_npi",
    "second_nearest_provider_npi",
]

for column in ML1_REFERENCE_COLUMNS:
    assert column in be3_ml1_source.columns

print("Nearest-provider reference identifiers: PASSED")


# ------------------------------------------------------------
# 8. Create ML-1 numerical feature matrix
# ------------------------------------------------------------

ML1_NUMERIC_FEATURES = ML1_SPATIAL_FEATURES.copy()

ml1_feature_matrix = (
    be3_ml1_source[
        ML1_NUMERIC_FEATURES
    ]
    .copy()
)

assert ml1_feature_matrix.shape == (
    281478,
    22
)

print("ML-1 numerical feature matrix: PASSED")


# ------------------------------------------------------------
# 9. Validate provider-level policy
# ------------------------------------------------------------

provider_mask = (
    be3_ml1_source["spatial_metric_policy"]
    == "provider_level"
)

area_mask = (
    be3_ml1_source["spatial_metric_policy"]
    == "area_level_fallback"
)

unresolved_mask = (
    be3_ml1_source["spatial_metric_policy"]
    == "unresolved"
)


# Provider-level metrics must only exist for provider-level rows
assert (
    be3_ml1_source.loc[
        ~provider_mask,
        ML1_PROVIDER_LEVEL_FEATURES
    ]
    .notna()
    .sum()
    .sum()
    == 0
)

# Area-level metrics must only exist for area-level rows
assert (
    be3_ml1_source.loc[
        ~area_mask,
        ML1_AREA_LEVEL_FEATURES
    ]
    .notna()
    .sum()
    .sum()
    == 0
)

# Unresolved rows must have no spatial features
assert (
    be3_ml1_source.loc[
        unresolved_mask,
        ML1_SPATIAL_FEATURES
    ]
    .notna()
    .sum()
    .sum()
    == 0
)

print("ML-1 spatial policy separation: PASSED")


# ------------------------------------------------------------
# 10. Create ML-1 metadata manifest
# ------------------------------------------------------------

ML1_HANDOFF_DIR = Path(
    "/content/drive/MyDrive/data/be3_final"
)

ML1_HANDOFF_PATH = (
    ML1_HANDOFF_DIR
    / "be3_ml1_handoff_281478.parquet"
)

ML1_FEATURE_MATRIX_PATH = (
    ML1_HANDOFF_DIR
    / "be3_ml1_spatial_features_281478.parquet"
)

ML1_MANIFEST_PATH = (
    ML1_HANDOFF_DIR
    / "be3_ml1_handoff_manifest.json"
)


# ------------------------------------------------------------
# 11. Export handoff dataframe
# ------------------------------------------------------------

be3_ml1_handoff.to_parquet(
    ML1_HANDOFF_PATH,
    index=False
)

ml1_feature_matrix.to_parquet(
    ML1_FEATURE_MATRIX_PATH,
    index=False
)


# ------------------------------------------------------------
# 12. Build manifest
# ------------------------------------------------------------

ml1_manifest = {

    "dataset": "BE-3 → ML-1 Handoff",

    "source": "be3_final_complete",

    "status": "ready_for_ml1",

    "row_count": int(
        len(be3_ml1_handoff)
    ),

    "unique_npi_count": int(
        be3_ml1_handoff["NPI"].nunique()
    ),

    "handoff_column_count": int(
        len(be3_ml1_handoff.columns)
    ),

    "spatial_feature_count": 22,

    "provider_level_spatial_feature_count": 12,

    "area_level_spatial_feature_count": 10,

    "reference_provider_identifier_count": 2,

    "provider_level_rows": int(
        provider_mask.sum()
    ),

    "area_level_fallback_rows": int(
        area_mask.sum()
    ),

    "unresolved_rows": int(
        unresolved_mask.sum()
    ),

    "feature_groups": {

        "provider_level": (
            ML1_PROVIDER_LEVEL_FEATURES
        ),

        "area_level": (
            ML1_AREA_LEVEL_FEATURES
        ),

        "reference_provider_identifiers": (
            ML1_REFERENCE_COLUMNS
        )
    },

    "identifiers": (
        ML1_IDENTIFIER_COLUMNS
    ),

    "provider_context": (
        ML1_PROVIDER_CONTEXT_COLUMNS
    ),

    "geography": (
        ML1_GEOGRAPHY_COLUMNS
    ),

    "spatial_metadata": (
        ML1_SPATIAL_METADATA_COLUMNS
    ),

    "policy": {

        "provider_metrics": (
            "Use only rows with "
            "spatial_metric_policy == "
            "'provider_level'."
        ),

        "area_metrics": (
            "Use only rows with "
            "spatial_metric_policy == "
            "'area_level_fallback'."
        ),

        "unresolved": (
            "Rows with "
            "spatial_metric_policy == "
            "'unresolved' have no spatial "
            "metrics and must not be assigned "
            "synthetic spatial values."
        ),

        "provider_coordinates": (
            "Provider-level metrics use "
            "actual Census-geocoded provider "
            "coordinates only."
        ),

        "area_coordinates": (
            "Area-level metrics use "
            "2020 Census ZCTA centroid "
            "fallback coordinates."
        )
    },

    "geography_provenance": {

        "zcta_vintage": "2020",

        "county_relationship": (
            "2020 Census ZCTA-to-county "
            "relationship"
        ),

        "county_selection_rule": (
            "Largest AREALAND_PART within "
            "each ZCTA"
        )
    },

    "outputs": {

        "handoff": str(
            ML1_HANDOFF_PATH
        ),

        "spatial_features": str(
            ML1_FEATURE_MATRIX_PATH
        ),

        "manifest": str(
            ML1_MANIFEST_PATH
        )
    }
}


with open(
    ML1_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ml1_manifest,
        f,
        indent=2,
        ensure_ascii=False
    )


# ------------------------------------------------------------
# 13. Final file checks
# ------------------------------------------------------------

assert ML1_HANDOFF_PATH.exists()
assert ML1_FEATURE_MATRIX_PATH.exists()
assert ML1_MANIFEST_PATH.exists()


# ------------------------------------------------------------
# 14. Final summary
# ------------------------------------------------------------

print("\n========== ML-1 HANDOFF SUMMARY ==========")

print(
    "Rows:",
    len(be3_ml1_handoff)
)

print(
    "Handoff columns:",
    len(be3_ml1_handoff.columns)
)

print(
    "Numerical spatial features:",
    len(ML1_SPATIAL_FEATURES)
)

print(
    "Provider-level spatial features:",
    len(ML1_PROVIDER_LEVEL_FEATURES)
)

print(
    "Area-level spatial features:",
    len(ML1_AREA_LEVEL_FEATURES)
)

print(
    "Provider-level rows:",
    int(provider_mask.sum())
)

print(
    "Area-level fallback rows:",
    int(area_mask.sum())
)

print(
    "Unresolved rows:",
    int(unresolved_mask.sum())
)

print(
    "\nHandoff:",
    ML1_HANDOFF_PATH
)

print(
    "Feature matrix:",
    ML1_FEATURE_MATRIX_PATH
)

print(
    "Manifest:",
    ML1_MANIFEST_PATH
)


# ------------------------------------------------------------
# 15. Final PASS
# ------------------------------------------------------------

print("\n============================================================")
print("STAGE 2C-41 CELL 5: PASSED")
print("============================================================")
print("BE-3 → ML-1 handoff prepared")
print("Rows: 281478")
print("Unique NPI: 281478")
print("Numerical spatial features: 22")
print("Provider-level features: 12")
print("Area-level features: 10")
print("Nearest-provider reference NPIs: 2")
print("Provider/area policy separation: PASSED")
print("Unresolved spatial exclusion: PASSED")
print("Handoff Parquet: PASSED")
print("Feature-matrix Parquet: PASSED")
print("Manifest: PASSED")
print("============================================================")

========== STAGE 2C-41 CELL 5 ==========
Preparing BE-3 → ML-1 handoff...
Source BE-3 layer: PASSED
ML-1 spatial feature inventory: PASSED
Nearest-provider reference identifiers: PASSED
ML-1 numerical feature matrix: PASSED
ML-1 spatial policy separation: PASSED

========== ML-1 HANDOFF SUMMARY ==========
Rows: 281478
Handoff columns: 38
Numerical spatial features: 22
Provider-level spatial features: 12
Area-level spatial features: 10
Provider-level rows: 2349
Area-level fallback rows: 274994
Unresolved rows: 4135

Handoff: /content/drive/MyDrive/data/be3_final/be3_ml1_handoff_281478.parquet
Feature matrix: /content/drive/MyDrive/data/be3_final/be3_ml1_spatial_features_281478.parquet
Manifest: /content/drive/MyDrive/data/be3_final/be3_ml1_handoff_manifest.json

STAGE 2C-41 CELL 5: PASSED
BE-3 → ML-1 handoff prepared
Rows: 281478
Unique NPI: 281478
Numerical spatial features: 22
Provider-level features: 12
Area-level features: 10
Nearest-provider reference NPIs: 2
Provider/area policy s

In [ ]:
# ============================================================
# FE-2 HANDOFF — STAGE 1
# Cell 1: Locate and load BE-3 production CSV
# ============================================================

import os
import glob
import pandas as pd
import numpy as np

# Search MyDrive for the BE-3 production CSV
matches = glob.glob(
    "/content/drive/MyDrive/**/be3_final_281478.csv",
    recursive=True
)

print("Files found:")
for path in matches:
    print(" -", path)

assert len(matches) > 0, (
    "be3_final_281478.csv was not found anywhere under "
    "/content/drive/MyDrive/"
)

BE3_CSV = matches[0]

print(f"\nUsing BE-3 source:")
print(BE3_CSV)

# Load
be3_final = pd.read_csv(
    BE3_CSV,
    low_memory=False
)

print("\nBE-3 dataset loaded successfully")
print(f"Rows: {len(be3_final):,}")
print(f"Columns: {len(be3_final.columns):,}")

# Integrity checks
assert len(be3_final) == 281_478
assert be3_final["NPI"].is_unique

print("NPI uniqueness: PASS")
print("Row-count validation: PASS")

Files found:
 - /content/drive/MyDrive/data/be3_final/be3_final_281478.csv

Using BE-3 source:
/content/drive/MyDrive/data/be3_final/be3_final_281478.csv

BE-3 dataset loaded successfully
Rows: 281,478
Columns: 46
NPI uniqueness: PASS
Row-count validation: PASS


In [ ]:
# ============================================================
# FE-2 HANDOFF — STAGE 1
# Cell 2: Define FE-2 map-ready schema
# ============================================================

FE2_MAP_COLUMNS = [
    # Provider identity
    "NPI",

    # Coordinates
    "spatial_latitude",
    "spatial_longitude",
    "spatial_coordinate_type",
    "spatial_coordinate_source",
    "spatial_coordinate_confidence",

    # Geographic identifiers
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address State Name",
    "zip_clean",
    "zcta_clean",
    "county_fips",

    # Spatial policy
    "spatial_metric_policy",

    # Provider-level proximity
    "nearest_provider_distance_miles",
    "second_nearest_provider_distance_miles",

    # Provider-level density
    "provider_count_5mi",
    "provider_count_10mi",
    "provider_count_25mi",
    "provider_count_50mi",
    "provider_count_100mi",

    # Provider-level same-taxonomy density
    "same_taxonomy_count_5mi",
    "same_taxonomy_count_10mi",
    "same_taxonomy_count_25mi",
    "same_taxonomy_count_50mi",
    "same_taxonomy_count_100mi",

    # Area-level density
    "area_provider_count_5mi",
    "area_provider_count_10mi",
    "area_provider_count_25mi",
    "area_provider_count_50mi",
    "area_provider_count_100mi",

    # Area-level same-taxonomy density
    "area_same_taxonomy_count_5mi",
    "area_same_taxonomy_count_10mi",
    "area_same_taxonomy_count_25mi",
    "area_same_taxonomy_count_50mi",
    "area_same_taxonomy_count_100mi",
]

# Verify every requested field exists
missing_columns = [
    col for col in FE2_MAP_COLUMNS
    if col not in be3_final.columns
]

assert not missing_columns, (
    f"Missing FE-2 columns: {missing_columns}"
)

print("FE-2 schema validation: PASS")
print(f"FE-2 columns: {len(FE2_MAP_COLUMNS)}")

print("\nFE-2 map-ready schema:")
for i, col in enumerate(FE2_MAP_COLUMNS, start=1):
    print(f"{i:02d}. {col}")

FE-2 schema validation: PASS
FE-2 columns: 34

FE-2 map-ready schema:
01. NPI
02. spatial_latitude
03. spatial_longitude
04. spatial_coordinate_type
05. spatial_coordinate_source
06. spatial_coordinate_confidence
07. Provider Business Practice Location Address City Name
08. Provider Business Practice Location Address State Name
09. zip_clean
10. zcta_clean
11. county_fips
12. spatial_metric_policy
13. nearest_provider_distance_miles
14. second_nearest_provider_distance_miles
15. provider_count_5mi
16. provider_count_10mi
17. provider_count_25mi
18. provider_count_50mi
19. provider_count_100mi
20. same_taxonomy_count_5mi
21. same_taxonomy_count_10mi
22. same_taxonomy_count_25mi
23. same_taxonomy_count_50mi
24. same_taxonomy_count_100mi
25. area_provider_count_5mi
26. area_provider_count_10mi
27. area_provider_count_25mi
28. area_provider_count_50mi
29. area_provider_count_100mi
30. area_same_taxonomy_count_5mi
31. area_same_taxonomy_count_10mi
32. area_same_taxonomy_count_25mi
33. area_

In [ ]:
# ============================================================
# FE-2 HANDOFF — STAGE 1
# Cell 3: Create map-ready FE-2 dataframe
# ============================================================

be3_fe2_map = be3_final[FE2_MAP_COLUMNS].copy()

print("FE-2 map dataframe created")
print(f"Rows: {len(be3_fe2_map):,}")
print(f"Columns: {len(be3_fe2_map.columns):,}")

# Core integrity checks
assert len(be3_fe2_map) == 281_478
assert be3_fe2_map["NPI"].is_unique
assert list(be3_fe2_map.columns) == FE2_MAP_COLUMNS

print("Row count: PASS")
print("NPI uniqueness: PASS")
print("Column order/schema: PASS")

# Display a compact preview
display(be3_fe2_map.head(5))

FE-2 map dataframe created
Rows: 281,478
Columns: 34
Row count: PASS
NPI uniqueness: PASS
Column order/schema: PASS


,NPI,spatial_latitude,spatial_longitude,spatial_coordinate_type,spatial_coordinate_source,spatial_coordinate_confidence,Provider Business Practice Location Address City Name,Provider Business Practice Location Address State Name,zip_clean,zcta_clean,...,area_provider_count_5mi,area_provider_count_10mi,area_provider_count_25mi,area_provider_count_50mi,area_provider_count_100mi,area_same_taxonomy_count_5mi,area_same_taxonomy_count_10mi,area_same_taxonomy_count_25mi,area_same_taxonomy_count_50mi,area_same_taxonomy_count_100mi
0,1184627820,37.675649,-77.336708,zcta_centroid,Census_ZCTA_Gazetteer_2020,area_level,MECHANICSVILLE,VA,23116,23116,...,69.0,156.0,977.0,1523.0,7770.0,1.0,1.0,1.0,3.0,11.0
1,1568460541,42.328708,-71.255900,zcta_centroid,Census_ZCTA_Gazetteer_2020,area_level,NEWTON,MA,2462,2462,...,378.0,3036.0,5214.0,8664.0,12600.0,8.0,30.0,51.0,79.0,96.0
2,1174610372,38.635332,-88.927355,zcta_centroid,Census_ZCTA_Gazetteer_2020,area_level,SALEM,IL,62881,62881,...,13.0,14.0,95.0,278.0,4085.0,2.0,2.0,8.0,14.0,138.0
3,1013997535,34.905022,-76.886672,zcta_centroid,Census_ZCTA_Gazetteer_2020,area_level,HAVELOCK,NC,28532,28532,...,22.0,23.0,212.0,506.0,1816.0,1.0,1.0,5.0,7.0,29.0
4,1932105079,40.632667,-73.996669,zcta_centroid,Census_ZCTA_Gazetteer_2020,area_level,BROOKLYN,NY,11219,11219,...,1535.0,4367.0,10397.0,15314.0,26203.0,37.0,107.0,208.0,296.0,537.0


In [ ]:
# ============================================================
# FE-2 HANDOFF — STAGE 1
# Cell 4: Coordinate and map eligibility audit — CORRECTED
# ============================================================

# ------------------------------------------------------------
# 1. Coordinate completeness
# ------------------------------------------------------------

lat_present = be3_fe2_map["spatial_latitude"].notna()
lon_present = be3_fe2_map["spatial_longitude"].notna()

both_coordinates = lat_present & lon_present
partial_coordinates = lat_present ^ lon_present
no_coordinates = ~lat_present & ~lon_present

print("Coordinate completeness")
print("----------------------")
print(f"Both coordinates:     {both_coordinates.sum():,}")
print(f"Partial coordinates:  {partial_coordinates.sum():,}")
print(f"No coordinates:       {no_coordinates.sum():,}")

assert partial_coordinates.sum() == 0
assert both_coordinates.sum() + no_coordinates.sum() == len(be3_fe2_map)

print("Coordinate completeness: PASS")


# ------------------------------------------------------------
# 2. Coordinate validity
# ------------------------------------------------------------

valid_lat = be3_fe2_map["spatial_latitude"].between(-90, 90)
valid_lon = be3_fe2_map["spatial_longitude"].between(-180, 180)

valid_coordinates = both_coordinates & valid_lat & valid_lon
invalid_coordinates = both_coordinates & ~(valid_lat & valid_lon)

print("\nCoordinate validity")
print("-------------------")
print(f"Usable valid coordinates: {valid_coordinates.sum():,}")
print(f"Invalid coordinate rows:  {invalid_coordinates.sum():,}")

assert invalid_coordinates.sum() == 0

print("Coordinate validity: PASS")


# ------------------------------------------------------------
# 3. Coordinate type distribution
# ------------------------------------------------------------

print("\nCoordinate type distribution")
print("----------------------------")

coordinate_type_counts = (
    be3_fe2_map["spatial_coordinate_type"]
    .value_counts(dropna=False)
)

display(coordinate_type_counts.to_frame("rows"))


# ------------------------------------------------------------
# 4. Spatial metric policy distribution
# ------------------------------------------------------------

print("\nSpatial metric policy distribution")
print("----------------------------------")

policy_counts = (
    be3_fe2_map["spatial_metric_policy"]
    .value_counts(dropna=False)
)

display(policy_counts.to_frame("rows"))


# ------------------------------------------------------------
# 5. FE-2 plotting eligibility
# ------------------------------------------------------------

plot_eligible = (
    both_coordinates
    & be3_fe2_map["spatial_coordinate_type"].isin([
        "provider_coordinate",
        "zcta_centroid"
    ])
)

# IMPORTANT:
# Canonical BE-3 value is "unresolved"
unresolved = (
    be3_fe2_map["spatial_coordinate_type"]
    == "unresolved"
)

print("\nFE-2 map eligibility")
print("--------------------")
print(f"Plot-eligible rows: {plot_eligible.sum():,}")
print(f"Unresolved rows:     {unresolved.sum():,}")

assert plot_eligible.sum() == 277_343
assert unresolved.sum() == 4_135

print("Map eligibility counts: PASS")


# ------------------------------------------------------------
# 6. Critical FE-2 policy checks
# ------------------------------------------------------------

provider_coordinate_rows = (
    be3_fe2_map["spatial_coordinate_type"]
    == "provider_coordinate"
)

zcta_centroid_rows = (
    be3_fe2_map["spatial_coordinate_type"]
    == "zcta_centroid"
)

# Provider coordinates must have provider-level policy
assert (
    be3_fe2_map.loc[
        provider_coordinate_rows,
        "spatial_metric_policy"
    ].eq("provider_level").all()
)

# ZCTA centroid coordinates must have area-level fallback policy
assert (
    be3_fe2_map.loc[
        zcta_centroid_rows,
        "spatial_metric_policy"
    ].eq("area_level_fallback").all()
)

# Unresolved rows must have no coordinates
assert (
    be3_fe2_map.loc[
        unresolved,
        ["spatial_latitude", "spatial_longitude"]
    ].isna().all().all()
)

print("\nSpatial policy separation: PASS")
print("Unresolved-coordinate exclusion: PASS")


# ------------------------------------------------------------
# 7. Final expected counts
# ------------------------------------------------------------

assert provider_coordinate_rows.sum() == 2_349
assert zcta_centroid_rows.sum() == 274_994
assert unresolved.sum() == 4_135

assert (
    provider_coordinate_rows.sum()
    + zcta_centroid_rows.sum()
    + unresolved.sum()
    == len(be3_fe2_map)
)

print("\n========================================")
print("FE-2 COORDINATE AUDIT: PASSED")
print("========================================")

Coordinate completeness
----------------------
Both coordinates:     277,343
Partial coordinates:  0
No coordinates:       4,135
Coordinate completeness: PASS

Coordinate validity
-------------------
Usable valid coordinates: 277,343
Invalid coordinate rows:  0
Coordinate validity: PASS

Coordinate type distribution
----------------------------


,rows
spatial_coordinate_type,
zcta_centroid,274994
unresolved,4135
provider_coordinate,2349



Spatial metric policy distribution
----------------------------------


,rows
spatial_metric_policy,
area_level_fallback,274994
unresolved,4135
provider_level,2349



FE-2 map eligibility
--------------------
Plot-eligible rows: 277,343
Unresolved rows:     4,135
Map eligibility counts: PASS

Spatial policy separation: PASS
Unresolved-coordinate exclusion: PASS

FE-2 COORDINATE AUDIT: PASSED


In [ ]:
# ============================================================
# FE-2 HANDOFF — STAGE 1
# Cell 5: Spatial feature availability and policy audit
# ============================================================

PROVIDER_NUMERIC_FEATURES = [
    "nearest_provider_distance_miles",
    "second_nearest_provider_distance_miles",
    "provider_count_5mi",
    "provider_count_10mi",
    "provider_count_25mi",
    "provider_count_50mi",
    "provider_count_100mi",
    "same_taxonomy_count_5mi",
    "same_taxonomy_count_10mi",
    "same_taxonomy_count_25mi",
    "same_taxonomy_count_50mi",
    "same_taxonomy_count_100mi",
]

AREA_NUMERIC_FEATURES = [
    "area_provider_count_5mi",
    "area_provider_count_10mi",
    "area_provider_count_25mi",
    "area_provider_count_50mi",
    "area_provider_count_100mi",
    "area_same_taxonomy_count_5mi",
    "area_same_taxonomy_count_10mi",
    "area_same_taxonomy_count_25mi",
    "area_same_taxonomy_count_50mi",
    "area_same_taxonomy_count_100mi",
]

PROVIDER_ROWS = (
    be3_fe2_map["spatial_metric_policy"] == "provider_level"
)

AREA_ROWS = (
    be3_fe2_map["spatial_metric_policy"] == "area_level_fallback"
)

UNRESOLVED_ROWS = (
    be3_fe2_map["spatial_metric_policy"] == "unresolved"
)

print("Spatial feature availability audit")
print("===================================")

print(f"Provider-level rows: {PROVIDER_ROWS.sum():,}")
print(f"Area-level rows:     {AREA_ROWS.sum():,}")
print(f"Unresolved rows:     {UNRESOLVED_ROWS.sum():,}")


# ------------------------------------------------------------
# Provider-level features
# ------------------------------------------------------------

print("\nProvider-level feature availability")
print("------------------------------------")

for col in PROVIDER_NUMERIC_FEATURES:
    provider_populated = be3_fe2_map.loc[
        PROVIDER_ROWS, col
    ].notna().sum()

    non_provider_populated = be3_fe2_map.loc[
        ~PROVIDER_ROWS, col
    ].notna().sum()

    print(
        f"{col}: "
        f"{provider_populated:,} provider rows populated; "
        f"{non_provider_populated:,} non-provider rows populated"
    )

    assert provider_populated == 2_349
    assert non_provider_populated == 0

print("Provider-level feature isolation: PASS")


# ------------------------------------------------------------
# Area-level features
# ------------------------------------------------------------

print("\nArea-level feature availability")
print("--------------------------------")

for col in AREA_NUMERIC_FEATURES:
    area_populated = be3_fe2_map.loc[
        AREA_ROWS, col
    ].notna().sum()

    non_area_populated = be3_fe2_map.loc[
        ~AREA_ROWS, col
    ].notna().sum()

    print(
        f"{col}: "
        f"{area_populated:,} area rows populated; "
        f"{non_area_populated:,} non-area rows populated"
    )

    assert area_populated == 274_994
    assert non_area_populated == 0

print("Area-level feature isolation: PASS")


# ------------------------------------------------------------
# Unresolved rows
# ------------------------------------------------------------

print("\nUnresolved-row feature audit")
print("----------------------------")

provider_unresolved_values = (
    be3_fe2_map.loc[
        UNRESOLVED_ROWS,
        PROVIDER_NUMERIC_FEATURES
    ]
    .notna()
    .sum()
    .sum()
)

area_unresolved_values = (
    be3_fe2_map.loc[
        UNRESOLVED_ROWS,
        AREA_NUMERIC_FEATURES
    ]
    .notna()
    .sum()
    .sum()
)

print(
    f"Provider-level values on unresolved rows: "
    f"{provider_unresolved_values:,}"
)

print(
    f"Area-level values on unresolved rows: "
    f"{area_unresolved_values:,}"
)

assert provider_unresolved_values == 0
assert area_unresolved_values == 0

print("Unresolved spatial feature exclusion: PASS")


# ------------------------------------------------------------
# Non-negative count checks
# ------------------------------------------------------------

ALL_COUNT_FEATURES = (
    PROVIDER_NUMERIC_FEATURES[2:]
    + AREA_NUMERIC_FEATURES
)

negative_values = (
    be3_fe2_map[ALL_COUNT_FEATURES]
    .lt(0)
    .sum()
    .sum()
)

print("\nNon-negative count audit")
print("------------------------")
print(f"Negative count values: {negative_values:,}")

assert negative_values == 0

print("Non-negative counts: PASS")


# ------------------------------------------------------------
# Final inventory
# ------------------------------------------------------------

print("\n========================================")
print("FE-2 SPATIAL FEATURE AUDIT: PASSED")
print("========================================")

print(f"Provider numeric features: {len(PROVIDER_NUMERIC_FEATURES)}")
print(f"Area numeric features:     {len(AREA_NUMERIC_FEATURES)}")
print(f"Total numeric features:    {len(PROVIDER_NUMERIC_FEATURES) + len(AREA_NUMERIC_FEATURES)}")

Spatial feature availability audit
Provider-level rows: 2,349
Area-level rows:     274,994
Unresolved rows:     4,135

Provider-level feature availability
------------------------------------
nearest_provider_distance_miles: 2,349 provider rows populated; 0 non-provider rows populated
second_nearest_provider_distance_miles: 2,349 provider rows populated; 0 non-provider rows populated
provider_count_5mi: 2,349 provider rows populated; 0 non-provider rows populated
provider_count_10mi: 2,349 provider rows populated; 0 non-provider rows populated
provider_count_25mi: 2,349 provider rows populated; 0 non-provider rows populated
provider_count_50mi: 2,349 provider rows populated; 0 non-provider rows populated
provider_count_100mi: 2,349 provider rows populated; 0 non-provider rows populated
same_taxonomy_count_5mi: 2,349 provider rows populated; 0 non-provider rows populated
same_taxonomy_count_10mi: 2,349 provider rows populated; 0 non-provider rows populated
same_taxonomy_count_25mi: 2,34

In [ ]:
# ============================================================
# FE-2 HANDOFF — STAGE 1
# Cell 6: Export FE-2 map handoff artifacts
# ============================================================

import os
import json
from datetime import datetime, timezone

FE2_OUTPUT_DIR = "/content/drive/MyDrive/data/be3_final"

os.makedirs(FE2_OUTPUT_DIR, exist_ok=True)

FE2_PARQUET_PATH = os.path.join(
    FE2_OUTPUT_DIR,
    "be3_fe2_map_handoff_281478.parquet"
)

FE2_CSV_PATH = os.path.join(
    FE2_OUTPUT_DIR,
    "be3_fe2_map_handoff_281478.csv"
)

FE2_MANIFEST_PATH = os.path.join(
    FE2_OUTPUT_DIR,
    "be3_fe2_map_handoff_manifest.json"
)


# ------------------------------------------------------------
# 1. Final pre-export validation
# ------------------------------------------------------------

assert len(be3_fe2_map) == 281_478
assert be3_fe2_map["NPI"].is_unique
assert list(be3_fe2_map.columns) == FE2_MAP_COLUMNS

assert (
    be3_fe2_map["spatial_coordinate_type"]
    .value_counts()
    .to_dict()
    == {
        "zcta_centroid": 274_994,
        "unresolved": 4_135,
        "provider_coordinate": 2_349,
    }
)

assert (
    be3_fe2_map["spatial_metric_policy"]
    .value_counts()
    .to_dict()
    == {
        "area_level_fallback": 274_994,
        "unresolved": 4_135,
        "provider_level": 2_349,
    }
)

print("Pre-export validation: PASS")


# ------------------------------------------------------------
# 2. Export Parquet
# ------------------------------------------------------------

be3_fe2_map.to_parquet(
    FE2_PARQUET_PATH,
    index=False
)

print(f"Parquet written: {FE2_PARQUET_PATH}")


# ------------------------------------------------------------
# 3. Export CSV
# ------------------------------------------------------------

be3_fe2_map.to_csv(
    FE2_CSV_PATH,
    index=False
)

print(f"CSV written: {FE2_CSV_PATH}")


# ------------------------------------------------------------
# 4. Build manifest
# ------------------------------------------------------------

manifest = {
    "artifact": "BE-3 FE-2 Map Data Handoff",
    "artifact_version": "1.0",
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "source": {
        "be3_source_file": "be3_final_281478.csv",
        "source_rows": 281478,
        "source_columns": 46
    },

    "handoff": {
        "rows": int(len(be3_fe2_map)),
        "columns": int(len(be3_fe2_map.columns)),
        "npi_unique": bool(be3_fe2_map["NPI"].is_unique),
        "column_count": 34
    },

    "coordinate_summary": {
        "provider_coordinate": 2349,
        "zcta_centroid": 274994,
        "unresolved": 4135,
        "plot_eligible": 277343
    },

    "coordinate_policy": {
        "provider_coordinate": (
            "Actual provider-level Census geocoded coordinate. "
            "May be used as provider-level map location."
        ),
        "zcta_centroid": (
            "Census 2020 ZCTA centroid used as area-level fallback. "
            "Must not be presented as an exact provider location."
        ),
        "unresolved": (
            "No usable spatial coordinate. Do not plot as a point."
        )
    },

    "metric_policy": {
        "provider_level_rows": 2349,
        "area_level_fallback_rows": 274994,
        "unresolved_rows": 4135
    },

    "feature_groups": {
        "provider_numeric_features": PROVIDER_NUMERIC_FEATURES,
        "area_numeric_features": AREA_NUMERIC_FEATURES,
        "total_numeric_features": 22
    },

    "validation": {
        "coordinate_completeness": "PASS",
        "coordinate_validity": "PASS",
        "provider_feature_isolation": "PASS",
        "area_feature_isolation": "PASS",
        "unresolved_feature_exclusion": "PASS",
        "non_negative_counts": "PASS",
        "npi_uniqueness": "PASS"
    },

    "files": {
        "parquet": os.path.basename(FE2_PARQUET_PATH),
        "csv": os.path.basename(FE2_CSV_PATH),
        "manifest": os.path.basename(FE2_MANIFEST_PATH)
    }
}

with open(FE2_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(f"Manifest written: {FE2_MANIFEST_PATH}")


# ------------------------------------------------------------
# 5. File-size report
# ------------------------------------------------------------

def size_mb(path):
    return os.path.getsize(path) / (1024 ** 2)

print("\n========================================")
print("FE-2 EXPORT COMPLETE")
print("========================================")

print(f"Rows:              {len(be3_fe2_map):,}")
print(f"Columns:           {len(be3_fe2_map.columns):,}")
print(f"Parquet size:      {size_mb(FE2_PARQUET_PATH):.2f} MB")
print(f"CSV size:          {size_mb(FE2_CSV_PATH):.2f} MB")
print(f"Manifest size:     {size_mb(FE2_MANIFEST_PATH):.4f} MB")

print("\nArtifacts:")
print(FE2_PARQUET_PATH)
print(FE2_CSV_PATH)
print(FE2_MANIFEST_PATH)

Pre-export validation: PASS
Parquet written: /content/drive/MyDrive/data/be3_final/be3_fe2_map_handoff_281478.parquet
CSV written: /content/drive/MyDrive/data/be3_final/be3_fe2_map_handoff_281478.csv
Manifest written: /content/drive/MyDrive/data/be3_final/be3_fe2_map_handoff_manifest.json

FE-2 EXPORT COMPLETE
Rows:              281,478
Columns:           34
Parquet size:      9.36 MB
CSV size:          54.40 MB
Manifest size:     0.0022 MB

Artifacts:
/content/drive/MyDrive/data/be3_final/be3_fe2_map_handoff_281478.parquet
/content/drive/MyDrive/data/be3_final/be3_fe2_map_handoff_281478.csv
/content/drive/MyDrive/data/be3_final/be3_fe2_map_handoff_manifest.json


In [ ]:
# ============================================================
# FE-2 HANDOFF — STAGE 1
# Cell 7: Reload and verify exported artifacts
# ============================================================

import json
import pandas as pd


# ------------------------------------------------------------
# 1. Reload Parquet
# ------------------------------------------------------------

fe2_parquet_check = pd.read_parquet(FE2_PARQUET_PATH)

print("Parquet reload")
print("--------------")
print(f"Rows:    {len(fe2_parquet_check):,}")
print(f"Columns: {len(fe2_parquet_check.columns):,}")

assert len(fe2_parquet_check) == 281_478
assert len(fe2_parquet_check.columns) == 34
assert fe2_parquet_check["NPI"].is_unique
assert list(fe2_parquet_check.columns) == FE2_MAP_COLUMNS

print("Parquet row count: PASS")
print("Parquet column count: PASS")
print("Parquet NPI uniqueness: PASS")
print("Parquet schema/order: PASS")


# ------------------------------------------------------------
# 2. Reload CSV
# ------------------------------------------------------------

fe2_csv_check = pd.read_csv(
    FE2_CSV_PATH,
    dtype={"NPI": "string"}
)

print("\nCSV reload")
print("----------")
print(f"Rows:    {len(fe2_csv_check):,}")
print(f"Columns: {len(fe2_csv_check.columns):,}")

assert len(fe2_csv_check) == 281_478
assert len(fe2_csv_check.columns) == 34
assert fe2_csv_check["NPI"].is_unique
assert list(fe2_csv_check.columns) == FE2_MAP_COLUMNS

print("CSV row count: PASS")
print("CSV column count: PASS")
print("CSV NPI uniqueness: PASS")
print("CSV schema/order: PASS")


# ------------------------------------------------------------
# 3. Coordinate counts from exported Parquet
# ------------------------------------------------------------

parquet_coordinate_counts = (
    fe2_parquet_check["spatial_coordinate_type"]
    .value_counts(dropna=False)
    .to_dict()
)

print("\nParquet coordinate counts")
print("-------------------------")
print(parquet_coordinate_counts)

assert parquet_coordinate_counts == {
    "zcta_centroid": 274_994,
    "unresolved": 4_135,
    "provider_coordinate": 2_349,
}

print("Parquet coordinate counts: PASS")


# ------------------------------------------------------------
# 4. Metric-policy counts from exported Parquet
# ------------------------------------------------------------

parquet_policy_counts = (
    fe2_parquet_check["spatial_metric_policy"]
    .value_counts(dropna=False)
    .to_dict()
)

print("\nParquet metric-policy counts")
print("----------------------------")
print(parquet_policy_counts)

assert parquet_policy_counts == {
    "area_level_fallback": 274_994,
    "unresolved": 4_135,
    "provider_level": 2_349,
}

print("Parquet metric-policy counts: PASS")


# ------------------------------------------------------------
# 5. Coordinate completeness after reload
# ------------------------------------------------------------

lat_present = fe2_parquet_check["spatial_latitude"].notna()
lon_present = fe2_parquet_check["spatial_longitude"].notna()

partial_coordinates = lat_present ^ lon_present
both_coordinates = lat_present & lon_present

assert partial_coordinates.sum() == 0
assert both_coordinates.sum() == 277_343

print("\nReloaded coordinate completeness: PASS")


# ------------------------------------------------------------
# 6. Compare Parquet with original handoff dataframe
# ------------------------------------------------------------

assert fe2_parquet_check.equals(be3_fe2_map)

print("Parquet vs in-memory dataframe: PASS")


# ------------------------------------------------------------
# 7. Verify CSV row identity against Parquet
# ------------------------------------------------------------

parquet_npis = set(fe2_parquet_check["NPI"].astype(str))
csv_npis = set(fe2_csv_check["NPI"].astype(str))

assert parquet_npis == csv_npis

print("CSV/Parquet NPI identity: PASS")


# ------------------------------------------------------------
# 8. Reload manifest
# ------------------------------------------------------------

with open(FE2_MANIFEST_PATH, "r", encoding="utf-8") as f:
    fe2_manifest_check = json.load(f)

print("\nManifest validation")
print("-------------------")

assert fe2_manifest_check["handoff"]["rows"] == 281_478
assert fe2_manifest_check["handoff"]["columns"] == 34
assert fe2_manifest_check["handoff"]["npi_unique"] is True

assert fe2_manifest_check["coordinate_summary"] == {
    "provider_coordinate": 2_349,
    "zcta_centroid": 274_994,
    "unresolved": 4_135,
    "plot_eligible": 277_343,
}

assert fe2_manifest_check["feature_groups"]["total_numeric_features"] == 22

print("Manifest row count: PASS")
print("Manifest schema count: PASS")
print("Manifest coordinate summary: PASS")
print("Manifest feature inventory: PASS")


# ------------------------------------------------------------
# 9. Final artifact verification
# ------------------------------------------------------------

print("\n========================================")
print("FE-2 EXPORTED ARTIFACT VERIFICATION")
print("========================================")

print("Parquet:  PASS")
print("CSV:      PASS")
print("Manifest: PASS")
print("Cross-file identity: PASS")
print("========================================")

Parquet reload
--------------
Rows:    281,478
Columns: 34
Parquet row count: PASS
Parquet column count: PASS
Parquet NPI uniqueness: PASS
Parquet schema/order: PASS

CSV reload
----------
Rows:    281,478
Columns: 34
CSV row count: PASS
CSV column count: PASS
CSV NPI uniqueness: PASS
CSV schema/order: PASS

Parquet coordinate counts
-------------------------
{'zcta_centroid': 274994, 'unresolved': 4135, 'provider_coordinate': 2349}
Parquet coordinate counts: PASS

Parquet metric-policy counts
----------------------------
{'area_level_fallback': 274994, 'unresolved': 4135, 'provider_level': 2349}
Parquet metric-policy counts: PASS

Reloaded coordinate completeness: PASS
Parquet vs in-memory dataframe: PASS
CSV/Parquet NPI identity: PASS

Manifest validation
-------------------
Manifest row count: PASS
Manifest schema count: PASS
Manifest coordinate summary: PASS
Manifest feature inventory: PASS

FE-2 EXPORTED ARTIFACT VERIFICATION
Parquet:  PASS
CSV:      PASS
Manifest: PASS
Cross-file

In [ ]:
# ============================================================
# FE-2 HANDOFF — STAGE 2
# Contract: BE-3 → FE-2 Map Data Contract
# ============================================================

from pathlib import Path

FE2_CONTRACT_PATH = (
    "/content/drive/MyDrive/data/be3_final/"
    "BE3_FE2_Map_Data_Contract.md"
)

contract = r"""# BE-3 → FE-2 Map Data Contract

## 1. Purpose

This document defines the map-data interface produced by BE-3
(Geospatial / External Integration / Cloud) for FE-2 (Frontend).

The handoff contains provider-level geographic identifiers,
coordinates, geographic confidence/provenance, spatial metric
features, and geographic policy metadata required for map
visualization and frontend filtering.

The handoff preserves all 281,478 geographically eligible U.S.
provider records from the BE-3 processing layer.

---

## 2. Source

Primary BE-3 source:

- `be3_final_281478.csv`
- 281,478 rows
- 46 columns

FE-2 map handoff:

- `be3_fe2_map_handoff_281478.parquet`
- `be3_fe2_map_handoff_281478.csv`
- `be3_fe2_map_handoff_manifest.json`

FE-2 handoff dimensions:

- Rows: 281,478
- Columns: 34
- Unique NPIs: 281,478

---

## 3. Coordinate Representation

Each provider row has one of three coordinate types:

| Coordinate type | Rows | Meaning |
|---|---:|---|
| `provider_coordinate` | 2,349 | Actual provider-level Census-geocoded coordinate |
| `zcta_centroid` | 274,994 | 2020 Census ZCTA centroid used as an area-level fallback |
| `unresolved` | 4,135 | No usable spatial coordinate |

Total rows: 281,478.

Rows with usable coordinates:

- 277,343

Rows without coordinates:

- 4,135

### Critical frontend rule

`zcta_centroid` MUST NOT be presented as if it were the exact
physical location of an individual provider.

It represents the geographic area associated with the provider's
ZCTA.

---

## 4. Coordinate Provenance

### provider_coordinate

Source:

- U.S. Census Geocoder
- Benchmark: `Public_AR_Current`

Coordinate confidence:

- `geocoded_address`

These coordinates represent provider-level geographic locations.

### zcta_centroid

Source:

- 2020 U.S. Census ZCTA Gazetteer

Source label:

- `Census_ZCTA_Gazetteer_2020`

These coordinates represent the centroid of the provider's ZCTA
and are used only as an area-level spatial fallback.

### unresolved

No usable coordinate was available.

These rows MUST NOT be plotted as geographic points.

---

## 5. Spatial Metric Policy

Each row has a `spatial_metric_policy` value:

| Policy | Rows | Meaning |
|---|---:|---|
| `provider_level` | 2,349 | Provider-level spatial calculations are available |
| `area_level_fallback` | 274,994 | Spatial calculations are based on ZCTA centroid / area representation |
| `unresolved` | 4,135 | No usable spatial representation |

Frontend logic MUST preserve this distinction.

Provider-level and area-level metrics MUST NOT be silently mixed.

---

## 6. FE-2 Map Eligibility

A row is map-coordinate eligible when:

- `spatial_latitude` is present
- `spatial_longitude` is present
- `spatial_coordinate_type` is either:
  - `provider_coordinate`, or
  - `zcta_centroid`

Expected map-coordinate eligible rows:

- 277,343

Unresolved rows:

- 4,135

Unresolved rows should remain available for filtering/table views
but should not be rendered as geographic points.

---

## 7. Provider-Level Spatial Features

The following 12 numerical features are provider-level features:

1. `nearest_provider_distance_miles`
2. `second_nearest_provider_distance_miles`
3. `provider_count_5mi`
4. `provider_count_10mi`
5. `provider_count_25mi`
6. `provider_count_50mi`
7. `provider_count_100mi`
8. `same_taxonomy_count_5mi`
9. `same_taxonomy_count_10mi`
10. `same_taxonomy_count_25mi`
11. `same_taxonomy_count_50mi`
12. `same_taxonomy_count_100mi`

These features are populated only for the 2,349
`provider_level` rows.

They are NOT populated for:

- ZCTA centroid rows
- unresolved rows

### Distance interpretation

Provider-to-provider distance features were calculated using
provider-level coordinates and are based on great-circle
(Haversine) distance.

ZCTA centroid coordinates were NOT substituted into provider-level
nearest-provider or provider-density calculations.

---

## 8. Area-Level Spatial Features

The following 10 numerical features are area-level features:

1. `area_provider_count_5mi`
2. `area_provider_count_10mi`
3. `area_provider_count_25mi`
4. `area_provider_count_50mi`
5. `area_provider_count_100mi`
6. `area_same_taxonomy_count_5mi`
7. `area_same_taxonomy_count_10mi`
8. `area_same_taxonomy_count_25mi`
9. `area_same_taxonomy_count_50mi`
10. `area_same_taxonomy_count_100mi`

These features are populated only for the 274,994
`area_level_fallback` rows.

They are not populated for unresolved rows.

---

## 9. Reference Provider IDs

The full BE-3 spatial layer contains nearest-provider reference IDs,
but the FE-2 map handoff intentionally contains only the numerical
distance features for the nearest-provider metrics.

FE-2 should use:

- `nearest_provider_distance_miles`
- `second_nearest_provider_distance_miles`

for geographic visualization/filtering.

---

## 10. Geographic Identifiers

The handoff includes:

- `zip_clean`
- `zcta_clean`
- `county_fips`
- City
- State

The ZCTA field represents the cleaned five-digit ZCTA/ZIP geographic
identifier used by the BE-3 geographic processing layer.

County assignment is based on the BE-3 Census ZCTA/county geographic
reference processing.

---

## 11. Required FE-2 Columns

The handoff contains exactly 34 columns:

1. `NPI`
2. `spatial_latitude`
3. `spatial_longitude`
4. `spatial_coordinate_type`
5. `spatial_coordinate_source`
6. `spatial_coordinate_confidence`
7. `Provider Business Practice Location Address City Name`
8. `Provider Business Practice Location Address State Name`
9. `zip_clean`
10. `zcta_clean`
11. `county_fips`
12. `spatial_metric_policy`
13. `nearest_provider_distance_miles`
14. `second_nearest_provider_distance_miles`
15. `provider_count_5mi`
16. `provider_count_10mi`
17. `provider_count_25mi`
18. `provider_count_50mi`
19. `provider_count_100mi`
20. `same_taxonomy_count_5mi`
21. `same_taxonomy_count_10mi`
22. `same_taxonomy_count_25mi`
23. `same_taxonomy_count_50mi`
24. `same_taxonomy_count_100mi`
25. `area_provider_count_5mi`
26. `area_provider_count_10mi`
27. `area_provider_count_25mi`
28. `area_provider_count_50mi`
29. `area_provider_count_100mi`
30. `area_same_taxonomy_count_5mi`
31. `area_same_taxonomy_count_10mi`
32. `area_same_taxonomy_count_25mi`
33. `area_same_taxonomy_count_50mi`
34. `area_same_taxonomy_count_100mi`

---

## 12. Data Quality Guarantees

The exported handoff was validated for:

- 281,478 rows
- 34 columns
- unique NPI
- no duplicate column names
- complete coordinate pairs
- valid latitude/longitude ranges
- correct coordinate-type counts
- correct spatial-policy counts
- provider-level feature isolation
- area-level feature isolation
- unresolved-feature exclusion
- non-negative spatial counts
- Parquet reload integrity
- CSV reload integrity
- Parquet/CSV NPI identity
- manifest consistency

Validation status: PASS.

---

## 13. Frontend Rendering Rules

### Exact provider location

When:

`spatial_coordinate_type == "provider_coordinate"`

FE-2 may display the coordinate as a provider-level location.

### Area representation

When:

`spatial_coordinate_type == "zcta_centroid"`

FE-2 should communicate that the coordinate represents the
provider's ZCTA/area rather than an exact provider location.

### Unresolved

When:

`spatial_coordinate_type == "unresolved"`

FE-2 should not plot the record geographically.

The record can still be retained for non-map views or filtering.

---

## 14. Recommended Map Usage

FE-2 may use the following fields for geographic filtering and
visualization:

- `spatial_latitude`
- `spatial_longitude`
- `spatial_coordinate_type`
- `spatial_coordinate_confidence`
- City
- State
- `zip_clean`
- `zcta_clean`
- `county_fips`
- spatial density/count features
- nearest-provider distance features
- `spatial_metric_policy`

The frontend should expose or internally preserve the distinction
between provider-level and area-level spatial information.

---

## 15. Important Limitations

1. Provider-level geocoded coordinates are available for only 2,349
   of the 281,478 rows.

2. Most geographically usable rows (274,994) use ZCTA centroids.

3. ZCTA centroid coordinates are area representations, not exact
   provider locations.

4. 4,135 rows have no usable spatial coordinate.

5. Provider-level nearest-provider and density features are available
   only where provider-level coordinates exist.

6. Area-level density features are calculated from unique ZCTA
   centroid locations and provider counts associated with those ZCTAs.

7. FE-2 must not infer an exact provider address from a ZCTA centroid.

---

## 16. Export Artifacts

All artifacts are stored under:

`/content/drive/MyDrive/data/be3_final/`

Files:

- `be3_fe2_map_handoff_281478.parquet`
- `be3_fe2_map_handoff_281478.csv`
- `be3_fe2_map_handoff_manifest.json`
- `BE3_FE2_Map_Data_Contract.md`

---

## 17. Handoff Status

BE-3 FE-2 map-data handoff:

**COMPLETE**

Export validation:

**PASSED**

Reload validation:

**PASSED**

Cross-file identity validation:

**PASSED**

FE-2 can consume the Parquet or CSV handoff according to the
coordinate and spatial-policy rules defined in this contract.
"""

Path(FE2_CONTRACT_PATH).write_text(
    contract,
    encoding="utf-8"
)

print("========================================")
print("FE-2 MAP DATA CONTRACT CREATED")
print("========================================")
print(FE2_CONTRACT_PATH)
print(f"Characters: {len(contract):,}")
print("Status: PASS")

FE-2 MAP DATA CONTRACT CREATED
/content/drive/MyDrive/data/be3_final/BE3_FE2_Map_Data_Contract.md
Characters: 9,083
Status: PASS


In [ ]:
# ============================================================
# FE-2 HANDOFF — FINAL VERIFICATION
# Verify Map Data Contract
# ============================================================

from pathlib import Path

contract_path = Path(FE2_CONTRACT_PATH)

assert contract_path.exists()
assert contract_path.is_file()

contract_text = contract_path.read_text(encoding="utf-8")

assert len(contract_text) > 8_000

required_sections = [
    "# BE-3 → FE-2 Map Data Contract",
    "## 1. Purpose",
    "## 2. Source",
    "## 3. Coordinate Representation",
    "## 4. Coordinate Provenance",
    "## 5. Spatial Metric Policy",
    "## 6. FE-2 Map Eligibility",
    "## 7. Provider-Level Spatial Features",
    "## 8. Area-Level Spatial Features",
    "## 11. Required FE-2 Columns",
    "## 12. Data Quality Guarantees",
    "## 13. Frontend Rendering Rules",
    "## 16. Export Artifacts",
    "## 17. Handoff Status",
]

missing_sections = [
    section
    for section in required_sections
    if section not in contract_text
]

assert not missing_sections, (
    f"Missing contract sections: {missing_sections}"
)

# Verify critical policy terminology
required_terms = [
    "provider_coordinate",
    "zcta_centroid",
    "unresolved",
    "provider_level",
    "area_level_fallback",
    "277,343",
    "4,135",
    "2,349",
    "274,994",
    "281,478",
    "34 columns",
]

missing_terms = [
    term
    for term in required_terms
    if term not in contract_text
]

assert not missing_terms, (
    f"Missing critical contract terms: {missing_terms}"
)

print("========================================")
print("FE-2 CONTRACT VERIFICATION")
print("========================================")
print(f"File exists:        PASS")
print(f"File readable:      PASS")
print(f"Characters:         {len(contract_text):,}")
print(f"Required sections:  PASS")
print(f"Critical terms:     PASS")
print("========================================")
print("FE-2 HANDOFF DOCUMENTATION: COMPLETE")
print("========================================")

FE-2 CONTRACT VERIFICATION
File exists:        PASS
File readable:      PASS
Characters:         9,083
Required sections:  PASS
Critical terms:     PASS
FE-2 HANDOFF DOCUMENTATION: COMPLETE
